# ZeroMRG Gate4B — image-conditioning diagnostic

Prepared only; **not submitted or executed**. Explicit new owner execution approval required. Reuse the successful Gate4 pinned environment, exact frozen CLIP ViT-L/14, epoch90 historical decoder/vocab/greedy max160, and unchanged ordered raw500 prompts/Eq6 attention/Eq5 residual/L2. No extra environment rebuild or Torch upgrade; only missing support packages and exact CLIP are installed as before.

Use the first up to five saved training IDs absent from the prompt bank (report-row identifier=image_id). Repeat the first image once, giving at most six forwards through the full generation pipeline. Train-only, no reference scoring or prompt changes. Reports may have been used in decoder text reconstruction; absence from prompts is not unseen-training generalization.

Capture actual raw visual and actual decoder-condition tensors without changing them. Save pairwise raw/enriched cosine matrices, exact feature differences, token/report comparisons and repeat-image determinism under identical settings. No seed reset or numerical backend change. Differing reports do not establish clinical correctness; identical reports do not alone prove an unused image condition. Features saved for independent verification. GPU memory observed after model loading.

Existing private COV/checkpoint datasets and verified baseline output; input/weight/source/config hashes checked. Exactly one worker, hard1800s, dependency commands300s, heartbeat30s and visible per-image/token phases. No retry. Scratch in /tmp; small JSON/logs/sources only in unique working output. If pins/mounts/selection/runtime fail, preserve failure and stop. Reverify target privacy and resources before authorized execution; no submission during preparation.

Mount recovery V2: only the approved owner/slug or short-slug attachment is searched. Bind the original annotation bytes, all746 physical image hashes, exact714 valid IDs, checkpoint571/71/72 manifests and ordered500 prompts. Accept a unique extracted/nested tree or original ZIP; extract canonical image bytes into /tmp only. Reject ambiguous roots or content/membership mismatch and save actionable JSON. Reference CSV bytes are used only for identity hashing, never for scoring. The prior submitted version1 is immutable evidence; this revised local notebook has not been submitted. Dataset differs from the paper population; do not claim numerical equivalence.

Local stage-progress revision after cancelled version2: preserve the exact frozen algorithm and dataset guards. Persist worker_start before framework imports; record stage start/completion/failure and dataset hashing/extraction progress. Persist each completed generation without extra inference. Write supervisor heartbeats to working files as well as flushed stdout; write timeout evidence before process-group kill. Explicit LF contract export is checked before child launch. These files improve surviving evidence but cannot guarantee Kaggle publishes outputs after cancellation. No v3 has been submitted.

In [ ]:
import hashlib, json, sys, uuid
from pathlib import Path
SOURCES = {'zeromrg_gate1_runtime.py': '"""Checkpoint-producing Kaggle decoder and fail-closed recovery utilities.\n\nThis profile preserves the recorded COV-CTR experiment, not modular defaults.\nNo training, model download, or inference is performed on import.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nARCHITECTURE = dict(vocab_size=368, max_tokens=160, d_model=768,\n                    num_layers=6, num_heads=8, ffn_dim=3072, dropout=0.1,\n                    activation="gelu", norm_first=False, final_layer_norm=False,\n                    scale_token_embeddings=False, pad_id=0, bos_id=1, eos_id=2)\nTRAINING = dict(seed=42, batch_size=6, learning_rate=0.001, momentum=0.99,\n                gradient_clip=5.0, patience=50, max_epochs=200,\n                precision="float32", scheduler=None,\n                selection_metric="validation_text_reconstruction_BLEU4_PTB_x100")\nMCLIP_NAME = "M-CLIP/XLM-Roberta-Large-Vit-L-14"\nDEPENDENCIES = ("train_manifest.csv", "validation_manifest.csv", "test_manifest.csv",\n                "prompt_manifest.csv", "validated_manifest.csv", "validation_audit.json",\n                "decoder_vocabulary.json", "mclip_text_features.pt", "mclip_prompt_bank.pt")\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_checkpoint(path):\n    require(Path(path).is_file(), f"Selected checkpoint does not exist: {path}")\n    # No unsafe unpickling fallback. Legacy project artifacts load with weights_only=True.\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\nclass CheckpointReportDecoder(nn.Module):\n    def __init__(self, architecture=None):\n        super().__init__()\n        self.architecture = dict(ARCHITECTURE if architecture is None else architecture)\n        a = self.architecture\n        require(not a["scale_token_embeddings"] and not a["final_layer_norm"],\n                "This class implements the recorded Kaggle decoder only")\n        self.token_embedding = nn.Embedding(a["vocab_size"], a["d_model"], padding_idx=a["pad_id"])\n        self.position_embedding = nn.Embedding(a["max_tokens"], a["d_model"])\n        layer = nn.TransformerDecoderLayer(\n            d_model=a["d_model"], nhead=a["num_heads"], dim_feedforward=a["ffn_dim"],\n            dropout=a["dropout"], activation=a["activation"], batch_first=True,\n            norm_first=a["norm_first"])\n        self.transformer = nn.TransformerDecoder(layer, num_layers=a["num_layers"])\n        self.output_projection = nn.Linear(a["d_model"], a["vocab_size"])\n\n    def forward(self, condition, input_ids):\n        a = self.architecture\n        require(input_ids.ndim == 2 and 0 < input_ids.shape[1] <= a["max_tokens"],\n                "Decoder IDs must be [B,T] within the saved positional limit")\n        require(tuple(condition.shape) == (input_ids.shape[0], a["d_model"]),\n                "Condition dimensions incompatible with checkpoint decoder")\n        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)\n        target = self.token_embedding(input_ids) + self.position_embedding(positions)\n        mask = torch.triu(torch.ones(input_ids.shape[1], input_ids.shape[1],\n                                    dtype=torch.bool, device=input_ids.device), diagonal=1)\n        hidden = self.transformer(target, condition.unsqueeze(1).to(target.dtype),\n                                  tgt_mask=mask, tgt_key_padding_mask=input_ids.eq(a["pad_id"]))\n        return self.output_projection(hidden)\n\n    @torch.no_grad()\n    def generate(self, condition, max_length=None):\n        self.eval()\n        a = self.architecture\n        limit = a["max_tokens"] if max_length is None else int(max_length)\n        require(2 <= limit <= a["max_tokens"], "Invalid generation length")\n        ids = torch.full((len(condition), 1), a["bos_id"], dtype=torch.long, device=condition.device)\n        finished = torch.zeros(len(condition), dtype=torch.bool, device=condition.device)\n        for _ in range(limit - 1):\n            next_id = self(condition, ids)[:, -1].argmax(-1)\n            next_id = torch.where(finished, torch.full_like(next_id, a["pad_id"]), next_id)\n            ids = torch.cat((ids, next_id[:, None]), dim=1)\n            finished |= next_id.eq(a["eos_id"])\n            if bool(finished.all()):\n                break\n        return ids\n\n\ndef validate_model_state(checkpoint, model):\n    require(isinstance(checkpoint, dict), "Checkpoint must be a state dictionary payload")\n    state = checkpoint.get("decoder")\n    require(isinstance(state, dict), "Required decoder state dictionary absent")\n    expected = model.state_dict()\n    require(set(state) == set(expected),\n            f"Decoder key mismatch: missing={sorted(set(expected)-set(state))}; unexpected={sorted(set(state)-set(expected))}")\n    for key, tensor in state.items():\n        require(isinstance(tensor, torch.Tensor) and tensor.shape == expected[key].shape,\n                f"Decoder shape mismatch: {key}")\n        require(tensor.dtype == expected[key].dtype, f"Decoder dtype mismatch: {key}")\n        require(bool(torch.isfinite(tensor).all()), f"Non-finite decoder weights: {key}")\n    if "architecture" in checkpoint:\n        require(checkpoint["architecture"] == model.architecture, "Saved architecture metadata differs")\n    for key, expected_value in (("vocabulary_size", model.architecture["vocab_size"]),\n                                ("max_tokens", model.architecture["max_tokens"])):\n        if key in checkpoint:\n            require(checkpoint[key] == expected_value, f"Saved {key} differs")\n\n\ndef inspect_training_state(checkpoint, model, best_checkpoint=None):\n    require(isinstance(checkpoint, dict), "Training checkpoint must be a dictionary payload")\n    required = {"epoch", "decoder", "optimizer", "history", "best_validation_bleu4", "epochs_without_improvement"}\n    require(required <= checkpoint.keys(), f"Required training state absent: {sorted(required-checkpoint.keys())}")\n    epoch = checkpoint["epoch"]\n    require(type(epoch) is int and epoch > 0, "Saved epoch must be a positive integer")\n    require(checkpoint.get("scheduler") is None or checkpoint.get("scheduler") == {},\n            "Unknown scheduler state; supply its verified implementation before resuming")\n    if "training_config" in checkpoint:\n        require(checkpoint["training_config"] == TRAINING, "Training configuration differs from recorded run")\n    history = checkpoint["history"]\n    require(isinstance(history, list) and len(history) == epoch, "History length differs from saved epoch")\n    require(all(isinstance(row, dict) and {"epoch", "training_loss", "validation_bleu4"} <= row.keys()\n                for row in history), "History record fields absent or invalid")\n    require([row["epoch"] for row in history] == list(range(1, epoch + 1)), "History is not contiguous")\n    for row in history:\n        require(all(math.isfinite(float(row[key])) for key in ("training_loss", "validation_bleu4")),\n                "Non-finite history value")\n    selected = max(history, key=lambda row: row["validation_bleu4"])\n    require(math.isclose(float(selected["validation_bleu4"]), float(checkpoint["best_validation_bleu4"]),\n                         rel_tol=0, abs_tol=1e-10), "History and saved best metric disagree")\n    require(type(checkpoint["epochs_without_improvement"]) is int and\n            checkpoint["epochs_without_improvement"] == epoch - selected["epoch"], "Patience state disagrees with history")\n    validate_model_state(checkpoint, model)\n    if best_checkpoint is not None:\n        validate_selected_best(checkpoint, model, best_checkpoint)\n    opt = checkpoint["optimizer"]\n    require(isinstance(opt, dict) and {"state", "param_groups"} <= opt.keys(), "Invalid optimizer state")\n    require(len(opt["param_groups"]) == 1, "Expected one decoder SGD parameter group")\n    group = opt["param_groups"][0]\n    for key, expected in dict(lr=0.001, momentum=0.99, dampening=0, weight_decay=0,\n                              nesterov=False, maximize=False).items():\n        require(group.get(key, False if key == "maximize" else None) == expected,\n                f"SGD setting mismatch: {key}")\n    parameters = list(model.named_parameters())\n    saved_ids = group["params"]\n    require(len(saved_ids) == len(parameters) and len(set(saved_ids)) == len(saved_ids),\n            "Optimizer parameter count or mapping incompatible")\n    require(set(opt["state"]) == set(saved_ids), "Missing or unexpected optimizer parameter states")\n    nonzero = 0\n    for pid, (name, parameter) in zip(saved_ids, parameters):\n        require(set(opt["state"][pid]) == {"momentum_buffer"}, f"Unknown/missing SGD state: {name}")\n        buffer = opt["state"][pid]["momentum_buffer"]\n        require(isinstance(buffer, torch.Tensor) and buffer.shape == parameter.shape and\n                buffer.dtype == parameter.dtype, f"Momentum shape/dtype incompatible: {name}")\n        require(bool(torch.isfinite(buffer).all()), f"Non-finite momentum: {name}")\n        nonzero += int(bool(buffer.ne(0).any()))\n    require(nonzero > 0, "All SGD momentum is zero")\n    if "parameter_names" in checkpoint:\n        require(checkpoint["parameter_names"] == [name for name, _ in parameters], "Optimizer parameter order differs")\n    return dict(epoch=epoch, next_epoch=epoch + 1, best_epoch=selected["epoch"],\n                best_validation_bleu4=float(selected["validation_bleu4"]),\n                epochs_without_improvement=checkpoint["epochs_without_improvement"],\n                momentum_buffers=len(saved_ids), nonzero_momentum_buffers=nonzero,\n                parameter_count=sum(p.numel() for _, p in parameters),\n                legacy_rng_missing="rng_state" not in checkpoint,\n                scheduler="none", compatibility="PASS")\n\n\ndef validate_selected_best(checkpoint, model, best_checkpoint):\n    require(isinstance(best_checkpoint, dict), "Selected best must be a dictionary payload")\n    selected = max(checkpoint["history"], key=lambda row: row["validation_bleu4"])\n    require(best_checkpoint.get("epoch") == selected["epoch"] and\n            math.isclose(float(best_checkpoint.get("validation_bleu4", float("nan"))),\n                         float(selected["validation_bleu4"]), rel_tol=0, abs_tol=1e-10),\n            f"Best checkpoint disagrees with history: need epoch {selected[\'epoch\']}, "\n            f"found epoch {best_checkpoint.get(\'epoch\')}. Recover matching weights; do not substitute latest weights.")\n    validate_model_state(best_checkpoint, model)\n\n\ndef restore_weights_optimizer(checkpoint, model, device="cpu"):\n    """Restore latest state for diagnostics; not authorization to continue training."""\n    result = inspect_training_state(checkpoint, model)\n    model.load_state_dict(checkpoint["decoder"], strict=True)\n    model.to(device)\n    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.99)\n    optimizer.load_state_dict(checkpoint["optimizer"])\n    for parameter, state in optimizer.state.items():\n        state["momentum_buffer"] = state["momentum_buffer"].to(parameter.device)\n    for pid, parameter in zip(checkpoint["optimizer"]["param_groups"][0]["params"], model.parameters()):\n        require(torch.equal(optimizer.state[parameter]["momentum_buffer"].cpu(),\n                            checkpoint["optimizer"]["state"][pid]["momentum_buffer"].cpu()),\n                "Momentum changed during restoration")\n    model.eval()\n    return optimizer, result\n\n\ndef restore_training(checkpoint, model, best_checkpoint, device="cpu"):\n    result = inspect_training_state(checkpoint, model, best_checkpoint)\n    optimizer, _ = restore_weights_optimizer(checkpoint, model, device)\n    return optimizer, result\n\n\ndef read_manifest(directory, name):\n    with (Path(directory) / name).open(encoding="utf-8-sig", newline="") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef validate_artifacts(directory, expected_hashes=None):\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in DEPENDENCIES}\n    if expected_hashes is not None:\n        require(hashes == expected_hashes, "Feature/vocabulary/split dependency hashes differ")\n    records = {name: read_manifest(directory, name + "_manifest.csv")\n               for name in ("train", "validation", "test", "prompt", "validated")}\n    ids = {}\n    for name, count in dict(train=571, validation=71, test=72, prompt=500, validated=714).items():\n        rows = records[name]\n        ids[name] = [row["image_id"] for row in rows]\n        require(len(rows) == count and len(set(ids[name])) == count, f"{name} count/ID uniqueness failed")\n    train, val, test = (set(ids[name]) for name in ("train", "validation", "test"))\n    require(not (train & val or train & test or val & test), "Split leakage")\n    require(train | val | test == set(ids["validated"]), "Split population differs")\n    require(set(ids["prompt"]) <= train, "Prompt bank contains non-training IDs")\n    by_id = {r["image_id"]: r for r in records["validated"]}\n    for name in ("train", "validation", "test", "prompt"):\n        require(all(row["report"] == by_id[row["image_id"]]["report"] for row in records[name]),\n                f"{name} reports differ from validated manifest")\n    audit = json.loads((directory / "validation_audit.json").read_text(encoding="utf-8"))\n    require(audit["csv_rows"] == 726 and audit["physical_images"] == 746, "Raw audit population differs")\n    vocabulary = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocabulary["max_tokens"] == 160 and len(vocabulary["vocabulary"]) == 368,\n            "Vocabulary or positional capacity differs")\n    require(vocabulary["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Special-token IDs differ")\n    expected_vocab = ["<pad>", "<bos>", "<eos>", "<unk>"] + sorted({\n        token for row in records["train"] for token in re.findall(r"\\w+|[^\\w\\s]", row["report"])})\n    require(vocabulary["vocabulary"] == expected_vocab, "Vocabulary is not the saved training-only lexicographic mapping")\n    text = load_checkpoint(directory / "mclip_text_features.pt")\n    require(text["model"] == MCLIP_NAME and text["normalization"] == "l2", "Text encoder metadata differs")\n    for name in ("train", "validation"):\n        require(set(text[name]) == set(ids[name]), f"{name} feature membership differs")\n        for vector in text[name].values():\n            require(vector.shape == (768,) and bool(torch.isfinite(vector).all()) and\n                    torch.allclose(vector.norm(), torch.tensor(1.0), atol=1e-4), "Invalid normalized text feature")\n    prompt = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    require(prompt["model"] == MCLIP_NAME and prompt.get("normalization") == "raw", "Prompt encoder/normalization differs")\n    require(prompt["prompt_ids"] == ids["prompt"], "Prompt tensor row order/provenance differs")\n    vectors = prompt["embeddings"]\n    require(vectors.shape == (500, 768) and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid prompt bank")\n    aligned = torch.stack([text["train"][key] for key in ids["prompt"]])\n    # Separate encodings are not bitwise equal. Record the observed difference;\n    # membership and frozen-cache hashes above are the provenance assertions.\n    normalized_prompts = F.normalize(vectors, dim=-1)\n    cosine = (normalized_prompts * aligned).sum(-1)\n    cache_comparison = dict(max_absolute_difference=float((normalized_prompts - aligned).abs().max()),\n                            minimum_cosine=float(cosine.min()), mean_cosine=float(cosine.mean()),\n                            exact_encoder_revision="UNVERIFIED")\n    return dict(records=records, vocabulary=vocabulary, text=text, prompt=prompt,\n                dependency_hashes=hashes, prompt_text_comparison=cache_comparison)\n\n\ndef encode_report(text, vocabulary):\n    lookup = {token: i for i, token in enumerate(vocabulary)}\n    ids = [1] + [lookup.get(token, 3) for token in re.findall(r"\\w+|[^\\w\\s]", text)] + [2]\n    require(len(ids) <= 160, "Report exceeds saved positional capacity; no silent truncation")\n    return ids\n\n\ndef decode_report(token_ids, vocabulary):\n    tokens = []\n    for raw in token_ids:\n        token = int(raw)\n        if token == 2:\n            break\n        require(0 <= token < len(vocabulary), "Generated token outside vocabulary")\n        if token not in {0, 1}:\n            tokens.append(vocabulary[token])\n    text = re.sub(r"\\s+([.,!?;:%)\\]])", r"\\1", " ".join(tokens))\n    return re.sub(r"([(\\[])\\s+", r"\\1", text).strip()\n\n\ndef enrich(image, prompts):\n    require(image.ndim == prompts.ndim == 2 and image.shape[-1] == prompts.shape[-1] == 768,\n            "Image/prompt dimensions must share the recorded 768-D CLIP space")\n    require(bool(torch.isfinite(image).all()) and bool(torch.isfinite(prompts).all()), "Non-finite encoder output")\n    scores = image @ prompts.T / math.sqrt(768)\n    attention = scores.softmax(-1)\n    residual = image + attention @ prompts\n    require(bool((residual.norm(dim=-1) > 0).all()), "Zero residual embedding")\n    phi = F.normalize(residual, p=2, dim=-1)\n    require(bool(torch.isfinite(phi).all()), "Non-finite enriched features")\n    return phi, attention\n\n\ndef capture_rng():\n    return dict(python=random.getstate(), torch_cpu=torch.get_rng_state(),\n                torch_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])\n\n\ndef restore_rng(state):\n    random.setstate(state["python"])\n    torch.set_rng_state(state["torch_cpu"].cpu())\n    if state["torch_cuda"]:\n        require(len(state["torch_cuda"]) == torch.cuda.device_count(), "CUDA RNG device count differs")\n        torch.cuda.set_rng_state_all([value.cpu() for value in state["torch_cuda"]])\n\n\ndef atomic_save(payload, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + ".tmp")\n    with temporary.open("wb") as handle:\n        torch.save(payload, handle)\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(temporary, destination)\n\n\ndef training_payload(model, optimizer, epoch, history, best_metric, bad_epochs, hashes, provenance):\n    return dict(schema_version="zeromrg_covctr_gate1_v1", epoch=epoch, decoder=model.state_dict(),\n                optimizer=optimizer.state_dict(), scheduler=None, history=history,\n                best_validation_bleu4=best_metric, epochs_without_improvement=bad_epochs,\n                architecture=model.architecture, parameter_names=[n for n, _ in model.named_parameters()],\n                training_config=TRAINING, dependency_hashes=hashes, provenance=provenance,\n                rng_state=capture_rng())\n', 'zeromrg_gate4_runtime.py': '"""Frozen COV-CTR Gate4 inference; no loading, downloading or execution on import."""\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom zeromrg_gate1_runtime import (\n    ARCHITECTURE, MCLIP_NAME, CheckpointReportDecoder, decode_report, enrich,\n    load_checkpoint, require, sha256, validate_model_state,\n)\n\nCLIP_COMMIT = "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"\nCLIP_SOURCE_SHA256 = "9540f200fbf8145479fa655382a56dab048d238cc698b9cbd8df3b6d86d3f1b6"\nCLIP_WEIGHT_SHA256 = "b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836"\nEPOCH90_SHA256 = "6166e36198737bfbe9e0c35f03f0b320c44e76f3e05068ef6299d74e9179c6b2"\nINFERENCE_FILES = (\n    "train_manifest.csv", "prompt_manifest.csv", "decoder_vocabulary.json", "mclip_prompt_bank.pt",\n)\n\n\ndef freeze(module):\n    module.eval()\n    module.requires_grad_(False)\n    require(all(p.grad is None for p in module.parameters()), "Unexpected existing gradients")\n    return module\n\n\ndef assert_frozen(module):\n    require(not module.training and all(not p.requires_grad and p.grad is None\n            for p in module.parameters()), "Inference module is not frozen/eval")\n\n\ndef load_inference_artifacts(directory, expected_hashes):\n    """Read train-only inputs. Never open validation/test references or text cache."""\n    directory = Path(directory)\n    hashes = {name: sha256(directory / name) for name in INFERENCE_FILES}\n    require(hashes == {name: expected_hashes[name] for name in INFERENCE_FILES},\n            "Inference artifact hashes differ from verified checkpoint family")\n    def ids(name):\n        with (directory / name).open(encoding="utf-8-sig", newline="") as handle:\n            return [row["image_id"] for row in csv.DictReader(handle)]\n    train_ids, prompt_ids = ids("train_manifest.csv"), ids("prompt_manifest.csv")\n    require(len(train_ids) == len(set(train_ids)) == 571, "Training IDs differ")\n    require(len(prompt_ids) == len(set(prompt_ids)) == 500 and\n            set(prompt_ids) <= set(train_ids), "Prompt membership is not training-only")\n    bank = load_checkpoint(directory / "mclip_prompt_bank.pt")\n    validate_prompt_bank(bank, prompt_ids)\n    vocab = json.loads((directory / "decoder_vocabulary.json").read_text(encoding="utf-8"))\n    require(vocab["max_tokens"] == 160 and len(vocab["vocabulary"]) == 368 and\n            vocab["special_token_ids"] == dict(pad=0, bos=1, eos=2, unk=3), "Vocabulary differs")\n    require(vocab["vocabulary"][:4] == ["<pad>", "<bos>", "<eos>", "<unk>"], "Token IDs differ")\n    return dict(train_ids=train_ids, prompt_ids=prompt_ids, prompts=bank["embeddings"].detach(),\n                vocabulary=vocab["vocabulary"], hashes=hashes)\n\n\ndef validate_prompt_bank(bank, prompt_ids):\n    require(bank["model"] == MCLIP_NAME and bank.get("normalization") == "raw",\n            "Wrong prompt encoder or normalization")\n    require(bank["prompt_ids"] == prompt_ids, "Prompt bank row ordering differs")\n    vectors = bank["embeddings"]\n    require(tuple(vectors.shape) == (500, 768) and vectors.dtype == torch.float32 and\n            not vectors.requires_grad and bool(torch.isfinite(vectors).all()) and\n            bool((vectors.norm(dim=-1) > 0).all()), "Invalid fixed raw prompt bank")\n\n\ndef restore_epoch90(path, device="cpu", expected_sha256=EPOCH90_SHA256):\n    require(sha256(path) == expected_sha256, "Epoch90 checkpoint hash differs")\n    checkpoint = load_checkpoint(path)\n    require(checkpoint.get("epoch") == 90, "Diagnostic requires verified physical epoch90")\n    with torch.device("meta"):\n        decoder = CheckpointReportDecoder()\n    validate_model_state(checkpoint, decoder)\n    decoder.load_state_dict(checkpoint["decoder"], strict=True, assign=True)\n    freeze(decoder)\n    decoder.to(device)\n    require(decoder.architecture == ARCHITECTURE, "Historical decoder architecture differs")\n    return decoder\n\n\ndef load_companion_clip(device, cache_directory):\n    """Explicit future Kaggle operation. Official package/source and weight are pinned."""\n    import clip\n    require(sha256(Path(clip.__file__).with_name("clip.py")) == CLIP_SOURCE_SHA256,\n            "Wrong OpenAI CLIP implementation")\n    url = clip.clip._MODELS["ViT-L/14"]\n    require(url.split("/")[-2] == CLIP_WEIGHT_SHA256, "Wrong companion weight identity")\n    model, preprocess = clip.load("ViT-L/14", device=device, jit=False,\n                                  download_root=str(cache_directory))\n    weight_path = Path(cache_directory) / "ViT-L-14.pt"\n    require(sha256(weight_path) == CLIP_WEIGHT_SHA256, "CLIP weight hash differs")\n    freeze(model)\n    vision = model.visual\n    require(int(vision.output_dim) == 768 and int(vision.input_resolution) == 224,\n            "Companion CLIP dimensions differ")\n    return vision, preprocess, dict(model="OpenAI ViT-L/14", commit=CLIP_COMMIT,\n        weight_sha256=CLIP_WEIGHT_SHA256, clip_source_sha256=CLIP_SOURCE_SHA256,\n        preprocess=repr(preprocess), historical_cache_revision="UNVERIFIED")\n\n\ndef preprocess_image(image, preprocess):\n    tensor = preprocess(image.convert("RGB"))\n    require(isinstance(tensor, torch.Tensor) and tuple(tensor.shape) == (3, 224, 224) and\n            tensor.dtype == torch.float32 and bool(torch.isfinite(tensor).all()),\n            "CLIP preprocessing must return finite float32 RGB [3,224,224]")\n    return tensor.unsqueeze(0)\n\n\ndef resolve_training_image(data_root, image_id, train_ids):\n    require(image_id in train_ids and Path(image_id).name == image_id and\n            "/" not in image_id and "\\\\" not in image_id, "Diagnostic image must be a saved training ID")\n    candidates = [p for p in Path(data_root).rglob(image_id) if p.is_file()]\n    require(len(candidates) == 1, "Training image missing or ambiguous")\n    require(candidates[0].resolve().is_relative_to(Path(data_root).resolve()), "Image escapes data root")\n    return candidates[0]\n\n\nclass FrozenImageToReport:\n    def __init__(self, vision, preprocess, decoder, prompts, vocabulary):\n        self.vision, self.preprocess, self.decoder = freeze(vision), preprocess, freeze(decoder)\n        require(decoder.architecture == ARCHITECTURE, "Historical decoder required")\n        require(tuple(prompts.shape) == (500, 768) and prompts.dtype == torch.float32 and\n                not prompts.requires_grad and bool(torch.isfinite(prompts).all()) and\n                bool((prompts.norm(dim=-1) > 0).all()), "Invalid raw fixed prompt tensor")\n        require(len(vocabulary) == 368, "Historical vocabulary required")\n        self.prompts, self.vocabulary = prompts.detach().clone(), list(vocabulary)\n        self.used = False\n\n    @torch.inference_mode()\n    def generate_one(self, image, max_length=160, progress=None):\n        require(not self.used, "Single-image diagnostic already attempted; no automatic retry")\n        require(2 <= max_length <= 160, "Generation exceeds historical positions")\n        self.used = True\n        log = progress or (lambda phase: None)\n        assert_frozen(self.vision)\n        assert_frozen(self.decoder)\n        device = next(self.decoder.parameters()).device\n        require(next(self.vision.parameters()).device == device, "Encoder/decoder device mismatch")\n        start = time.monotonic()\n        log("preprocess")\n        pixels = preprocess_image(image, self.preprocess).to(device)\n        log("image_encoder")\n        dtype = self.vision.conv1.weight.dtype\n        visual = self.vision(pixels.to(dtype=dtype)).float()\n        require(tuple(visual.shape) == (1, 768) and bool(torch.isfinite(visual).all()),\n                "CLIP output must be finite [1,768]")\n        log("prompt_enrichment_eq6_eq5")\n        prompts = self.prompts.to(device)\n        phi, attention = enrich(visual, prompts)\n        residual = visual + attention @ prompts\n        require(tuple(attention.shape) == (1, 500) and\n                torch.allclose(attention.sum(-1), torch.ones(1, device=device), atol=1e-5) and\n                torch.allclose(phi.norm(dim=-1), torch.ones(1, device=device), atol=1e-5),\n                "Attention/residual normalization failed")\n        # Use the existing historical greedy generation, with a hook to verify EVERY\n        # autoregressive forward is finite and visibly log bounded token progress.\n        forward_count = 0\n        def check_logits(module, inputs, output):\n            nonlocal forward_count\n            require(bool(torch.isfinite(output).all()), "Non-finite autoregressive logits")\n            forward_count += 1\n            if forward_count == 1 or forward_count % 20 == 0:\n                log("greedy_token_" + str(forward_count))\n        hook = self.decoder.register_forward_hook(check_logits)\n        try:\n            tokens = self.decoder.generate(phi, max_length=max_length).cpu()\n        finally:\n            hook.remove()\n        require(tuple(tokens.shape[:1]) == (1,) and 2 <= tokens.shape[1] <= max_length and\n                bool(((tokens >= 0) & (tokens < 368)).all()), "Invalid generated IDs")\n        assert_frozen(self.vision)\n        assert_frozen(self.decoder)\n        require(not self.prompts.requires_grad and torch.equal(prompts.cpu(), self.prompts.cpu()),\n                "Prompt tensor changed")\n        if device.type == "cuda":\n            torch.cuda.synchronize(device)\n        ids = tokens[0].tolist()\n        return dict(shapes=dict(pixels=list(pixels.shape), image_embedding=list(visual.shape),\n            prompts=list(prompts.shape), attention=list(attention.shape), residual=list(residual.shape),\n            decoder_condition=list(phi.shape), tokens=list(tokens.shape)),\n            norms=dict(image=visual.norm(dim=-1).cpu().tolist(),\n                prompt_min=float(prompts.norm(dim=-1).min()), prompt_max=float(prompts.norm(dim=-1).max()),\n                residual=residual.norm(dim=-1).cpu().tolist(), enriched=phi.norm(dim=-1).cpu().tolist()),\n            output_tokens=ids, report_text=decode_report(ids, self.vocabulary),\n            termination="EOS" if 2 in ids else "MAX_LENGTH", forward_count=forward_count,\n            device=str(device), elapsed_seconds=time.monotonic()-start,\n            gradients_disabled=True, all_modules_frozen=True, optimizer_created=False,\n            paired_supervision_used=False, test_references_accessed=False)\n', 'gate4_diagnostic_worker.py': '"""One future Kaggle training-image diagnostic, launched only by the bounded notebook."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport zipfile\nfrom importlib import metadata\nfrom pathlib import Path\n\nfrom zeromrg_gate1_runtime import require, sha256\nfrom zeromrg_gate4_runtime import (\n    CLIP_COMMIT, EPOCH90_SHA256, INFERENCE_FILES, FrozenImageToReport,\n    load_companion_clip, load_inference_artifacts, resolve_training_image, restore_epoch90,\n)\n\n\ndef log(phase):\n    print(json.dumps(dict(phase=phase, elapsed_seconds=round(time.monotonic()-START, 3))), flush=True)\n\n\ndef version(name):\n    try: return metadata.version(name)\n    except metadata.PackageNotFoundError: return None\n\n\ndef run_checked(args, timeout=300):\n    log("dependency_command")\n    subprocess.run(args, check=True, timeout=timeout)\n\n\ndef diagnostic(run_root, config):\n    import torch\n    require(config["max_images"] == 1 and config["max_tokens"] == 160 and\n            config["training_enabled"] is False and config["test_inference_enabled"] is False, "Diagnostic scope differs")\n    require(Path("/kaggle/input").is_dir() and run_root.resolve().is_relative_to(Path("/kaggle/working")),\n            "Kaggle-only diagnostic")\n    require(torch.cuda.is_available(), "Kaggle GPU required; no local fallback")\n    device = torch.device("cuda:0")\n    log("provided_environment_checks")\n    # These are the observed baseline stack. Changed images stop for review, not upgrade.\n    require(torch.__version__ == "2.11.0+cu128" and torch.version.cuda == "12.8", "Torch/CUDA drift; review required")\n    require(version("numpy") == "2.1.3" and version("Pillow") == "12.3.0", "NumPy/Pillow drift")\n    require(version("torchvision") is not None and version("torchvision").split("+")[0] == "0.26.0",\n            "Expected torchvision0.26.0 companion; do not auto-install Torch/torchvision")\n    import torchvision\n    from PIL import Image\n    # /tmp is ephemeral and excluded from Kaggle saved notebook outputs.\n    scratch = Path(tempfile.mkdtemp(prefix="zeromrg_gate4_",dir="/tmp"))\n    package_root = scratch / "inference_packages"\n    package_root.mkdir(exist_ok=False)\n    required = dict(ftfy="6.3.1", regex="2025.11.3", tqdm="4.67.1", wcwidth="0.9.2")\n    missing = [line for line in config["support_lock"].splitlines()\n               if version(line.split("==")[0]) != required[line.split("==")[0]]]\n    if missing:\n        lock = run_root / "support.lock"\n        lock.write_text("\\n".join(missing)+"\\n", encoding="utf-8")\n        run_checked([sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--require-hashes","--only-binary=:all:","--retries","0","--timeout","45",\n            "-r",str(lock)])\n    sys.path.insert(0,str(package_root))\n    for name, expected in required.items(): require(version(name) == expected, "Support package pin mismatch: "+name)\n    if version("clip") is None:\n        require(version("setuptools") is not None and version("wheel") is not None,\n                "Provided CLIP build tools missing; no implicit build-tool install")\n        run_checked([sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--no-build-isolation","--retries","0","--timeout","45",\n            "clip @ git+https://github.com/openai/CLIP.git@"+CLIP_COMMIT])\n    # The exact source hash is checked by load_companion_clip, even for provided clip.\n    log("verified_checkpoint_attachment")\n    candidates = list(Path("/kaggle/input").rglob(config["zip_name"]))\n    require(len(candidates) == 1, "Expected exactly one verified baseline ZIP attachment")\n    archive_path = candidates[0]\n    require(sha256(archive_path) == config["zip_sha256"], "Baseline ZIP differs")\n    artifact_root = scratch / "selected_inputs"\n    artifact_root.mkdir(exist_ok=False)\n    selected = ["recovery_best_epoch_090.pt"] + ["dependencies/"+name for name in INFERENCE_FILES]\n    with zipfile.ZipFile(archive_path) as archive:\n        require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP members")\n        for name in selected:\n            destination = artifact_root / name\n            require(destination.resolve().is_relative_to(artifact_root.resolve()), "Unsafe ZIP member")\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            # Only selected training inputs: reading to EOF verifies their ZIP CRC.\n            with archive.open(name) as source, destination.open("xb") as output:\n                for block in iter(lambda: source.read(1024*1024), b""): output.write(block)\n            require(sha256(destination) == config["selected_hashes"][name], "Selected input differs: "+name)\n    data_root = Path(config["data_root"])\n    require(data_root.is_dir() and data_root.resolve().is_relative_to(Path("/kaggle/input")), "COV data mount differs")\n    artifacts = load_inference_artifacts(artifact_root/"dependencies", config["dependency_hashes"])\n    require(hashlib.sha256(json.dumps(artifacts["prompt_ids"],separators=(",",":")).encode()).hexdigest() ==\n            config["prompt_ids_sha256"], "Prompt ordering fingerprint differs")\n    image_id = config["image_id"]\n    require(image_id == artifacts["train_ids"][0], "Fixed first saved training image differs")\n    image_path = resolve_training_image(data_root,image_id,artifacts["train_ids"])\n    image_sha256 = sha256(image_path)\n    log("strict_epoch90_restore")\n    decoder = restore_epoch90(artifact_root/"recovery_best_epoch_090.pt",device)\n    log("pinned_clip_load")\n    vision, preprocess, encoder = load_companion_clip(device,scratch/"clip_weights")\n    import clip\n    direct_url = metadata.distribution("clip").read_text("direct_url.json")\n    if direct_url:\n        vcs = json.loads(direct_url).get("vcs_info",{})\n        require(vcs.get("commit_id") == CLIP_COMMIT, "CLIP VCS provenance differs")\n    pipeline = FrozenImageToReport(vision,preprocess,decoder,artifacts["prompts"],artifacts["vocabulary"])\n    versions = {name:version(name) for name in\n                ("torch","torchvision","numpy","Pillow","clip","ftfy","regex","tqdm","wcwidth")}\n    before = dict(decoder=[p._version for p in decoder.parameters()],\n                  encoder=[p._version for p in vision.parameters()])\n    torch.cuda.synchronize()\n    inference_start = time.monotonic()\n    with Image.open(image_path) as image:\n        result = pipeline.generate_one(image,max_length=160,progress=log)\n    require(before == dict(decoder=[p._version for p in decoder.parameters()],\n                           encoder=[p._version for p in vision.parameters()]), "Parameter mutation detected")\n    result.update(status="PASS_SINGLE_TRAINING_IMAGE_DIAGNOSTIC",stage="zero_shot_image_to_report_diagnostic",\n        image_id=image_id,image_sha256=image_sha256,split="checkpoint_train",image_count=1,\n        checkpoint=dict(epoch=90,sha256=EPOCH90_SHA256,zip_sha256=config["zip_sha256"],\n                        reconstruction_bleu4=54.21140850351637,selection="available_physical_epoch90; historical85_missing"),\n        dependencies=versions,python=platform.python_version(),cuda=torch.version.cuda,\n        gpu=torch.cuda.get_device_name(0),encoder=encoder,artifact_hashes=artifacts["hashes"],\n        prompt_ids_sha256=config["prompt_ids_sha256"],config=config,\n        source_sha256={name:sha256(Path(__file__).parent/name) for name in\n                       ("zeromrg_gate1_runtime.py","zeromrg_gate4_runtime.py","gate4_diagnostic_worker.py")},\n        parameter_version_counters_unchanged=True,\n        total_worker_seconds=time.monotonic()-START,inference_seconds=time.monotonic()-inference_start,\n        scientific_limit="Historical encoder revision/cache alignment unverified; one image is not evaluation")\n    (run_root/"diagnostic.json").write_text(json.dumps(result,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n    log("diagnostic_saved")\n    print(json.dumps({k:result[k] for k in ("status","image_id","shapes","norms","output_tokens","report_text","termination")}),flush=True)\n\n\nif __name__ == "__main__":\n    START = time.monotonic()\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--run-root",required=True)\n    parser.add_argument("--config",required=True)\n    args = parser.parse_args()\n    destination = Path(args.run_root)\n    try:\n        diagnostic(destination,json.loads(Path(args.config).read_text(encoding="utf-8")))\n    except Exception as error:\n        # Never retry or change the architecture/environment automatically.\n        (destination/"failure.json").write_text(json.dumps(dict(status="FAILED",error_type=type(error).__name__,\n            error=str(error),elapsed_seconds=time.monotonic()-START),indent=2)+"\\n",encoding="utf-8")\n        log("failed_no_retry")\n        raise\n', 'zeromrg_gate4b_runtime.py': '"""Gate4B observations around the unchanged Gate4 inference path; CPU mocks on import."""\nimport torch\nfrom zeromrg_gate1_runtime import require\nfrom zeromrg_gate4_runtime import FrozenImageToReport, assert_frozen\n\n\ndef select_nonprompt_ids(train_ids, prompt_ids, limit=5):\n    require(type(limit) is int and 1 <= limit <= 5, "Select one to five images")\n    require(len(train_ids) == len(set(train_ids)) and len(prompt_ids) == len(set(prompt_ids)),\n            "Duplicate canonical train/prompt IDs")\n    require(set(prompt_ids) <= set(train_ids), "Prompts must be training-only")\n    selected = [identifier for identifier in train_ids if identifier not in set(prompt_ids)][:limit]\n    require(bool(selected), "No non-prompt training images available")\n    return selected\n\n\ndef compare_runs(runs):\n    """Compare observed tensors in float64 on CPU; this does not change conditioning."""\n    require(2 <= len(runs) <= 6, "Expected selected images plus one repeat")\n    unique = runs[:-1]\n    require(len({r["image_id"] for r in unique}) == len(unique) and\n            runs[-1]["image_id"] == unique[0]["image_id"], "Invalid unique/repeat order")\n    matrices = {}\n    pairs = []\n    for name in ("raw", "enriched"):\n        vectors = torch.stack([r[name].reshape(-1).double() for r in unique])\n        require(tuple(vectors.shape) == (len(unique), 768) and bool(torch.isfinite(vectors).all()) and\n                bool((vectors.norm(dim=-1) > 0).all()), "Invalid captured features")\n        unit = vectors / vectors.norm(dim=-1, keepdim=True)\n        matrices[name] = (unit @ unit.T).clamp(-1, 1).tolist()\n    for i in range(len(unique)):\n        for j in range(i+1, len(unique)):\n            a, b = unique[i], unique[j]\n            pairs.append(dict(image_ids=[a["image_id"], b["image_id"]],\n                raw_cosine=matrices["raw"][i][j], enriched_cosine=matrices["enriched"][i][j],\n                raw_features_exact_equal=torch.equal(a["raw"], b["raw"]),\n                enriched_features_exact_equal=torch.equal(a["enriched"], b["enriched"]),\n                raw_max_abs_difference=float((a["raw"]-b["raw"]).abs().max()),\n                enriched_max_abs_difference=float((a["enriched"]-b["enriched"]).abs().max()),\n                token_sequences_equal=a["result"]["output_tokens"] == b["result"]["output_tokens"],\n                report_text_equal=a["result"]["report_text"] == b["result"]["report_text"]))\n    first, repeat = unique[0], runs[-1]\n    raw_equal = torch.equal(first["raw"], repeat["raw"])\n    enriched_equal = torch.equal(first["enriched"], repeat["enriched"])\n    token_equal = first["result"]["output_tokens"] == repeat["result"]["output_tokens"]\n    report_equal = first["result"]["report_text"] == repeat["result"]["report_text"]\n    return dict(matrix_image_ids=[r["image_id"] for r in unique], pairwise_cosine=matrices,\n        pairs=pairs, different_raw_features_observed=any(not p["raw_features_exact_equal"] for p in pairs),\n        different_enriched_features_observed=any(not p["enriched_features_exact_equal"] for p in pairs),\n        different_token_sequences_observed=any(not p["token_sequences_equal"] for p in pairs),\n        different_reports_observed=any(not p["report_text_equal"] for p in pairs),\n        distinct_report_count=len({r["result"]["report_text"] for r in unique}),\n        determinism=dict(image_id=first["image_id"], total_executions_of_this_image=2,\n            raw_features_exact_equal=raw_equal, enriched_features_exact_equal=enriched_equal,\n            tokens_exact_equal=token_equal, report_exact_equal=report_equal,\n            raw_max_abs_difference=float((first["raw"]-repeat["raw"]).abs().max()),\n            enriched_max_abs_difference=float((first["enriched"]-repeat["enriched"]).abs().max()),\n            all_observed_outputs_exact_equal=raw_equal and enriched_equal and token_equal and report_equal),\n        sensitivity_conclusion=("REPORT_CHANGES_OBSERVED" if any(not p["report_text_equal"] for p in pairs)\n                                else "NO_REPORT_CHANGES_OBSERVED_IN_THIS_SMALL_SET"),\n        clinical_correctness_assessed=False)\n\n\n@torch.inference_mode()\ndef diagnose_images(vision, preprocess, decoder, prompts, vocabulary, image_ids,\n                    train_ids, prompt_ids, open_image, progress=None, on_record=None):\n    require(image_ids == select_nonprompt_ids(train_ids, prompt_ids), "Fixed saved selection differs")\n    progress = progress or (lambda phase: None)\n    frozen_prompt_copy = prompts.detach().clone()\n    require(not prompts.requires_grad, "Prompt bank must be frozen")\n    versions = [[p._version for p in module.parameters()] for module in (vision, decoder)]\n    runs = []\n    for index, image_id in enumerate(image_ids + [image_ids[0]]):\n        repeat = index == len(image_ids)\n        progress("image_"+str(index+1)+("_repeat_" if repeat else "_")+image_id)\n        captured = {}\n        def raw_hook(module, inputs, output):\n            require("raw" not in captured, "More than one encoder call per image")\n            captured["raw"] = output.detach().float().cpu().clone()\n        def condition_hook(module, inputs):\n            if "enriched" not in captured:\n                captured["enriched"] = inputs[0].detach().float().cpu().clone()\n        encoder_hook = vision.register_forward_hook(raw_hook)\n        decoder_hook = decoder.register_forward_pre_hook(condition_hook)\n        try:\n            pipeline = FrozenImageToReport(vision, preprocess, decoder, prompts, vocabulary)\n            with open_image(image_id) as image:\n                result = pipeline.generate_one(image, max_length=160, progress=progress)\n        finally:\n            encoder_hook.remove()\n            decoder_hook.remove()\n        for key in ("raw", "enriched"):\n            require(key in captured and tuple(captured[key].shape) == (1, 768) and\n                    bool(torch.isfinite(captured[key]).all()), "Missing/invalid feature capture: "+key)\n        for module in (vision, decoder): assert_frozen(module)\n        require(versions == [[p._version for p in module.parameters()] for module in (vision, decoder)],\n                "Model parameter mutation detected")\n        require(torch.equal(prompts, frozen_prompt_copy), "Original prompt bank mutated")\n        runs.append(dict(image_id=image_id, repeat=repeat, raw=captured["raw"],\n                         enriched=captured["enriched"], result=result))\n        if on_record is not None:\n            on_record(index, dict(image_id=image_id,report_id=image_id,split="checkpoint_train",\n                in_prompt_bank=False,repeat=repeat,raw_embedding=captured["raw"].tolist(),\n                enriched_embedding=captured["enriched"].tolist(),**result))\n        progress("generation_completed_"+str(index+1))\n    comparisons = compare_runs(runs)\n    # Features are small (<=6 x2 x768), retained for independent cosine/difference verification.\n    records = [dict(image_id=r["image_id"], report_id=r["image_id"], split="checkpoint_train",\n                    in_prompt_bank=False, repeat=r["repeat"], raw_embedding=r["raw"].tolist(),\n                    enriched_embedding=r["enriched"].tolist(), **r["result"]) for r in runs]\n    return dict(status="COMPLETED_IMAGE_CONDITIONING_DIAGNOSTIC", image_count=len(image_ids),\n        generation_count=len(records), split="checkpoint_train", records=records, comparisons=comparisons,\n        all_modules_frozen=True, gradients_disabled=True, parameter_version_counters_unchanged=True,\n        prompt_bank_unchanged=True, optimizer_created=False, paired_supervision_used=False,\n        test_references_accessed=False, test_scoring=False)\n', 'gate4b_diagnostic_worker.py': '"""Future Kaggle-only Gate4B worker; no execution or downloads on import."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport sys\nimport tempfile\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom importlib import metadata\nfrom contextlib import nullcontext\nfrom zeromrg_gate4b_progress import StageProgress, run_stage, utc_now\n\nPROGRESS = None\nif __name__ == "__main__":\n    # Export evidence before Torch imports, CUDA initialization, network or mount I/O.\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--run-root", required=True)\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n    destination = Path(args.run_root)\n    if not destination.is_dir() or not destination.resolve().is_relative_to(Path("/kaggle/working").resolve()):\n        raise ValueError("Kaggle working output required")\n    PROGRESS = StageProgress(destination)\n    PROGRESS.emit("worker_start")\n\nwith PROGRESS.stage("runtime_imports") if PROGRESS else nullcontext():\n    import gate4_diagnostic_worker as base\n    from gate4_diagnostic_worker import version, run_checked\n    from zeromrg_gate1_runtime import require, sha256\n    from zeromrg_gate4_runtime import (CLIP_COMMIT, EPOCH90_SHA256, INFERENCE_FILES,\n        load_companion_clip, load_inference_artifacts, restore_epoch90)\n    from zeromrg_gate4b_runtime import diagnose_images, select_nonprompt_ids\n    from zeromrg_cov_data_resolver import resolve_cov_dataset, verify_checkpoint_manifests, CovDataError\n\n\ndef log(phase, **details):\n    if PROGRESS:\n        return PROGRESS.emit(phase, **details)\n    return base.log(phase)\n\n\ndef diagnostic(run_root, config):\n    import torch\n    contract_path = run_root / "dataset-contract.json"\n    require(sha256(contract_path) == config["dataset_contract_sha256"], "Dataset identity contract hash differs")\n    contract = json.loads(contract_path.read_text(encoding="utf-8"))\n    require(1 <= config["max_images"] <= 5 and config["max_tokens"] == 160 and\n            config["training_enabled"] is False and config["test_inference_enabled"] is False, "Diagnostic scope differs")\n    require(Path("/kaggle/input").is_dir() and run_root.resolve().is_relative_to(Path("/kaggle/working")),\n            "Kaggle-only diagnostic")\n    require(run_stage(PROGRESS,"cuda_availability",torch.cuda.is_available), "Kaggle GPU required; no local fallback")\n    device = torch.device("cuda:0")\n    log("provided_environment_checks")\n    # These are the observed baseline stack. Changed images stop for review, not upgrade.\n    require(torch.__version__ == "2.11.0+cu128" and torch.version.cuda == "12.8", "Torch/CUDA drift; review required")\n    require(version("numpy") == "2.1.3" and version("Pillow") == "12.3.0", "NumPy/Pillow drift")\n    require(version("torchvision") is not None and version("torchvision").split("+")[0] == "0.26.0",\n            "Expected torchvision0.26.0 companion; do not auto-install Torch/torchvision")\n    with PROGRESS.stage("vision_support_imports") if PROGRESS else nullcontext():\n        import torchvision\n        from PIL import Image\n    # /tmp is ephemeral and excluded from Kaggle saved notebook outputs.\n    scratch = Path(tempfile.mkdtemp(prefix="zeromrg_gate4_",dir="/tmp"))\n    package_root = scratch / "inference_packages"\n    package_root.mkdir(exist_ok=False)\n    required = dict(ftfy="6.3.1", regex="2025.11.3", tqdm="4.67.1", wcwidth="0.9.2")\n    missing = [line for line in config["support_lock"].splitlines()\n               if version(line.split("==")[0]) != required[line.split("==")[0]]]\n    if missing:\n        lock = run_root / "support.lock"\n        lock.write_text("\\n".join(missing)+"\\n", encoding="utf-8")\n        run_stage(PROGRESS,"support_package_setup",run_checked,[sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--require-hashes","--only-binary=:all:","--retries","0","--timeout","45",\n            "-r",str(lock)])\n    sys.path.insert(0,str(package_root))\n    for name, expected in required.items(): require(version(name) == expected, "Support package pin mismatch: "+name)\n    if version("clip") is None:\n        require(version("setuptools") is not None and version("wheel") is not None,\n                "Provided CLIP build tools missing; no implicit build-tool install")\n        run_stage(PROGRESS,"clip_package_setup",run_checked,[sys.executable,"-m","pip","install","--target",str(package_root),\n            "--no-deps","--no-build-isolation","--retries","0","--timeout","45",\n            "clip @ git+https://github.com/openai/CLIP.git@"+CLIP_COMMIT])\n    # The exact source hash is checked by load_companion_clip, even for provided clip.\n    log("verified_checkpoint_attachment")\n    candidates = list(Path("/kaggle/input").rglob(config["zip_name"]))\n    require(len(candidates) == 1, "Expected exactly one verified baseline ZIP attachment")\n    archive_path = candidates[0]\n    require(run_stage(PROGRESS,"baseline_zip_sha256",sha256,archive_path) == config["zip_sha256"], "Baseline ZIP differs")\n    artifact_root = scratch / "selected_inputs"\n    artifact_root.mkdir(exist_ok=False)\n    selected = ["recovery_best_epoch_090.pt"] + ["dependencies/"+name for name in INFERENCE_FILES]\n    with zipfile.ZipFile(archive_path) as archive:\n        require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP members")\n        run_stage(PROGRESS,"checkpoint_manifest_identity",verify_checkpoint_manifests,archive,contract)\n        for name in selected:\n            log("selected_input_extract_start", member=name)\n            destination = artifact_root / name\n            require(destination.resolve().is_relative_to(artifact_root.resolve()), "Unsafe ZIP member")\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            # Only selected training inputs: reading to EOF verifies their ZIP CRC.\n            with archive.open(name) as source, destination.open("xb") as output:\n                for block in iter(lambda: source.read(1024*1024), b""): output.write(block)\n            require(run_stage(PROGRESS,"selected_input_sha256",sha256,destination) == config["selected_hashes"][name], "Selected input differs: "+name)\n            log("selected_input_extract_complete", member=name)\n    artifacts = run_stage(PROGRESS,"train_artifacts_load",load_inference_artifacts,artifact_root/"dependencies",config["dependency_hashes"])\n    require(hashlib.sha256(json.dumps(artifacts["prompt_ids"],separators=(",",":")).encode()).hexdigest() ==\n            config["prompt_ids_sha256"], "Prompt ordering fingerprint differs")\n    image_ids = select_nonprompt_ids(artifacts["train_ids"], artifacts["prompt_ids"])\n    require(image_ids == config["image_ids"] and len(image_ids) == config["max_images"] and\n            config["repeat_image_id"] == image_ids[0] and config["max_generations"] == len(image_ids)+1,\n            "Fixed train-minus-prompts selection/repeat differs")\n    require(len(artifacts["train_ids"])-len(artifacts["prompt_ids"]) == config["eligible_count"],\n            "Eligible membership differs")\n    log("cov_content_bound_data_resolution")\n    resolved = run_stage(PROGRESS,"dataset_resolution",resolve_cov_dataset,Path("/kaggle/input"), config["data_root"], contract,\n                         artifacts["train_ids"], artifacts["prompt_ids"], scratch, progress=log)\n    (run_root/"data_resolution.json").write_text(json.dumps(resolved["evidence"],indent=2)+"\\n",encoding="utf-8")\n    print(json.dumps(dict(phase="cov_data_resolved", **resolved["evidence"])),flush=True)\n    paths = {identifier:resolved["images"][identifier] for identifier in image_ids}\n    image_hashes = {identifier:sha256(filename) for identifier,filename in paths.items()}\n    require(image_hashes == config["image_sha256"], "Selected image bytes differ from local raw archive")\n    log("strict_epoch90_restore")\n    decoder = run_stage(PROGRESS,"decoder_restore",restore_epoch90,artifact_root/"recovery_best_epoch_090.pt",device)\n    log("pinned_clip_load")\n    vision, preprocess, encoder = run_stage(PROGRESS,"clip_initialize",load_companion_clip,device,scratch/"clip_weights")\n    direct_url = metadata.distribution("clip").read_text("direct_url.json")\n    if direct_url:\n        require(json.loads(direct_url).get("vcs_info",{}).get("commit_id") == CLIP_COMMIT,\n                "CLIP VCS provenance differs")\n    log("controlled_images_and_one_repeat")\n    torch.cuda.synchronize()\n    torch.cuda.reset_peak_memory_stats(device)\n    inference_start = time.monotonic()\n    def save_completed_generation(index, record):\n        evidence = dict(status="COMPLETED_DIAGNOSTIC_IMAGE",run_id=config["run_id"],\n                        checkpoint_sha256=EPOCH90_SHA256,generation_index=index+1,\n                        image_sha256=image_hashes[record["image_id"]],record=record)\n        with (run_root/("generation_"+str(index+1).zfill(2)+".json")).open("x",encoding="utf-8") as stream:\n            json.dump(evidence,stream,indent=2,allow_nan=False)\n        log("generation_saved", generation=index+1, image_id=record["image_id"])\n    result = run_stage(PROGRESS,"six_generations",diagnose_images,vision,preprocess,decoder,artifacts["prompts"],artifacts["vocabulary"],\n        image_ids,artifacts["train_ids"],artifacts["prompt_ids"],\n        lambda identifier: Image.open(paths[identifier]),progress=log,on_record=save_completed_generation)\n    torch.cuda.synchronize()\n    inference_seconds = time.monotonic()-inference_start\n    require(sha256(artifact_root/"recovery_best_epoch_090.pt") == EPOCH90_SHA256,\n            "Checkpoint file changed")\n    require(sha256(artifact_root/"dependencies/mclip_prompt_bank.pt") ==\n            config["dependency_hashes"]["mclip_prompt_bank.pt"], "Prompt file changed")\n    for record in result["records"]: record["image_sha256"] = image_hashes[record["image_id"]]\n    result.update(run_id=config["run_id"],image_ids=image_ids,eligible_count=config["eligible_count"],\n        checkpoint=dict(epoch=90,sha256=EPOCH90_SHA256,zip_sha256=config["zip_sha256"],\n                        selection="available_physical_epoch90; historical85_missing"),\n        encoder=encoder,dependencies={name:version(name) for name in\n            ("torch","torchvision","numpy","Pillow","clip","ftfy","regex","tqdm","wcwidth")},\n        python=platform.python_version(),cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),\n        artifact_hashes=artifacts["hashes"],prompt_ids_sha256=config["prompt_ids_sha256"],config=config,\n        source_sha256={name:sha256(Path(__file__).parent/name) for name in config["source_files"]},\n        dataset_resolution=resolved["evidence"],dataset_contract_sha256=config["dataset_contract_sha256"],\n        inference_seconds=inference_seconds,total_worker_seconds=time.monotonic()-base.START,\n        gpu_memory=dict(peak_allocated_bytes=torch.cuda.max_memory_allocated(device),\n                       peak_reserved_bytes=torch.cuda.max_memory_reserved(device),\n                       scope="after model load; reset before selected images plus repeat"),\n        numerical_settings=dict(cudnn_benchmark=torch.backends.cudnn.benchmark,\n            cudnn_deterministic=torch.backends.cudnn.deterministic,\n            deterministic_algorithms=torch.are_deterministic_algorithms_enabled(),\n            matmul_allow_tf32=torch.backends.cuda.matmul.allow_tf32),\n        scientific_limit="Training-only small diagnostic, not clinical scoring; decoder trained on these reports; historical encoder/cache revision unverified")\n    require(result["generation_count"] <= 6 and result["image_count"] <= 5, "Generation bound exceeded")\n    (run_root/"diagnostic.json").write_text(json.dumps(result,indent=2,allow_nan=False)+"\\n",encoding="utf-8")\n    log("diagnostic_saved")\n    print(json.dumps(dict(status=result["status"],image_ids=image_ids,comparisons=result["comparisons"])),flush=True)\n\n\nif __name__ == "__main__":\n    base.START = PROGRESS.started\n    base.log = log  # Observe the unchanged Gate4 dependency helper in this worker only.\n    try:\n        diagnostic(destination,json.loads(Path(args.config).read_text(encoding="utf-8")))\n    except Exception as error:\n        if isinstance(error, CovDataError):\n            (destination/"data_resolution_failure.json").write_text(\n                json.dumps(error.diagnostic,indent=2)+"\\n",encoding="utf-8")\n        (destination/"failure.json").write_text(json.dumps(dict(status="FAILED",error_type=type(error).__name__,\n            phase="worker_failure",state="failed",timestamp_utc=utc_now().isoformat(),\n            error=str(error),elapsed_seconds=time.monotonic()-base.START,no_retry=True),indent=2)+"\\n",encoding="utf-8")\n        log("failed_no_retry",state="failed",error_type=type(error).__name__)\n        raise\n', 'zeromrg_cov_data_resolver.py': '"""Content-bound COV attachment resolution. No models, network or reference scoring."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\nIMAGE_EXTENSIONS = {\'.png\', \'.jpg\', \'.jpeg\', \'.bmp\', \'.tif\', \'.tiff\', \'.webp\'}\nATTACHMENT = dict(ref=\'inshasadaf114/dataset\', dataset_id=11916862)\n\n\nclass CovDataError(ValueError):\n    def __init__(self, code, **details):\n        self.diagnostic = dict(code=code, **details)\n        super().__init__(json.dumps(self.diagnostic, sort_keys=True))\n\n\ndef ensure(condition, code, **details):\n    if not condition:\n        raise CovDataError(code, **details)\n\n\ndef stream_hash(stream):\n    h = hashlib.sha256()\n    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n        h.update(block)\n    return h.hexdigest()\n\n\ndef file_hash(path):\n    with Path(path).open(\'rb\') as stream:\n        return stream_hash(stream)\n\n\ndef ids_hash(ids):\n    return hashlib.sha256(json.dumps(ids, separators=(\',\', \':\')).encode()).hexdigest()\n\n\ndef validate_contract(contract, train_ids, prompt_ids):\n    ensure(contract[\'schema\'] == \'zeromrg_cov_identity_v1\' and\n           contract[\'attachment\'] == ATTACHMENT, \'ATTACHMENT_CONTRACT_MISMATCH\')\n    groups = contract[\'splits\']\n    for name, count in dict(validated=714, train=571, validation=71, test=72, prompt=500).items():\n        ids = groups[name]\n        ensure(len(ids) == len(set(ids)) == count, \'CANONICAL_ID_COUNT\', split=name, expected=count)\n        ensure(all(isinstance(i, str) and PurePosixPath(i).name == i and\n                   i not in (\'.\', \'..\') and \'\\\\\' not in i for i in ids), \'UNSAFE_IMAGE_ID\', split=name)\n    train, validation, test = (set(groups[n]) for n in (\'train\', \'validation\', \'test\'))\n    ensure(not (train & validation or train & test or validation & test) and\n           train | validation | test == set(groups[\'validated\']), \'SPLIT_MEMBERSHIP_MISMATCH\')\n    ensure(groups[\'train\'] == list(train_ids), \'TRAIN_ORDER_OR_MEMBERSHIP_MISMATCH\')\n    ensure(groups[\'prompt\'] == list(prompt_ids) and set(prompt_ids) <= train and\n           ids_hash(prompt_ids) == contract[\'prompt_ids_sha256\'], \'PROMPT_BANK_MEMBERSHIP_OR_ORDER_MISMATCH\')\n    for name, ids in groups.items():\n        ensure(ids_hash(ids) == contract[\'split_ids_sha256\'][name],\n               \'SPLIT_ID_FINGERPRINT_MISMATCH\', split=name)\n    ensure(set(groups[\'validated\']) <= set(contract[\'images\']) and\n           len(contract[\'images\']) == contract[\'physical_image_count\'] == 746,\n           \'IMAGE_CONTRACT_POPULATION_MISMATCH\')\n\n\ndef verify_checkpoint_manifests(archive, contract):\n    """Only hash report-bearing CSV bytes; no held-out report parsing or scoring."""\n    for name, expected in contract[\'manifest_sha256\'].items():\n        member = \'dependencies/\' + name\n        ensure(archive.namelist().count(member) == 1, \'CHECKPOINT_MANIFEST_MISSING_OR_AMBIGUOUS\', file=name)\n        with archive.open(member) as stream:\n            actual = stream_hash(stream)  # reading to EOF also verifies ZIP CRC\n        ensure(actual == expected, \'CHECKPOINT_MANIFEST_HASH_MISMATCH\', file=name,\n               expected=expected, actual=actual)\n\n\ndef _index(entries):\n    index = {}\n    for name, entry in entries:\n        ensure(name not in index, \'DUPLICATE_IMAGE_IDENTIFIER\', image_id=name)\n        index[name] = entry\n    return index\n\n\ndef _bounded_entries(root):\n    entries = []\n    for path in root.rglob(\'*\'):\n        ensure(len(entries) < 10000, \'DATASET_SCAN_BOUND\', root=str(root))\n        entries.append(path)\n    return entries\n\n\ndef _validate_population(index, contract, hash_image, progress=None):\n    expected = contract[\'images\']\n    missing, extra = sorted(set(expected) - set(index)), sorted(set(index) - set(expected))\n    ensure(not missing and not extra, \'IMAGE_INVENTORY_MISMATCH\', missing_count=len(missing),\n           extra_count=len(extra), missing=missing[:10], extra=extra[:10])\n    for count, identifier in enumerate(sorted(expected),1):\n        actual = hash_image(index[identifier])\n        ensure(actual == expected[identifier][\'sha256\'], \'IMAGE_CONTENT_MISMATCH\',\n               image_id=identifier, expected=expected[identifier][\'sha256\'], actual=actual)\n        if progress and (count==1 or count%100==0 or count==len(expected)):\n            progress(\'dataset_image_hash_progress\', verified=count, total=len(expected))\n\n\ndef _directory_candidate(report, attachment_root, contract, progress=None):\n    root = report.parent.resolve()\n    ensure(report.resolve().is_relative_to(attachment_root), \'REPORT_ESCAPES_ATTACHMENT\', path=str(report))\n    ensure(file_hash(report) == contract[\'annotation\'][\'sha256\'], \'ANNOTATION_CONTENT_MISMATCH\', path=str(report))\n    files = _bounded_entries(root)\n    images = [p for p in files if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]\n    for p in images:\n        ensure(p.resolve().is_relative_to(root) and p.resolve().is_relative_to(attachment_root),\n               \'IMAGE_ESCAPES_DATA_ROOT\', image_id=p.name)\n    index = _index((p.name, p) for p in images)\n    _validate_population(index, contract, file_hash, progress)\n    return dict(kind=\'directory\', root=root, report=report, images=index)\n\n\ndef _archive_candidate(path, contract, progress=None):\n    with zipfile.ZipFile(path) as archive:\n        entries = archive.infolist()\n        ensure(len(entries) <= 10000 and len({e.filename for e in entries}) == len(entries),\n               \'ARCHIVE_ENTRY_BOUND_OR_DUPLICATE\', path=str(path))\n        reports = [e for e in entries if not e.is_dir() and PurePosixPath(e.filename).name == contract[\'annotation\'][\'filename\']]\n        ensure(len(reports) == 1, \'ARCHIVE_REPORT_MISSING_OR_AMBIGUOUS\', path=str(path), count=len(reports))\n        prefix = PurePosixPath(reports[0].filename).parent\n        for e in entries:\n            name = PurePosixPath(e.filename)\n            ensure(not name.is_absolute() and \'..\' not in name.parts and \'\\\\\' not in e.filename,\n                   \'UNSAFE_ARCHIVE_MEMBER\', member=e.filename)\n            ensure((e.external_attr >> 16) & 0o170000 != 0o120000,\n                   \'ARCHIVE_SYMLINK_REJECTED\', member=e.filename)\n        with archive.open(reports[0]) as stream:\n            ensure(stream_hash(stream) == contract[\'annotation\'][\'sha256\'], \'ANNOTATION_CONTENT_MISMATCH\', path=str(path))\n        images = [e for e in entries if not e.is_dir() and\n                  PurePosixPath(e.filename).is_relative_to(prefix) and\n                  PurePosixPath(e.filename).suffix.lower() in IMAGE_EXTENSIONS]\n        index = _index((PurePosixPath(e.filename).name, e) for e in images)\n        def hash_image(entry):\n            ensure(entry.file_size == contract[\'images\'][PurePosixPath(entry.filename).name][\'bytes\'],\n                   \'IMAGE_SIZE_MISMATCH\', image_id=PurePosixPath(entry.filename).name)\n            with archive.open(entry) as stream:\n                return stream_hash(stream)\n        _validate_population(index, contract, hash_image, progress)\n        return dict(kind=\'archive\', root=path.resolve(), report=reports[0].filename,\n                    images={k: v.filename for k, v in index.items()})\n\n\ndef resolve_cov_dataset(input_root, configured_root, contract, train_ids, prompt_ids, scratch, progress=None):\n    """Resolve only approved owner/slug or Kaggle short-slug mounts, bound by bytes.\n\n    Multiple candidates fail even if one validates. Ordering never chooses a root.\n    CSV contents are hashed, never parsed here; no clinical target reaches decoding.\n    """\n    validate_contract(contract, train_ids, prompt_ids)\n    input_root = Path(input_root).resolve()\n    ensure(input_root.is_dir(), \'INPUT_ROOT_MISSING\', path=str(input_root))\n    ref = contract[\'attachment\'][\'ref\']; owner, slug = ref.split(\'/\')\n    layouts = [input_root/\'datasets\'/owner/slug, input_root/slug]\n    mounts = []\n    for path in layouts:\n        if path.is_dir():\n            resolved = path.resolve()\n            ensure(resolved.is_relative_to(input_root), \'ATTACHMENT_ESCAPES_INPUT\', path=str(path))\n            if resolved not in mounts:\n                mounts.append(resolved)\n    ensure(mounts, \'APPROVED_ATTACHMENT_NOT_FOUND\', attachment=ref,\n           expected_layouts=[str(p) for p in layouts], configured_root=str(configured_root),\n           observed_top_directories=sorted(p.name for p in input_root.iterdir() if p.is_dir())[:40])\n    candidates = []\n    for mount in mounts:\n        # rglob is layout-independent; bounds reject unexpected enormous attachments.\n        entries = _bounded_entries(mount)\n        for path in entries:\n            if not path.is_file():\n                continue\n            if path.name == contract[\'annotation\'][\'filename\']:\n                candidates.append((\'directory\', path, mount))\n            elif path.name == contract[\'archive_filename\']:\n                ensure(path.resolve().is_relative_to(mount), \'ARCHIVE_ESCAPES_ATTACHMENT\', path=str(path))\n                candidates.append((\'archive\', path, mount))\n    unique = {(kind, path.resolve()): (kind, path, mount) for kind, path, mount in candidates}\n    candidates = list(unique.values())\n    ensure(len(candidates) == 1, \'DATASET_ROOT_MISSING_OR_AMBIGUOUS\', count=len(candidates),\n           candidates=[str(path) for _, path, _ in candidates][:10], attachment=ref,\n           action=\'Keep one original COV annotation tree OR its original archive inside the approved attachment; do not substitute data\')\n    kind, path, mount = candidates[0]\n    if progress: progress(\'dataset_candidate_selected\',kind=kind,path=str(path))\n    try:\n        resolved = (_directory_candidate(path, mount, contract, progress) if kind == \'directory\'\n                    else _archive_candidate(path, contract, progress))\n    except CovDataError:\n        raise\n    except (OSError, zipfile.BadZipFile, RuntimeError) as error:\n        raise CovDataError(\'DATASET_READ_OR_CRC_FAILURE\', path=str(path), error_type=type(error).__name__) from None\n    evidence = dict(status=\'PASS_CONTENT_BOUND_DATA_RESOLUTION\', attachment=contract[\'attachment\'],\n                    attachment_root=str(mount), kind=kind, discovered_source=str(resolved[\'root\']),\n                    configured_root=str(configured_root), configured_root_exists=Path(configured_root).is_dir(),\n                    annotation_sha256=contract[\'annotation\'][\'sha256\'], physical_images_verified=746,\n                    valid_pairs=714, split_counts=dict(train=571, validation=71, test=72),\n                    prompt_count=500, prompt_ids_sha256=contract[\'prompt_ids_sha256\'],\n                    report_text_parsed=False, images_decoded=False, scoring=False)\n    if kind == \'archive\':\n        destination = Path(scratch)/\'verified_cov_images\'\n        destination.mkdir(exist_ok=False, parents=True)\n        # Materialize canonical images only into ephemeral scratch, never into input.\n        with zipfile.ZipFile(path) as archive:\n            for count, identifier in enumerate(contract[\'splits\'][\'validated\'],1):\n                with archive.open(resolved[\'images\'][identifier]) as source, (destination/identifier).open(\'xb\') as out:\n                    shutil.copyfileobj(source, out)\n                ensure(file_hash(destination/identifier) == contract[\'images\'][identifier][\'sha256\'],\n                       \'EXTRACTED_IMAGE_CONTENT_MISMATCH\', image_id=identifier)\n                if progress and (count==1 or count%100==0 or count==714):\n                    progress(\'dataset_archive_extract_progress\',completed=count,total=714)\n        images = {i: destination/i for i in contract[\'splits\'][\'validated\']}\n        root = destination\n    else:\n        root, images = resolved[\'root\'], resolved[\'images\']\n    evidence[\'resolved_image_root\'] = str(root)\n    return dict(root=root, images=images, evidence=evidence)\n', 'zeromrg_gate4b_progress.py': '"""Standard-library stage evidence; no model imports, downloads or execution on import."""\nimport json\nimport os\nimport time\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n\ndef utc_now():\n    return datetime.now(timezone.utc)\n\n\nclass StageProgress:\n    def __init__(self, directory, clock=time.monotonic, utc_clock=utc_now):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.clock = clock\n        self.utc_clock = utc_clock\n        self.started = clock()\n        self.sequence = 0\n\n    def emit(self, phase, state=\'event\', **details):\n        self.sequence += 1\n        event = dict(phase=phase, state=state, sequence=self.sequence,\n                     timestamp_utc=self.utc_clock().isoformat(),\n                     elapsed_seconds=round(self.clock()-self.started, 6), **details)\n        line = json.dumps(event, allow_nan=False)\n        with (self.directory/\'stage_progress.jsonl\').open(\'a\', encoding=\'utf-8\', newline=\'\\n\') as stream:\n            stream.write(line+\'\\n\')\n            stream.flush()\n            os.fsync(stream.fileno())\n        temporary = self.directory/\'latest_progress.json.tmp\'\n        temporary.write_text(line+\'\\n\', encoding=\'utf-8\', newline=\'\\n\')\n        temporary.replace(self.directory/\'latest_progress.json\')\n        print(line, flush=True)\n        return event\n\n    @contextmanager\n    def stage(self, phase):\n        start = self.clock()\n        self.emit(phase, state=\'start\')\n        try:\n            yield\n        except BaseException as error:\n            # Exception text can contain URLs or data; retain class only in progress.\n            self.emit(phase, state=\'failed\', duration_seconds=self.clock()-start,\n                      error_type=type(error).__name__)\n            raise\n        else:\n            self.emit(phase, state=\'complete\', duration_seconds=self.clock()-start)\n\n\ndef run_stage(observer, phase, operation, *args, **kwargs):\n    if observer is None:\n        return operation(*args, **kwargs)\n    with observer.stage(phase):\n        return operation(*args, **kwargs)\n'}
SOURCE_HASHES = {'zeromrg_gate1_runtime.py': 'e897aa5e800ce18911744cc13e16b75b239477a4edb8bc1df0f8e9ab3f9e038a', 'zeromrg_gate4_runtime.py': 'c9117e4106168a907da3c52e7c99c0ca34655abf88b98e60842b241908c679d8', 'gate4_diagnostic_worker.py': 'e1ba00f6dd02a1ddfccd85c158283857d210b7bdee6148c4b528ffcf8dcc0812', 'zeromrg_gate4b_runtime.py': '6c5a86d875c1f4ccd4d88d6760b08902333ba9c3ab4c57246e027c1cf346e2a3', 'gate4b_diagnostic_worker.py': '4c3116e0ba8251e59b5ecf8afb72266cf8124264b7f76fd117a6ef41e6b69701', 'zeromrg_cov_data_resolver.py': '4083bfbe93f9b725f223bee2cc211f0770974be5b3ddcb42fb1b661123f4d2fe', 'zeromrg_gate4b_progress.py': 'e7c163583cf07a69ded4bf45e43a5ae8e09a154508dc7895a9b17b81ab3f0e3e'}
CONFIG = {'run_id': 'ZM-010_GATE4B_STAGE_PROGRESS_V3', 'data_root': '/kaggle/input/datasets/inshasadaf114/dataset/COV-CTR with English reports', 'zip_name': 'recovery_export_epoch_090_54468b70.zip', 'zip_sha256': 'dec3cd3391e2c107a7c9ed6209997f00554cd47cb0940ca339a74446253680b1', 'selected_hashes': {'recovery_best_epoch_090.pt': '6166e36198737bfbe9e0c35f03f0b320c44e76f3e05068ef6299d74e9179c6b2', 'dependencies/train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'dependencies/prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'dependencies/decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'dependencies/mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}, 'dependency_hashes': {'train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de', 'decoder_vocabulary.json': 'c0e169c3558fe0cbf2e36431281aa223f64e2e4fb88f7361e23b33e12bcc3ad5', 'mclip_prompt_bank.pt': '3cb770a2a470c70d95952828cf23abc088250571897b960a2a39e9bae6c8e1ce'}, 'prompt_ids_sha256': '2e1e001a0d2e45d35e8ee16522e4347bc9ee1bccb67ff9532d8c21cb9c2f65d0', 'support_lock': 'ftfy==6.3.1 --hash=sha256:7c70eb532015cd2f9adb53f101fb6c7945988d023a085d127d1573dc49dd0083\nregex==2025.11.3 --hash=sha256:4aecb6f461316adf9f1f0f6a4a1a3d79e045f9b71ec76055a791affa3b285850\ntqdm==4.67.1 --hash=sha256:26445eca388f82e72884e0d580d5464cd801a3ea01e63e5601bdff9ba6a48de2\nwcwidth==0.9.2 --hash=sha256:751bef0ab404b6a1dc028b56b4b85d46486be1c55833f80da533e42dc691f389\n', 'hard_timeout_seconds': 1800, 'max_images': 5, 'max_tokens': 160, 'training_enabled': False, 'test_inference_enabled': False, 'approval_scope': 'D-021 cancelled-v2 local diagnosis/progress preparation only; no submission or cancellation; new execution approval required', 'image_ids': ['25%2.jpg', 'PIIS0140673620303603%5.png', '1094.png', '2%0.jpg', '1868.png'], 'repeat_image_id': '25%2.jpg', 'max_generations': 6, 'eligible_count': 71, 'image_sha256': {'25%2.jpg': '1495a7e9ed347b7f1773d6c2b144177abce0c742b1ecbd45ebfba176f1a8e0fa', 'PIIS0140673620303603%5.png': '68a8a202e2559d2c1f4b3455672170dae3b01692542714193f29a08c3e01475f', '1094.png': '0ee58d47f00c4a7092ca140576b75a550ee94f702810bea318f1a5c4a714ea4b', '2%0.jpg': '415a614f360c8f7e350cda2ad23212add7e00d8e8d4236b9f6cefab930ba2b87', '1868.png': '0ca6766ae838f16eecfba0da8e5ecad7868cc50f9dd8291854793d17503ea0f7'}, 'source_files': ['zeromrg_gate1_runtime.py', 'zeromrg_gate4_runtime.py', 'gate4_diagnostic_worker.py', 'zeromrg_gate4b_runtime.py', 'gate4b_diagnostic_worker.py', 'zeromrg_cov_data_resolver.py', 'zeromrg_gate4b_progress.py'], 'dataset_contract_sha256': '0efeac6636bb971f27580cbe89eb792620feffeb13b42900c625b8309b277772', 'approved_dataset_attachment': {'ref': 'inshasadaf114/dataset', 'dataset_id': 11916862}}
DATASET_CONTRACT = {'schema': 'zeromrg_cov_identity_v1', 'attachment': {'ref': 'inshasadaf114/dataset', 'dataset_id': 11916862}, 'archive_filename': 'COV-CTR with English reports.zip', 'annotation': {'filename': 'reports_ZH_EN.csv', 'sha256': 'bfb892f3453276f652d8a14dcfdc6f215dd3eecd435394a5c05bdb8209a04489', 'bytes': 578429, 'columns': ['image_id', 'findings', 'terminologies', 'COVID', 'impression', 'reports_En'], 'row_count': 726}, 'images': {'0.jpg': {'bytes': 52972, 'sha256': '27fe403dea74c1171fef69c7f40ee7a1c43aab0069867a3d0aa27def214fad38'}, '1%0.jpg': {'bytes': 39783, 'sha256': '74fb74fbd22f21db117ed1ad3d3fc9242596dd91d47f908a2ca71298fb8c5524'}, '1%1.jpg': {'bytes': 43673, 'sha256': 'c59ea3755f0b08fa6af848460c6a91bca58c5917c24a1c50c5076556f640e0f6'}, '1%2.jpg': {'bytes': 54492, 'sha256': '606f88d834cf7ed92833cfcf1da572932b1b5a7e6b2c67b7e4c49c591a0f9b2a'}, '10%0.jpg': {'bytes': 47558, 'sha256': 'd59a2bf5b6c6b97b066f6b2bc8c23dfcd26831969350bdcb47a841eb5d712a44'}, '10%1.jpg': {'bytes': 48978, 'sha256': '6ac79b27c99d53b817f57443caad71577d1c23be7a697faa727d66c9dab256fb'}, '10%2.jpg': {'bytes': 51807, 'sha256': 'f9b5d160c3e4cd96af4b3b994052be69d90a0cc70374f2c46bfef376b23c594f'}, '102.png': {'bytes': 102450, 'sha256': 'bec6cfbb075d9b16dd5ed43082ef02f1d61c7ad77fbe8c793db73004ca0b1bb1'}, '1029.png': {'bytes': 161710, 'sha256': 'e57b2d31d261fbfbb94793a13d70226d63887c901d2f4ef84a25edbbcf34b220'}, '103.png': {'bytes': 102442, 'sha256': '9d63a0b3ca357e7e3a60c4a523a654562a54a7d330d3f229083e576722415ece'}, '1030.png': {'bytes': 233875, 'sha256': 'c614d8ebc61d84ca336a2ce163dd8187aeb8b440f14160255cf1bb5ef8a49a57'}, '104.png': {'bytes': 100422, 'sha256': '0de9ace92c4b8bf5df5b5b76832cb79c97ea3fdb8eb477dddeb6e541c72ee655'}, '1046.png': {'bytes': 140403, 'sha256': '8ae35783b1ff57e72f9149609074d6635d7cf47208589701346299fbeceec387'}, '1047.png': {'bytes': 143021, 'sha256': 'cc1b31855eb02aeba971e830e365dec6c97edc4b7c49914d0047f1082df13605'}, '1048.png': {'bytes': 143769, 'sha256': 'ddfd8b930cd213b0e50cf8f8b6e42b3ac09e79c261407fccc15415bf7fa2a09c'}, '1049.png': {'bytes': 149532, 'sha256': '025e190e791cfcd0cbd0b8822e395da29a719e1c87fa6c748c052df73ceb3168'}, '1050.png': {'bytes': 145122, 'sha256': '603eb17507a31d501ae2fc2adc414a049e4f3de3cf13e0e521df29f90b0f3db3'}, '1058.png': {'bytes': 158820, 'sha256': '500f1785f8e8ba8467c3d01bee0ebb788a21c3b6245bbf7641722de6977780c0'}, '1059.png': {'bytes': 137790, 'sha256': '2f66e44c151b3f27ecfa46b4bbb208cdf229012eb664763488488d751da37e55'}, '1060.png': {'bytes': 138636, 'sha256': '9f7ebd3426df73ddc5b9895332000e1b781b811fa099cc6b92e3129f255f91d9'}, '1061.png': {'bytes': 136143, 'sha256': '8921705915e4c9d0758dcf6b94e1829a042bb95a07becee830a23cbcd76b6609'}, '1062.png': {'bytes': 138232, 'sha256': '494d019b5cb474547be629c5c91964df4bd91919954c71a26207406fa3d12dad'}, '1065.png': {'bytes': 142918, 'sha256': 'c3d9b5f2bbf2ade9e250303789a4b42d3a24127dfa6dc7d1b98b34b555d4a0c1'}, '1070.png': {'bytes': 145878, 'sha256': '72ee379c1488de31d3053d7dc500472d7c44d10c406d7991db2512879b259ce5'}, '1071.png': {'bytes': 141904, 'sha256': '12a7287691bb1b5c9a4a83c0546d2549050eeea77c4dbef256655db8b14f872f'}, '1072.png': {'bytes': 147134, 'sha256': 'c6eb40ed38084ef8224c67056b01cd65aadc0b5fd33cf2298ded93263f6662c5'}, '1078.png': {'bytes': 147228, 'sha256': '0c89d567ddf7fb3c02d5e5b332270e7f1a2fe3cc723f78f94f9f602d3f0bad9f'}, '1079.png': {'bytes': 155545, 'sha256': '901070050e52ab764f649f8af3c4e613f72f7a2147df80b240270beeb5fd1873'}, '1093.png': {'bytes': 180713, 'sha256': '0e5225eb6c8c6c63a31cb9727fda7e28ad7a7419273dc7b09c42061d3abd2183'}, '1094.png': {'bytes': 184771, 'sha256': '0ee58d47f00c4a7092ca140576b75a550ee94f702810bea318f1a5c4a714ea4b'}, '1098.png': {'bytes': 152088, 'sha256': '1fef7b529807ba879f4046deaaf00ebe2d86a6ac9ad810f8882f029d7a9dfc52'}, '1099.png': {'bytes': 141658, 'sha256': 'f78859c1d45d9ca6c9fadc196763fef467b2f0b32719ca481beaf889fcd0ccfe'}, '11%0.jpg': {'bytes': 64068, 'sha256': '50aabe3f23dd41d871b489f3db1495ff8cd1cf53657ae4213fc1dbb20b7c1b99'}, '11%1.jpg': {'bytes': 59161, 'sha256': '90b8225adb6bf97f509dcbddc2cc184515147016ca3275de8ff9a1f48b9635ec'}, '11%2.jpg': {'bytes': 52363, 'sha256': 'd2ed416b20a64ff7feae1d8a2241e333f62dc3fdb7550df1cc3e08459689a105'}, '1100.png': {'bytes': 146020, 'sha256': 'd50b93f5ed120f1f486a96e986b9a25e555d158d3a79296eabe3538ee4939620'}, '1101.png': {'bytes': 173942, 'sha256': '3e8bbc7366e262bf5a7adb506326b19fa09be785b15f8c994d4ca72ca96a8ec1'}, '1115.png': {'bytes': 201473, 'sha256': 'eb9e1a6b1a7c2f86a1336b6df3f1c4afa8d6c2957adee72e1fba8ecf3a605d3f'}, '112.png': {'bytes': 154319, 'sha256': '7ec5bb4f786e3e68daec84d3812104437d1be6d9511111f439d7ea0fc84371d2'}, '1123.png': {'bytes': 260456, 'sha256': 'ff73991548361dcbb9e4dd67d0ca8bcd88b02d65b76100aec74570734e002554'}, '1124.png': {'bytes': 283594, 'sha256': '324f993471145e3669c7ac386c1e5c79db7047c7e18053b37ea4b10673f18199'}, '114.png': {'bytes': 195164, 'sha256': '5584aab5b543ed882fd37eb2006629734876b79ffadfeeb28294459de3925296'}, '115.png': {'bytes': 202895, 'sha256': '1b216cbd7e2262112a14ec538c9fe7c675ded9931063cf43bdb510997873c68b'}, '116.png': {'bytes': 198299, 'sha256': '0d7538b2f51d32133d01a068d135d59af578aee37af2fef5dae52107f2ed88ec'}, '1164.png': {'bytes': 249035, 'sha256': '5be47b71a85de2a30f47a0b827165f8b877da627f37daa77272ff6a394312e47'}, '117.png': {'bytes': 204678, 'sha256': 'fdc66c5a2893776f1b309be1c571b98b944410ff85c67b0c917cecd039f13dad'}, '118.png': {'bytes': 199135, 'sha256': '0cf56be50807c6a4277416e204c9e0470757daf3a6116bd1bce688778d046e21'}, '12.jpg': {'bytes': 77399, 'sha256': '23f6c7d765b9bba71911a52deb29a61292e918f27015507767727696268f52e8'}, '120.png': {'bytes': 135554, 'sha256': 'f8f57ac09d326697651140d437916dfec0784330d1b4a0015612a0ee660d4d47'}, '1210.png': {'bytes': 122428, 'sha256': 'faaae38f62b2fe2768cee7b3f4fb736cb614727246ceb1265d4975438487851e'}, '1211.png': {'bytes': 127209, 'sha256': 'c1b19282c32be20d999ee9946cac65a2c217f8f8caa42b8b07d3097606fa6e9b'}, '1212.png': {'bytes': 129088, 'sha256': '9b05f6744a4c2b034f553f400b3f6691d3bd387e241872ead5ace499741e0b4b'}, '1213.png': {'bytes': 133492, 'sha256': '5c3fa73a64716795efe3de8bc2d74591d3671cc699ec1c175975bda015ceeed6'}, '1214.png': {'bytes': 133348, 'sha256': '085d7634b07f910eaa73acdbc7931cca06b309c04271d571f0742a6b728d384e'}, '1215.png': {'bytes': 135356, 'sha256': 'ee4a889a33b387955914e01ade684f51cfcbcdc6559088e04914f1a2747aea0a'}, '1216.png': {'bytes': 135528, 'sha256': '02d8e06c55125d10b4d41e2e9cb258f029a3089604f5858f3322822e9fb98181'}, '1217.png': {'bytes': 136829, 'sha256': '6599eca55e0ae7ac1ea5745f979d350a307f883e72362350b6c865f5a8b3a3f9'}, '1218.png': {'bytes': 134068, 'sha256': '430b532abfc56f60dec5d94bed15da5001848085fc3b5dae9702591bac58654e'}, '1221.png': {'bytes': 140185, 'sha256': '873a14ac81f6196e286057b5fa860d4b7f3f7b374862ace926d98df09af1c64f'}, '1238.png': {'bytes': 185395, 'sha256': '248081e959cc36be111ecaae061be0313f698b381cd43e50f15e14dd2ac17d2b'}, '1239.png': {'bytes': 190773, 'sha256': 'df685aca6977035d05abd750feb8aa0b4e8e86b97ccc680d4d42418ea636859c'}, '1240.png': {'bytes': 192496, 'sha256': '7f654f71aaaa04e398de61a6b2e26ad56551848f40459c3b456c5a62724949b9'}, '1261.png': {'bytes': 209499, 'sha256': '2ff2aa5d51439552577a329659828fce04ae2a27e66215d0615b9e1ec8c37060'}, '1262.png': {'bytes': 212546, 'sha256': '57bb0b90cb5d958cc5e654aa8a9c9c363eeddd2e969b226a43a518e8503d41f3'}, '1263.png': {'bytes': 208934, 'sha256': '1ddf0a0f66cbefc022707882dcaca9b05b569f86822361097579c4c2e3d24f49'}, '1264.png': {'bytes': 182327, 'sha256': 'dd16996ff91bfb996fdab28e1a32f023636a4f817cbd1eca70ca8f1234fa0488'}, '1277.png': {'bytes': 282820, 'sha256': '08a58c27a1ba9f222d5ee149c5597915a3a39054a61bb50defe255c4f9c81524'}, '1297.png': {'bytes': 97051, 'sha256': 'a256f389a3e317105bbd5e8b1d2fcd1b2e6e8e82aa6ad78dfef77a88fc943950'}, '1298.png': {'bytes': 92913, 'sha256': '13ff2a14a767a92ae6eecdab7f4f1b5fdcffbd253c91d0db6e9deb22c66726cd'}, '13.jpg': {'bytes': 82366, 'sha256': 'ab80454b1dca2beb4b6eb919505d231f8b9e9ae4fb97737ffec24cc530e43e93'}, '1305.png': {'bytes': 143521, 'sha256': '0dfeace6a0d1f303dbbbee34a90902367e2f7c63ffa79f4120816871a3467897'}, '1306.png': {'bytes': 173551, 'sha256': 'd4c79b00944e75d0c4a83841e15dbe84d354d1f73f7f80d1689e1555dea69507'}, '1307.png': {'bytes': 171908, 'sha256': '7d1cdbdc1fa44817bbdd9590d1f632297a2d70cd9d57d2d457001f76a175cca8'}, '1308.png': {'bytes': 111051, 'sha256': 'f1aaa89ca15e846e4f76776ee999d3ee7e81d594deb5992eff4230c7e0c4b970'}, '1309.png': {'bytes': 123893, 'sha256': 'db8b794414b5570f79d1f3557c9e1a0bc51094df4fc6baac402910fd9fff37a6'}, '1310.png': {'bytes': 136378, 'sha256': '90b2f399971a0ec5dcb74a2f2f72754ac1fca8593d92d4607e0c91d9d098cbc9'}, '1312.png': {'bytes': 153274, 'sha256': '655ce3bd5b852a516a0b8abc02c1dafb7ad77f1235e41de847db2779344168b4'}, '1313.png': {'bytes': 132257, 'sha256': '1ed7150ca2911be977b1201925f23dd07afcaaeac283617a3202e415a7f0b2d3'}, '1328.png': {'bytes': 174030, 'sha256': 'c611b2352e1238fbedd1b7560f9cf9beecde2d2b1e859ba6446106a38264c9b4'}, '1330.png': {'bytes': 171331, 'sha256': 'd42eaddf23820da44054da517440683ae38502c2e62b9e4cfeda9ca462826c23'}, '1331.png': {'bytes': 171255, 'sha256': '4b7db0b653e58a5f70568a054d9a63fd66fb6c2d88894240ca8a02306c211549'}, '1332.png': {'bytes': 168908, 'sha256': '8e7008a30f5891846f3857ffacba77d58ed7bfcb20148f5653de2ad4c1ae9ee5'}, '1333.png': {'bytes': 165091, 'sha256': '495b8b50c5b7481f80cc93d5d3ba40cab29cedbcaf7923710a90fcae8f9c3862'}, '1335.png': {'bytes': 118093, 'sha256': '068f966928d0950744d530a0e1bc2dcb43b8dbe88b6939864dd638bb7bffae96'}, '1336.png': {'bytes': 203925, 'sha256': '364802ff8f5b3ad02b4a85386da22c4d2e24c54f02373c533bd786844efa2605'}, '14%0.jpg': {'bytes': 55845, 'sha256': 'fd63057efa623a80b37d2359614f321403b27a4c16aa57292ac8bf63b31a79b2'}, '14%1.jpg': {'bytes': 55450, 'sha256': 'a5e07bbf12a7ac39cfbb70b3f99d4c8b5de6ed0de8078fe96748a51383743ea1'}, '14%2.jpg': {'bytes': 66948, 'sha256': '24084475d8806617eac92f26ed56dbeecb8d170bbae5eab9d17dc8c9b15831fb'}, '14%3.jpg': {'bytes': 54451, 'sha256': 'ee5a5ad838e65a6ee853d87bf83d2f47668c17950b6ab254b88a554f607cbf0b'}, '14.png': {'bytes': 179276, 'sha256': '4a0b4b6cae2a7a3f56a5c476ee163ee50687935d33bda3c8393a2d2238d370bd'}, '1428.png': {'bytes': 189062, 'sha256': 'b5aac6389ab71c01198adf016691f5a13b388fe62d89c83fb6dc9b571ba5b46f'}, '1463.png': {'bytes': 148139, 'sha256': '94a90823520823671ed7e2894534ffd4a38411d018ee38f1486afca270da59a4'}, '1497.png': {'bytes': 175251, 'sha256': '90601fa2080c752c51dd536031957c2db2c7395912a69a3bff0035cfa83c3bc8'}, '1498.png': {'bytes': 173836, 'sha256': '42cbc7aca50ae22bb50c45efe1578a11ae279aa747c4097fbeda437ffbdecce9'}, '1499.png': {'bytes': 175553, 'sha256': 'b9586bd75575ac5aab1d1292a79994283ed9ad3ec3c96de61fd305d9c0e657a9'}, '15%0.jpg': {'bytes': 76400, 'sha256': '2c40c453d8e9da67e5603dc357bdb8d0b6e2ee733c45d9db322de5902a4b88b3'}, '15%1.jpg': {'bytes': 92168, 'sha256': '060b52592604b1d210f77055447502e9c396ce97120e4246c4f2e94bbeee79ea'}, '15%2.jpg': {'bytes': 67281, 'sha256': '253a38a035abc74e653ed859058762c82f1bbd50e45c877dee09c1bc1c605529'}, '15%3.jpg': {'bytes': 87070, 'sha256': 'f7b192055af8e76343f45fcae4b11a952f68ded363b562490345ff41e4921f74'}, '15.png': {'bytes': 157575, 'sha256': 'e6e4f2f18baaa06ec500f5d6b216525f9f0d388ec35105588d071e4c7f92adbf'}, '1500.png': {'bytes': 177474, 'sha256': '04c907462f0b367a275933313346337ac105f83249d8151602147d0d6f79175d'}, '1501.png': {'bytes': 173875, 'sha256': 'b48d6eb579aee38dbf55a13a63b0eda46ec72d5cb2d9ea32d9403657b56692a2'}, '1561.png': {'bytes': 207323, 'sha256': '9881d94b401d6b2e5fab0a6f7db5f7a18ec17b556215b078a6e258caf595a8e2'}, '1562.png': {'bytes': 235629, 'sha256': '89aa0976d95817d2d0dd8f07c2c419f2fc806ee371b86754dcd0f8af5498cc82'}, '1575.png': {'bytes': 179697, 'sha256': '9c15845d5c1e41d648ef8046a06fa7935cd35b90802c90c13060ae76d1acd9c4'}, '1576.png': {'bytes': 181507, 'sha256': '05c48e5eee8c7618d6720f53cb4c3cb3a3558df36a63ed16af85cb447f9cee1b'}, '1577.png': {'bytes': 176736, 'sha256': '92399424119c71e662622173bcf9bc865caac3ad871c1edc64ab5235850b095a'}, '158.png': {'bytes': 162756, 'sha256': '9b78d1917d7d72ad971e879b5b449dabe6b407c2a45be7bf2b7f6569f36088fa'}, '16.jpg': {'bytes': 50240, 'sha256': '1976aadc3e60749c431c61713295429dca0bbc30d2bb4c92c8b1caf9834c7860'}, '1638.png': {'bytes': 270687, 'sha256': '768ea5c7d8b4be6842357a0fbb386f87dd771abae337b6639eb1efe8aea0e959'}, '166.png': {'bytes': 91093, 'sha256': '22c2348989c3d4e6d5949b4068220bc6387ab3e3e8deec2f768a2982759cc703'}, '167.png': {'bytes': 86953, 'sha256': '36cae61d42694c0cc47ebbf69aee03460e58b7266ee16facec16c5c4d361613c'}, '17%0.jpg': {'bytes': 50887, 'sha256': '1cf66f034db53cdd245f7c762995800120546fe98ba1f22c8a400db1ce87030f'}, '17%1.jpg': {'bytes': 47806, 'sha256': 'cef45456f192e86c8b232943ed4a024d197401353bed0f26cdfde967c8ac3090'}, '1702.png': {'bytes': 177869, 'sha256': 'a817425a0b8ccc3b245769c0cc04ca70fedf032db51d84ed764d87177fabe754'}, '171.png': {'bytes': 193369, 'sha256': 'ba00b4881d6c9c2724795c4109072a1360200f4a369ed51ace4f9db6ea6908b3'}, '174.png': {'bytes': 210914, 'sha256': 'bef29e44071cf635be2beb10b1969d780b20be1c6540c4cb80792d613cbf2b69'}, '176.png': {'bytes': 202813, 'sha256': '229649eebbf17bfc197c5eec6e5a3563c7bee4b60c69fc9e0a1db0a4f28bcaf9'}, '177.png': {'bytes': 197203, 'sha256': 'b12a4ae36d06b31bfd7e919a56281915c8aeb0aae2f3ab9336d361f12ac2747a'}, '18%0.jpg': {'bytes': 45864, 'sha256': 'b473f3bbb2aeba82fd155a89fffce15a4f2e80b427b123ac8cd1a037f055bc94'}, '18%1.jpg': {'bytes': 44426, 'sha256': 'f06881ebfa599a1974e0633e88fb86b4a35f0b0952b9a36a2e41598bff0ef472'}, '18%2.jpg': {'bytes': 60526, 'sha256': 'cc8f9924314428550970d650bd414a33ea407b7b6dae0fa44ee031cd49e466cc'}, '18%3.jpg': {'bytes': 63923, 'sha256': '3e1bb340d5fd218f7481cd3865cdae6f4f88c5a48686d8eec7f03919389eca17'}, '1814.png': {'bytes': 232362, 'sha256': 'f00a3ae087dda2d441f5ae0d29df493f8545472936df7604db467e8545d07049'}, '1845.png': {'bytes': 239249, 'sha256': '3e472908b8abea0a76095078932ab2d00bca34167c27c070831a2d5493c6d15f'}, '1846.png': {'bytes': 223810, 'sha256': 'dbf08be161446c6d4836f09a66c7218b71a2d2148a744451b80c778ff695f9db'}, '1857.png': {'bytes': 193828, 'sha256': 'd24c99798e35f31002c86619f30996ae14f5e42dbdb8de646bd89a5ee59a59f6'}, '1858.png': {'bytes': 208206, 'sha256': '96895994e25d06c18da69f7c0c4477d41146f7cca817eb75b35cfaac17da2a2d'}, '1859.png': {'bytes': 192830, 'sha256': '87b1a5a7fd03d9df1689d56de2822c153c8cd9a5c32540aff5a92af06ab43b69'}, '1868.png': {'bytes': 172478, 'sha256': '0ca6766ae838f16eecfba0da8e5ecad7868cc50f9dd8291854793d17503ea0f7'}, '1876.png': {'bytes': 136513, 'sha256': '398668a5d0a31eb739d16d60cd3f2a878d4b74c25f79e828fa1464117a7a56fb'}, '1877.png': {'bytes': 147173, 'sha256': 'da6e54ef1b6c1e60f3ef82b4df31f223d8b7d3a8f478a231de74b1aa3650dda8'}, '1888.png': {'bytes': 239916, 'sha256': '6807c51c90b058b14b83b0c7aed91395e13764d9c6095765ee53d183c80c9305'}, '1889.png': {'bytes': 226287, 'sha256': '0243becca5a341bb89b306ff084ab116ace227f4b925e6b999b30499d3bc3bf4'}, '19%0.jpg': {'bytes': 48400, 'sha256': '2b6c937ce0eb1e4f6276c1e9fbaf87b5b0013f76f87b6cb12179dfe1421b2abb'}, '19%1.jpg': {'bytes': 46320, 'sha256': 'ba5efeb81ad9c34fb9ea39e0e39e164e9a98d9cff3e72843874d3a83af8e89ae'}, '1919.png': {'bytes': 188725, 'sha256': '21e812f1aaa464a006798d43fc947af4ed706e3605201e77f606ff0331800228'}, '1920.png': {'bytes': 181863, 'sha256': '823dc003776ebabe5da77ab62591b3dc5494384a056157f9ff7c26a42567af0a'}, '1921.png': {'bytes': 188719, 'sha256': '4d145a10ed645078622ffb4220611478291abffcffd8fe639fff398552b2bdd8'}, '1922.png': {'bytes': 190423, 'sha256': '8839e37e1951efb8bc92355c9c6e9d7f0d09c2f4da9e5518347a2cde13999fe5'}, '1923.png': {'bytes': 194730, 'sha256': '50f803181598fe473c35d4748e625cebc158f06363cae14f51f788d29a9a4c92'}, '1924.png': {'bytes': 190276, 'sha256': 'f009021913b7406df30608c48a26af5c3d1e10d9502a9e235aaea4ea3fe6081e'}, '1952.png': {'bytes': 200617, 'sha256': '32c972d0a7abef363b19782e4874f5dfac206b5b7ee9cb935eb42d14ea28911b'}, '2%0.jpg': {'bytes': 56660, 'sha256': '415a614f360c8f7e350cda2ad23212add7e00d8e8d4236b9f6cefab930ba2b87'}, '2%1.jpg': {'bytes': 48671, 'sha256': '2605c4b172d7869e62609da5385425c5d9bf0ad379d2bdf88da3cc8980778338'}, '2%2.jpg': {'bytes': 58428, 'sha256': 'b11f8b33b87db6349f667dd34374e6ac21632cfb418f77f3807ba4f210e3d8f3'}, '2%3.jpg': {'bytes': 58609, 'sha256': '7fd5b5f32b25bd4fb71a034309864b824cf035f5b65607ef2ff9681e21c78090'}, '20.jpg': {'bytes': 267002, 'sha256': '3f1d4d3db39884c239c253bc67eb05eb8697fceb6bcca4d0f2efa504277ea398'}, '2007.png': {'bytes': 222866, 'sha256': '051ccaa8841126ed16dd66c0030769d95eeef997ede878d222ef8d03e43c993e'}, '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%0.png': {'bytes': 224115, 'sha256': '011cbf8a0f05e1844b7db062a8dc2935ed9c5471fd6d14b521cc86d94b5fe86e'}, '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%1.png': {'bytes': 209383, 'sha256': '1bb4950bb476e78b66b88ad5be4e566ac575c44ce2c31cda6e3a7344e62d1f02'}, '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-91.png': {'bytes': 202531, 'sha256': 'b2445b752cb077ece2a79fddf4da04913b918a197258925bf32c11964cbb2023'}, '2020.01.24.919183-p27-132.png': {'bytes': 333798, 'sha256': '469ee7f675c5609c1d0dcd0d03b399dcfe99edc99759f55cdf9c9e8d7c56791d'}, '2020.01.24.919183-p27-133.png': {'bytes': 322966, 'sha256': '13a7ea458fd3fdb6a2345be8f2a95b3b65fbb72e396b14b87ffdcd7e19709f49'}, '2020.01.24.919183-p27-134.png': {'bytes': 314047, 'sha256': '9fb41e6588aebd07f753b4ac5156ece6bb32fc24c076585ebc7876578257cd71'}, '2020.01.24.919183-p27-135.png': {'bytes': 301102, 'sha256': 'e55e777caac8179510f6fb3073a8fbcf6ce8841acd9860735dbb277062c1ba44'}, '2020.02.10.20021584-p6-52%0.png': {'bytes': 98464, 'sha256': '1ff8c43169673e590a4dbcd16133911b2c7cac6363a6e7f4e41187bca9a8c2d9'}, '2020.02.10.20021584-p6-52%1.png': {'bytes': 104679, 'sha256': 'a5633ee53333b9c1d8f3e3ae3caf18fde055a1d06632a831bc04f114742d3c77'}, '2020.02.10.20021584-p6-52%10.png': {'bytes': 91610, 'sha256': '057daed5ca1b837f9a2efb8a7e62e042bd85769217a14a8377ef6d32f59803bd'}, '2020.02.10.20021584-p6-52%11.png': {'bytes': 95671, 'sha256': '3f82cb6df89ea21bf86e307895ff6130f4c91d744d54f3d934e7000fb9f893de'}, '2020.02.10.20021584-p6-52%12.png': {'bytes': 85921, 'sha256': '5afb8e3d0c5dab9c9fe593cc64211a839313c8d181ba6f491687f0bed6af1fa4'}, '2020.02.10.20021584-p6-52%13.png': {'bytes': 92922, 'sha256': '32f5c2f51b6e83e354c9cacd48063ea35c7141ef7930a292e4321c1493ab5c9c'}, '2020.02.10.20021584-p6-52%14.png': {'bytes': 94285, 'sha256': '5e3c53904383584cba5e1bd43f13ce843a4d8b904d97397cfb955abd1d035b52'}, '2020.02.10.20021584-p6-52%15.png': {'bytes': 95271, 'sha256': 'cebd8ec6eede64d4ddc11de7daf04cd1eac494ddcd9ef411b0068329f42fc2fe'}, '2020.02.10.20021584-p6-52%2.png': {'bytes': 104064, 'sha256': '719a952cd89029a3c89c499e6436696f7c18f4fd486de1711b34fa10f9b83448'}, '2020.02.10.20021584-p6-52%3.png': {'bytes': 106735, 'sha256': 'affcf6b774bc171f668194a1ecd921c072e29efb77c0bf088e7db7a698d16ea4'}, '2020.02.10.20021584-p6-52%4.png': {'bytes': 93118, 'sha256': '8495c66de2f560b25015f7c158e68a0a6d9604de696f975743e6225c40a79729'}, '2020.02.10.20021584-p6-52%5.png': {'bytes': 95893, 'sha256': 'a9f951929cd875a5c2f37b7b5ee565966f6d9e67e7eaf9e28f00047b9d66be0b'}, '2020.02.10.20021584-p6-52%6.png': {'bytes': 99472, 'sha256': '39e7bd9f10f091a39ab467a509d0bb360bcab70916c93905e4b753fcb2855826'}, '2020.02.10.20021584-p6-52%7.png': {'bytes': 98103, 'sha256': '4a6ffe7aa008f648333cd0593b1cf5e567b63e93317b0085ea7cbb77af25a49b'}, '2020.02.10.20021584-p6-52%8.png': {'bytes': 91377, 'sha256': '3bbdd1fdd1a57b4266f225a0f28f9fe8a30afa076b5cd6252010bd6f0e7c4f20'}, '2020.02.10.20021584-p6-52%9.png': {'bytes': 91283, 'sha256': '915bb8417257b03a0f06e4a9ea3e0e1faa3a432b4c8a126ade48ad7677427bef'}, '2020.02.11.20021493-p16-109%0.png': {'bytes': 353057, 'sha256': 'dcf6cb759abb71c6f1ac48a6acaac6fb5a52400a5f3ec171ae89ff00442f1b55'}, '2020.02.11.20021493-p16-109%1.png': {'bytes': 383969, 'sha256': '3784195082e466faaa687a6b9a0fff63c447bcb71db801d54a5ccd56f2ce44f9'}, '2020.02.11.20021493-p16-109%2.png': {'bytes': 382040, 'sha256': '9e529980a32b081c88900a656cb56eec2a5a3e4a6bb86d4637ea17a1eb80f148'}, '2020.02.11.20022053-p12-67%0.png': {'bytes': 323036, 'sha256': '88c70e82069bb88aa584fd6f2d5345a25fe4888c9cd6da0d5b5d3fb8598a206f'}, '2020.02.11.20022053-p12-67%1.png': {'bytes': 272527, 'sha256': '063dfed348811ce7975989125d01c55a06edcde77c684ecd46529d076820a11c'}, '2020.02.11.20022053-p12-67%2.png': {'bytes': 232684, 'sha256': 'e082cbab27af3c185733c6bed6ff7359c1a86ddc193512e405cc416f008c8cf4'}, '2020.02.11.20022053-p12-67%3.png': {'bytes': 225882, 'sha256': 'd1d568391c78873a238e1b19592ddf5e6b6bd6869039dd674b75c538f5329357'}, '2020.02.13.20022673-p13-77%0.png': {'bytes': 216268, 'sha256': '18534400a5f6927190c89c4c202a22a123e7bcd1c5ff376e990559d2e6a3a5bf'}, '2020.02.13.20022673-p13-77%1.png': {'bytes': 222925, 'sha256': '0813c288ddeecb0fdb5bfd3a5e441b98ad5a96f3b237c232216b8b88270cfa26'}, '2020.02.13.20022673-p13-77%2.png': {'bytes': 384720, 'sha256': 'c345a4bf97c07dc812a9ebf8993e9c0310c50360e8ea536e051901c6714d4c5d'}, '2020.02.17.20024018-p17-61%0.png': {'bytes': 209033, 'sha256': '30330783fa29bb1899a0bd2c1a8f526bf7ce9bcf9926272c7a0b1d1a88ba0cc0'}, '2020.02.17.20024018-p17-61%1.png': {'bytes': 228924, 'sha256': 'd7ac397cce6bd5383440bc40dc4f8368ab756e4c708ddaecc4e38327f9a00ddf'}, '2020.02.17.20024018-p17-61%2.png': {'bytes': 206370, 'sha256': '5162722a0d0e6dd157471995c5726166f9dfccceebd038a941ff5e06175b2b2e'}, '2020.02.17.20024018-p17-61%3.png': {'bytes': 216059, 'sha256': '8ff91f5665770c477cf8b9a1ca7f365c8f9d4b082528f41234e1d7ce9ade1f8d'}, '2020.02.17.20024018-p17-61%4.png': {'bytes': 194218, 'sha256': '7f23475857788bc9b4334ba118cbcd8d06b6b7d2df39ea2b2fcc386e857f960f'}, '2020.02.19.20025023-p20-93%0.png': {'bytes': 92682, 'sha256': '90141b828caa6d5192685b250c2415a4c5656f0df46df447f4653f7a1b938f72'}, '2020.02.19.20025023-p20-93%1.png': {'bytes': 82378, 'sha256': 'a42f584884071499df80610c13ed3d9f72c2013379156a734d2feefc6afeb656'}, '2020.02.19.20025023-p20-93%2.png': {'bytes': 63155, 'sha256': '9ee0c865c547646cf06014e4bb6a53049d5174a1db3fca45e1972f17d3aa5c22'}, '2020.02.19.20025023-p20-93%3.png': {'bytes': 94824, 'sha256': '493e59a00e3071feb264439a009fcc232aeb5a83c93c9165477ba52ba91968e0'}, '2020.02.22.20024927-p18-66%0.png': {'bytes': 109721, 'sha256': 'be927e2c37ea0b529e3b6672cc518b86685885cc09ba6a504dc16e590fdfc9b9'}, '2020.02.22.20024927-p18-66%1.png': {'bytes': 218361, 'sha256': '7fd5ce22b684d8dfe0a1ffc5be4b1baf7f561a983a469feb6f8ea45ffffb186b'}, '2020.02.22.20024927-p18-66%2.png': {'bytes': 220080, 'sha256': 'd7fac1b240a3fdea8da8790149c8e02186789544870f26b384b29c2dd30320b5'}, '2020.02.22.20024927-p19-68%0.png': {'bytes': 65318, 'sha256': 'a240c11fd4d101ee3c8220b1956336da1a4bcfd4f8d3f59378e822de70dbf4e0'}, '2020.02.22.20024927-p19-68%1.png': {'bytes': 68795, 'sha256': 'f9d6722262ad63893ad6b08d33f6f1ca8973e389ceda2a95fd645c95aa54ddef'}, '2020.02.22.20024927-p19-68%2.png': {'bytes': 126230, 'sha256': 'de2e80bb2f6ed46a3097843a1c097137f1da6ed3d6745835086f5eac7b9a00f0'}, '2020.02.22.20024927-p19-68%3.png': {'bytes': 133553, 'sha256': 'f6ddf8b892c3ed5c53e82c15002a2716883e3a5968f817189255bd8c4e3519bd'}, '2020.02.22.20024927-p19-68%4.png': {'bytes': 136668, 'sha256': '7d1bea926bc739b52e8fb1b62c4a752e3c838077d518b2944a53de5f22b2ec34'}, '2020.02.22.20024927-p19-68%5.png': {'bytes': 139078, 'sha256': '70dbf9e192b70d8c96009e0415a98cff130f689deeabff8fbee9fed20b349030'}, '2020.02.22.20024927-p20-70%0.png': {'bytes': 74050, 'sha256': '4d5341ea140ab2c0590ae9c96466db37bc621c775234e3785b140d0da9653db2'}, '2020.02.22.20024927-p20-70%1.png': {'bytes': 77559, 'sha256': '4cf0af0e9684b3f4a7514fbc630e813de12857e0ce2bf918dcf8d435bf7845a1'}, '2020.02.22.20024927-p20-70%2.png': {'bytes': 34095, 'sha256': '39b7b5818bcb4aa904c4ce0b7596d86679a4b9e425e14b826fcc8a5c695ebcd8'}, '2020.02.22.20024927-p20-70%3.png': {'bytes': 79925, 'sha256': 'a3b072dadf19acfd3b520e09cf719338250cd96f2f877fd3e285f61cf9aa838d'}, '2020.02.23.20026856-p17-115%0.png': {'bytes': 42133, 'sha256': 'f97b56b69ae8c32643fe326bd24aeea53777f5d5eadb443c05f7b65d35bd92ee'}, '2020.02.23.20026856-p17-115%1.png': {'bytes': 44942, 'sha256': 'c8714ac32d8cdc31747c3c5b074c9855c10eef7ab7f8ca4dcf148ab2b964206e'}, '2020.02.23.20026856-p17-115%2.png': {'bytes': 39890, 'sha256': '1e2256ae49b3be6d2d3a8ef801d6e50fb078f5d7a0f6a28765bb63ce88f9df5a'}, '2020.02.23.20026856-p17-115%3.png': {'bytes': 46128, 'sha256': '9315bd86f5a7d161a3f80cff14b97880cfcd78d3d7e19c45653eafc55d99e7b4'}, '2020.02.23.20026856-p17-115%4.png': {'bytes': 38805, 'sha256': '3e7cdd3ef1e868bcbba4acad1019139bbe908f522ce3c203bb4660a4b02dd7d2'}, '2020.02.23.20026856-p17-115%5.png': {'bytes': 40474, 'sha256': '4dedab6013b636e140a9b2085924f847e27670a4c0593ea16d441b50fb5596ff'}, '2020.02.24.20027052-p8-73%0.png': {'bytes': 135830, 'sha256': 'c32875c0d4c0f653dd00de4b20fc0588090049b00e70603ae9f3b3634734273c'}, '2020.02.24.20027052-p8-73%1.png': {'bytes': 141721, 'sha256': 'b050690cae4fe4a4e7eed5c5ee16e5ea9f92c3a679ccd66430e6484a52fd83f5'}, '2020.02.24.20027052-p8-73%2.png': {'bytes': 123239, 'sha256': '42925930f16d128f238997427ffc9f16518aed7a4af5907305c43d02b8b13e41'}, '2020.02.24.20027052-p8-73%3.png': {'bytes': 143892, 'sha256': '019cf3d9da1ebc2f4781cd4bedca09806e82530bd81f747589a8743a48a14efa'}, '2020.02.24.20027201-p19-669.png': {'bytes': 122394, 'sha256': '939da2cc8890aa1ff003c54c308d461393f545444273575e7690f12613157dd4'}, '2020.02.24.20027201-p19-670.png': {'bytes': 121004, 'sha256': '402606da4288ac06dd5003d7e983856e606fcb865e7f421b6ac578b2fd64dfda'}, '2020.02.25.20021568-p23-108%0.png': {'bytes': 24867, 'sha256': '9fb7833d60823b90a1141f53c66dee4028fc09b607a30794fd6eff18347572fc'}, '2020.02.25.20021568-p23-108%1.png': {'bytes': 29319, 'sha256': '2b97de37675bad0ccd21c4738387e982176e485c95a8739a30e2b729c3f49cdf'}, '2020.02.25.20021568-p23-108%10.png': {'bytes': 25201, 'sha256': '8a7238ccb669a7a0b4625c20b9c0339e9cf16d435bec7f620d89893462787937'}, '2020.02.25.20021568-p23-108%11.png': {'bytes': 20570, 'sha256': '8d034c47e3b9112a86c392bcac80210739a058a640bad29c799196b9eff61e3b'}, '2020.02.25.20021568-p23-108%12.png': {'bytes': 17793, 'sha256': 'a8af75e063a0e744ef349e19316407719a9d0b4b1c4e41e339e4cc9207c86e46'}, '2020.02.25.20021568-p23-108%13.png': {'bytes': 22490, 'sha256': 'c79664526bfe7d728140bc513934d6deda11b744fc0bdbd8fab28c3d0e55173a'}, '2020.02.25.20021568-p23-108%14.png': {'bytes': 24012, 'sha256': 'c77c85ddcf6d954713604963aaefe15c0a8cfce52c99ddbea471d63081c1d2e3'}, '2020.02.25.20021568-p23-108%15.png': {'bytes': 23836, 'sha256': 'b938a197d5e15837952950e24c78524facf29be456cf32181179ac028043fb08'}, '2020.02.25.20021568-p23-108%2.png': {'bytes': 26517, 'sha256': '9a67bab8075140c950252e90b8852d2cfb22a997b83994cc999f2072089fed42'}, '2020.02.25.20021568-p23-108%3.png': {'bytes': 28467, 'sha256': '9a98f583cfe6e36c898a426825a68e168f772abebe3a4d3cd376056a75a19b61'}, '2020.02.25.20021568-p23-108%4.png': {'bytes': 23855, 'sha256': 'c3f865c5299eaf3fea6d296900c059188013e08bc8659b1c1ffa9fd9f0260d09'}, '2020.02.25.20021568-p23-108%5.png': {'bytes': 23844, 'sha256': '70d9bb86e5425ee9e7042e5374a09b75c7526b0df0096992dcc6610826bf862d'}, '2020.02.25.20021568-p23-108%6.png': {'bytes': 23016, 'sha256': '9a838f7ca8dc2fa9a84e18e24731e347d6ef53ee7713e9ae9183829785f8f323'}, '2020.02.25.20021568-p23-108%7.png': {'bytes': 23058, 'sha256': '7d444d53b8a9234c25580da72315f583d5c4daa25f1329e1e47f3bd0af4b2aaa'}, '2020.02.25.20021568-p23-108%8.png': {'bytes': 23691, 'sha256': '6e74aa3d814bee99017d8ca09600ebe476be524d94496ef7f5b4edb3c6617fe7'}, '2020.02.25.20021568-p23-108%9.png': {'bytes': 22815, 'sha256': '8068de1dc877195a8d5690424bef3167455b39828174e8d5e77da7efcd503cbf'}, '2020.02.25.20021568-p24-111%0.png': {'bytes': 27654, 'sha256': '110558c5979f91a1d43972be7ecee8774631a906a68d4d8369476debc0343b45'}, '2020.02.25.20021568-p24-111%1.png': {'bytes': 19365, 'sha256': 'd71fc198454a89fc81d9177e9d677a5f0a2d5716263e5430bae27e5b9a15dd03'}, '2020.02.25.20021568-p24-111%2.png': {'bytes': 20538, 'sha256': 'fd9f74b4a7b85c863776f09d48bff28fd9005c974a839154e38884226375d5ea'}, '2020.02.25.20021568-p24-111%3.png': {'bytes': 23745, 'sha256': '1a4e2942591b1e3a1b97c24dc8716b441984f20106493a0a1b3e2d35c8fa5db5'}, '2020.02.25.20021568-p24-111%4.png': {'bytes': 19654, 'sha256': '2303a103a02f299acc3dd8832d08f2801d7ce714e97c1719b3b0ea122fcbc13e'}, '2020.02.25.20021568-p24-111%5.png': {'bytes': 15558, 'sha256': '2aec7b82e1837130ffe3179161f421f7f92b27d10da8ddfccef910972aadfb4d'}, '2020.02.25.20021568-p24-111%6.png': {'bytes': 13674, 'sha256': 'e12eeac29a69c6cc53efc5d7d90e0dfc2e6f9cdf16036b81a012f6b5314cbb3d'}, '2020.02.25.20021568-p24-111%7.png': {'bytes': 9451, 'sha256': '7ebe9e4950d51c50ef6d940231102b984753ec05955dfcfc0fb49ff5099c5ae8'}, '2020.02.25.20021568-p24-111%8.png': {'bytes': 13950, 'sha256': '20a416ca2c761dc925c78c867763f93ef70aababf11855541a2b5630d67f7534'}, '2020.02.25.20021568-p24-111%9.png': {'bytes': 16141, 'sha256': '3f10c209f1a9b2b8be2bc440e16578d104872b904ac75cb55adbc463dde205db'}, '2020.02.25.20027763-p15-53%0.png': {'bytes': 1594100, 'sha256': 'caab493f9f6a2d6f36b5e848172964b935163df62f2b9f94be31561cfe41f8a3'}, '2020.02.25.20027763-p15-53%1.png': {'bytes': 1371015, 'sha256': 'bc26d24567adbda993f0515d3d16bac6f705e67244fef05ecbf70fe69b62e5d3'}, '2020.02.26.20026989-p34-114_1%0.png': {'bytes': 196522, 'sha256': '5685911f211d53747d2c250e3e3fd80094384220edae80fb7133b1fae8e9d30c'}, '2020.02.26.20026989-p34-114_1%1.png': {'bytes': 193298, 'sha256': '4eab1c8412bdb527dbf9b7bbcb776725b49ed8fc8e429f8cc761a5ba6dadd6bf'}, '2020.02.26.20026989-p34-114_2%0.png': {'bytes': 188716, 'sha256': '0752fc95f1f227f22971bdf41b4232da6d2fc4bfe6c2294f4fb75345b3d57064'}, '2020.02.26.20026989-p34-114_2%1.png': {'bytes': 202132, 'sha256': '44fe3a2f3c2b8c66518d08b8958418d7738a1fe0b851543fe73f3a87089afd28'}, '2020.02.26.20027938-p6-80_1%0.png': {'bytes': 33158, 'sha256': 'be05edf8031a911122302bef73e6f1b42cac6fea931ad67a2c0daa4e728e0ec9'}, '2020.02.26.20027938-p6-80_1%1.png': {'bytes': 46955, 'sha256': 'd96c751ce6df07897ec99101eaee6aff0c4bd51241bde41c2d333b53496aebd7'}, '2020.02.26.20027938-p6-80_2%0.png': {'bytes': 36927, 'sha256': 'dd9286173d4156d6ca960c5d62e7b878ad46d1676675acd9a4f7e95b4e7d0f4e'}, '2020.02.26.20027938-p6-80_2%1.png': {'bytes': 62750, 'sha256': '61af40afa261f1068332b8a871f4706bf8b5d64ae898cd754d1b007bd1a15517'}, '2020.02.26.20027938-p6-80_3%0.png': {'bytes': 38703, 'sha256': '33abb6c2d142ecf6276f9ad4706d587aaf5a408561c64bdc7a67c846e51f0618'}, '2020.02.26.20027938-p6-80_3%1.png': {'bytes': 38153, 'sha256': '214e94973c3235eedcb4fccdd9aea97e56d98704d34cf06d506b3e8966d4f193'}, '2020.02.27.20027557-p25-137%0.png': {'bytes': 23681, 'sha256': 'e7141237a1c829f30b540f6eda6cf6a41f6d4c8810d0aef7ba074f66f880ccae'}, '2020.02.27.20027557-p25-137%1.png': {'bytes': 24589, 'sha256': '39485916e7bc1b2c863c222d5cc05400948c198dae836bd66cab24d114e5d1b9'}, '2020.02.27.20027557-p25-137%2.png': {'bytes': 26989, 'sha256': 'd177cb864c40f5f5ce4b6b8183f12e305a0d2144fd0c76873748ebc3ab4d8d2a'}, '2020.02.27.20027557-p25-137%3.png': {'bytes': 26772, 'sha256': '379d06783ec5e56fcc5f7cf303dedf55813dfeb79a66abedf2c39deeb688c642'}, '2020.02.27.20027557-p25-137%4.png': {'bytes': 25863, 'sha256': 'bdc83282ce6361c8ed8df1166304cd3eb715e87ad2b0453615efd82ba55244b0'}, '2020.02.28.20029025-p23-182.png': {'bytes': 214369, 'sha256': 'cbcaa0b5dc999a3871ee99a54510b7aa6b51076db5ad721b41fc56733c6a978a'}, '2020.02.28.20029025-p23-183.png': {'bytes': 197451, 'sha256': '42a14c456c07596781dfe66bf8d23ddec3abbd21ce65bcf05b340480bd1f6b4c'}, '2020.03.01.20029769-p21-73_1%0.png': {'bytes': 189234, 'sha256': '5df96ebe67430445df934bc2e34014115c18540a4437e8b3c00e66b2e6e74f3d'}, '2020.03.01.20029769-p21-73_1%1.png': {'bytes': 203147, 'sha256': 'c154de52c38d9a4d7beb98bbc0aee42109d923e5601b0718732471551d44d91b'}, '2020.03.01.20029769-p21-73_1%2.png': {'bytes': 182728, 'sha256': '5a00123a9667c4f184b51b535c5fc1a9eb865e4c55af04264fb27564cd10c6bd'}, '2020.03.01.20029769-p21-73_2%0.png': {'bytes': 209179, 'sha256': '53421485593be87abc55a552845d1dab889f7dbb0cb41c43fe3b65d93a3e8847'}, '2020.03.01.20029769-p21-73_2%1.png': {'bytes': 204593, 'sha256': 'c717e1487027b9d6c956397d0746be53a867b228934f5ec4262d4f178fd63f48'}, '2020.03.01.20029769-p21-73_2%2.png': {'bytes': 183969, 'sha256': '9a25353714b50c9219915504f14ab0836e29b8518a18cedf3675be89ba39a53b'}, '2020.03.01.20029769-p21-73_3.png': {'bytes': 115832, 'sha256': '253f6b45bea2a5416984f19396d1ac20d1d7ff68c4d8b8606244758d842562a5'}, '2020.03.01.20029769-p21-73_4.png': {'bytes': 110176, 'sha256': '412482e7f87a6d0faf6509b64419fdf9fa947875861fa288d7500364279d17d8'}, '2020.03.03.20030353-p17-106%0.png': {'bytes': 196140, 'sha256': '110536e32944cd2f6851a848beebf8a5176323d7b5f9872d336c40219765c6b8'}, '2020.03.03.20030353-p17-106%1.png': {'bytes': 179673, 'sha256': '939b7223e05a39dc028faaf35755439820fce52148db588c038f4a63cc704af8'}, '2020.03.03.20030353-p17-106%2.png': {'bytes': 244824, 'sha256': '72603554234e10b77054d6be1884eaf0f2579fdac5f59a23c4454661b6e4c60f'}, '2020.03.03.20030775-p10-88%0.png': {'bytes': 961895, 'sha256': 'c1cafb1ece70923780a78c67021493fc27d6c797d379307b22d1566fa5d1cd2f'}, '2020.03.03.20030775-p10-88%1.png': {'bytes': 957367, 'sha256': '08ede19acfb00648d6dd0a1d6efe25805c04c507a499c6c1cd3622869e4c54d5'}, '2020.03.03.20030775-p11-91.png': {'bytes': 1100086, 'sha256': 'a7a700a8345f2f7108bd0bbcc3411d38075c0463951020d6337ec13bd870e4ad'}, '2020.03.03.20030775-p12-93%0.png': {'bytes': 1235316, 'sha256': 'dfd65e33518e1c14022f2ca9078d2994b18ef0a131f7b798f14ff6b1f071d522'}, '2020.03.03.20030775-p12-93%1.png': {'bytes': 1295110, 'sha256': '8ba6c8f39821e6c9127d047f8f2d1e2be9b53f172f276fa0c1aa76b30285c9fc'}, '2020.03.04.20026005-p12-1946%0.png': {'bytes': 91021, 'sha256': '7b8e92df2db99020a0d6f8e62e26241b01672eadabd139d6a6c5aebe4f97b138'}, '2020.03.04.20026005-p12-1946%1.png': {'bytes': 83888, 'sha256': '73a4e883b388fe4b08fa076763c473db60bd972a384a32ed9eb9d7cd585af5b2'}, '2020.03.04.20026005-p12-1946%2.png': {'bytes': 82950, 'sha256': '89f859e8930f16c261370e5dd56678e8a272c37c09f4624f9837818360e3dc37'}, '2020.03.04.20026005-p12-1946%3.png': {'bytes': 92411, 'sha256': '58483975df014eb02389a12a7f8e77e8d7db3c48aa6ab8d332485ab8340cd546'}, '2020.03.04.20026005-p12-1946%4.png': {'bytes': 87864, 'sha256': 'a0c6ce8f728e331b709514176815edbbe9d72a8a3da39fa4f41bc57ff05ccd20'}, '2020.03.04.20026005-p12-1946%5.png': {'bytes': 73197, 'sha256': 'ef72001cb8ecdc004da2e2b115f6b9acb34ef363f0dc6fb88cc96c06a914aedc'}, '2020.03.04.20030395-p27-108%0.png': {'bytes': 18191, 'sha256': 'c4200469d19740026340e88b212a4251152277aba07c30e7ae2e92df908e33e1'}, '2020.03.04.20030395-p27-108%1.png': {'bytes': 15852, 'sha256': '0d95909fd22f4c35f827d9835ec9445d2259af214a0d957110dd5ef757f708b0'}, '2020.03.04.20030395-p27-108%2.png': {'bytes': 16382, 'sha256': '9fd65d97de3ad4a08b5d5596b437febf304308deb81f439041adf5e70e16ee64'}, '2020.03.04.20030395-p27-108%3.png': {'bytes': 35264, 'sha256': 'cd0b3112a32fc1d41a973679f819c756d4db49d9d6151d14ff47a213094c105c'}, '2020.03.04.20030395-p27-108%4.png': {'bytes': 37156, 'sha256': '7c79f7e8eebf04defc4a04e0c14c7662ad4ba60d0fc2c679f1ee003a3115f949'}, '2020.03.04.20030395-p27-108%5.png': {'bytes': 36452, 'sha256': 'fba55d16e6aea606e6290a76acffe9dc238c3681a0cf4b7b106e4d465d55fc23'}, '2020.03.04.20030395-p27-108%6.png': {'bytes': 34860, 'sha256': '1a7a9ea7cbb678f4f9e58f486fd84f87fa12a285f868e6ae3d2406374f25779d'}, '2020.03.04.20030395-p27-108%7.png': {'bytes': 35590, 'sha256': '9789e6f815ba3c3a05c35904812a7b48405b000952fef488bb2c2d0db7247095'}, '2020.03.04.20030395-p27-108%8.png': {'bytes': 36770, 'sha256': 'a70cd111a06567f4bb483472932d4d6135dfb271ada2ef34699f0dc2507f5333'}, '2020.03.04.20031039-p23-97_1%0.png': {'bytes': 22350, 'sha256': '99d471737695785e8ca8cd96eaffc2d96afb16a925a413c439b5688d31fbd76d'}, '2020.03.04.20031039-p23-97_1%1.png': {'bytes': 69325, 'sha256': 'd2e2dd496a3afb1a74deba188d4bb58bd1b5e2e35b5a14229e918facd905b66a'}, '2020.03.04.20031039-p23-97_1%2.png': {'bytes': 66924, 'sha256': 'fb88d5730b4c849399cf7ab3fb964805566cd268aaa057d390a7766546f4c1da'}, '2020.03.04.20031039-p23-97_2%0.png': {'bytes': 69156, 'sha256': '87cc0f8aca9b3973c0baf1c825beefac1eeca1fef5fbbd1fbd24fef17c12c354'}, '2020.03.04.20031039-p23-97_2%1.png': {'bytes': 77325, 'sha256': 'ee020a4cecf3e09628017b81c3639135943afb3e4d0d857497af0f34db500127'}, '2020.03.04.20031039-p23-97_2%2.png': {'bytes': 73408, 'sha256': '3e4ccbad435cc08232fa9773bd9920f0ecebf0ccb914eafc5ce9c0c1e34e28de'}, '2020.03.04.20031047-p12-81%0.png': {'bytes': 114774, 'sha256': '0663944ef548e4592367d6f71bd33a07a305445f1e52c2bbfa13773201301b15'}, '2020.03.04.20031047-p12-81%1.png': {'bytes': 121134, 'sha256': 'f55256364b56d88e521064e539dd762493f8dd0ac2fbba963ba14eb492057da8'}, '2020.03.04.20031047-p12-81%2.png': {'bytes': 112838, 'sha256': '50ccb84bffc3851c3e93cccda952eb5e321e9ec859d3e25496add21c4e9288ad'}, '2020.03.04.20031047-p12-81%3.png': {'bytes': 116854, 'sha256': 'f385b7cfaf20a68ec155db5547f0c3bb7abd9a577628be9d59df5dc4b34dd3e8'}, '2020.03.04.20031047-p13-84%0.png': {'bytes': 134951, 'sha256': '01bcd0eee3e685e2317af9a6113f7efa5a54ae23b0f759e0a04ed95f30bb3028'}, '2020.03.04.20031047-p13-84%1.png': {'bytes': 119957, 'sha256': '13857bb8f5fec58e1059f99a62d0ebe4089b8f612d9e9f0502fef8b5260839e0'}, '2020.03.04.20031047-p13-84%2.png': {'bytes': 103428, 'sha256': '7acf55bcf14e1cd50fbd49dae421e0c3ef1819d96ff5928f5c8c586c5440bd55'}, '2020.03.04.20031047-p13-84%3.png': {'bytes': 123441, 'sha256': 'dcdf7624248420e794de70cef33eb726bf959095dc0919b5440c0d9d4730403c'}, '2020.03.04.20031047-p13-84%4.png': {'bytes': 135860, 'sha256': '978e4b7821b11101151088bb0c0201d5f31696d79508198ffcf62983bd301e8c'}, '2020.03.04.20031047-p13-84%5.png': {'bytes': 140856, 'sha256': '35b83b5218fd896043e8c3e66ba9c06d1ad24f41b276dca9ccf8762a3f75bd37'}, '2020.03.04.20031047-p14-87%0.png': {'bytes': 337858, 'sha256': '93eebbc334f20e1cbf8f02f1f599c0d4a0287a0df168b9d630c78b2004386321'}, '2020.03.04.20031047-p14-87%1.png': {'bytes': 298135, 'sha256': 'e969a716c34cdbf0bda5eb22ff9e0108e8598d120cfb78da319a946435f46396'}, '2020.03.07.20031393-p7-50%0.png': {'bytes': 189293, 'sha256': 'a4313f4e159a550a61151e70df434d5d3dda8e17e38981763f8ed988ad8985b6'}, '2020.03.07.20031393-p7-50%1.png': {'bytes': 192834, 'sha256': 'edb93c260173a7ac58493cebe77e50520942fc1711eb86be993f99fb86ad0b7b'}, '2020.03.07.20031393-p7-50%2.png': {'bytes': 198601, 'sha256': '379c1315c09b7ba5e04bcf5f7de9878afd35f699d360c3e6b1fe1a49a10e4033'}, '2020.03.07.20031393-p7-50%3.png': {'bytes': 203043, 'sha256': '62ed4ea7887f9cde23c99e76c2a0d7edb59b23cd9db055650601a051762325f1'}, '2020.03.08.20031658-p15-103.png': {'bytes': 260875, 'sha256': 'f70a22fd26c6a2d3a0ec8daad07262030e51829a35fb7d5dc90ac0c11e8fe02d'}, '2020.03.08.20031658-p15-104.png': {'bytes': 274060, 'sha256': 'b843102dff4896d94f7bd28955537a911e161bf6c16a45cc35e15bcae5b5ac81'}, '2020.03.08.20031658-p15-105.png': {'bytes': 158674, 'sha256': '83c63d64b1cae1757223ebb612403d8c7708a4be0c83ceabfd016f5d32ba5f5f'}, '2020.03.08.20031658-p15-106.png': {'bytes': 121873, 'sha256': 'bfcb116685cd471c5867d133246fa9a25934e11ced30b4c0da6fffbc6e23d83f'}, '2020.03.09.20033118-p20-93%0.png': {'bytes': 199721, 'sha256': 'bc390ec67b05dd32b625f49e377f070bbe1467920301d205b7666e218c0544c7'}, '2020.03.09.20033118-p20-93%1.png': {'bytes': 356070, 'sha256': 'a56edfc4e0f5e9ab187987ff693cd6c085698b4ff0f42aa4ec7e44f42285cd75'}, '2020.03.09.20033118-p20-93%2.png': {'bytes': 365118, 'sha256': '8c9a7e49e2e366d3afac68ac27cbe821117e776830dd4e98dab0ad4785429c84'}, '2020.03.09.20033118-p20-93%3.png': {'bytes': 212226, 'sha256': '92df659ae88d0b2078b2c23b831acd29c1183ca56b972b958826fc57d76e9e88'}, '2020.03.10.20032136-p34-118_1%0.png': {'bytes': 130980, 'sha256': '2dd5610f95d2d67db4d423128f992084e43b6068642433663fc6e217ae00d86c'}, '2020.03.10.20032136-p34-118_1%1.png': {'bytes': 133690, 'sha256': '9c115f3009fffc4399f42aeb5bc64d2f4c24fd5dfd96f97ccec4d0b5a70d8842'}, '2020.03.10.20032136-p34-118_2%0.png': {'bytes': 123706, 'sha256': 'e2430445796c9e9cfee7cc126346213da7d32eb23f72edb759697e32000ddd31'}, '2020.03.10.20032136-p34-118_2%1.png': {'bytes': 124421, 'sha256': 'b63284cd4735dc1125d9d25d2f9722c93d07f3d124cd8c09a4aecad3c4259c2d'}, '2020.03.11.20033159-p12-45%0.png': {'bytes': 76774, 'sha256': 'f9197c9c8b41a9d329da020bd434b43aa649eb3ca0eadcf0e7f63a369a0cee6d'}, '2020.03.11.20033159-p12-45%1.png': {'bytes': 118510, 'sha256': '976e55d0f9a5c22b5066fb70b5d052b30223d94ab1637d88ef18efeb33198087'}, '2020.03.11.20033159-p12-45%2.png': {'bytes': 66542, 'sha256': '6f9a5f73d48bf3b7aa166396412303c5e8ba73b92604aca30f6b906b4255e6a5'}, '2020.03.11.20033159-p12-45%3.png': {'bytes': 120282, 'sha256': 'db4c7cc2ad475ac8b2a01589c9f2a282222adcc5b168bd9c3ea43c5dd751f064'}, '2020.03.11.20033159-p12-45%4.png': {'bytes': 149338, 'sha256': '9d47ca46cd032fcc8acb42e4085d849b8fa24cf3e7471b5b1be44529d2352c8d'}, '2020.03.11.20033159-p12-45%5.png': {'bytes': 129527, 'sha256': '55d407025307055ddb6bd3a7650ba231c86970b2f9a7c6c4b7c4034b79ac5d10'}, '2020.03.12.20027185-p8-216.png': {'bytes': 115772, 'sha256': 'd44134009de29bcaa523cc5dbf86cd3abddbc2c15f6e4cab5a287c15a10cf298'}, '2020.03.12.20027185-p8-217.png': {'bytes': 97954, 'sha256': '3e32b234b526e2d196ccaa5db8e2de601bfd6a20c10ae14b23f4ed2e74f5fa2a'}, '2020.03.12.20027185-p8-218.png': {'bytes': 121202, 'sha256': 'd15350ca381de866caf6bfc098ce9d3200d88b39e85a50dd2c2865312d3f2fee'}, '2020.03.12.20027185-p8-221.png': {'bytes': 116309, 'sha256': '15242385c036e900bb1f496bb18a632707dda591551813fec93c261e91fed69b'}, '2020.03.12.20034686-p17-91-1.png': {'bytes': 61288, 'sha256': '722d26fb16e3ef2b9c6bb334e0f12d96854dfeab6a41038fb7d98defb8965051'}, '2020.03.12.20034686-p17-91-2.png': {'bytes': 58962, 'sha256': '3b0eaef392ac3182d2e80c138d23c11710d68afb31594a700628e11e8f3e17ee'}, '2020.03.12.20034686-p17-91-3.png': {'bytes': 182450, 'sha256': '07c75ce882d9dd171d2da59e0d583425b5d55015c2a2c9e756f00bd47cdea0b0'}, '2020.03.12.20034686-p17-91-4.png': {'bytes': 157705, 'sha256': '016e731079d9bcdabbf6bdfbfdecfc5dec8750022e5ba5da7f0445ff7cee5772'}, '2020.03.12.20034686-p17-91-5.png': {'bytes': 172236, 'sha256': 'a42b55f7502538bf9f202fd9f88aa7494bce978e44fd5d89afe361d430371772'}, '2020.03.12.20034686-p17-91-6.png': {'bytes': 142579, 'sha256': 'fa3587c3e4b1e1354a7def9a30866f7d39059ec7d4d2cb2f69bd8b868e52b68d'}, '2020.03.13.20035212-p23-152.png': {'bytes': 126644, 'sha256': 'ab3acae5a27e23accb4b32e89e094d5a9f2827bc39db2ef08a6ad21fa093b9c9'}, '2020.03.13.20035212-p23-153.png': {'bytes': 64017, 'sha256': '70899801466f457569ef85a11107ea1bc90dabb7c6c5cef395f6e384732925a1'}, '2020.03.13.20035212-p23-154.png': {'bytes': 88232, 'sha256': '7008cdcb5219e67fce94dc3de912dca3b46505a1185bf958e17fab2cba04ebf0'}, '2020.03.13.20035212-p23-155.png': {'bytes': 104618, 'sha256': '225a2d7c11fc344c8565ffc873b4bb3041519a36744c852ddbc0c4af83aebeb3'}, '2020.03.13.20035212-p23-156.png': {'bytes': 66062, 'sha256': '9e68c23a99436295f0d58ea3e9941909fcc46047cdf2a30432f5c476d56e84f7'}, '2020.03.13.20035212-p23-157.png': {'bytes': 72880, 'sha256': 'acc86830bed3a82d7339da368d2d76caa559367ae4005603d33db9f6f231c614'}, '2020.03.13.20035212-p23-158.png': {'bytes': 102441, 'sha256': 'c20e9d66ac3082c73782853af7ee715074dea4c9e68775aa5f412c14f21faac0'}, '2020.03.13.20035212-p23-159.png': {'bytes': 104438, 'sha256': '8641ac03f0bebdb5b3e5b864f74136df8f39ca6fefbadfb8adb88e8609d0397a'}, '2020.03.13.20035212-p23-160.png': {'bytes': 87910, 'sha256': 'b31ed86d6666e1c26be957b2622afaea8c1caf935ebcb16c38f7ba5d7ba76cae'}, '2020.03.13.20035212-p23-161.png': {'bytes': 68806, 'sha256': '6085a38d2e122292fa6a495e8b9bb83fd6b6d8c77e11b84bf3d8d82d7c6384aa'}, '2020.03.13.20035212-p23-162.png': {'bytes': 111834, 'sha256': 'b2d728098bd294b853592a8cb796c2373476651cbcd61c1332a5f6ccb1f1256f'}, '2020.03.13.20035212-p23-163.png': {'bytes': 79584, 'sha256': '33e5834b50a91efc1c59aa9abb45d246a7fb6e04a82047e95394e272be284e76'}, '2020.03.16.20035105-p6-64-1.png': {'bytes': 34399, 'sha256': '171d9f9312dd5f3dce2a5697c4fa4f877b22ff8a0f5bcc4dbde382bf0a2bf3f0'}, '2020.03.16.20035105-p6-64-2.png': {'bytes': 105951, 'sha256': '99d411cff30ad1de3ac836b1af9e0b19aeec556deb6fda59a7a4bb078938e5c5'}, '2020.03.16.20035105-p6-64-3.png': {'bytes': 70315, 'sha256': '0309046f4fdc0b03620c31e7ff9cb097eed6339951f63a41bf7ef2481396f01f'}, '2020.03.16.20035105-p6-64-4.png': {'bytes': 144784, 'sha256': '1b615b43476cca684c02a2f93c1502b7ad01e9ee09a91fbd9e63016b96b8f733'}, '2020.03.16.20036145-p19-128-1.png': {'bytes': 175491, 'sha256': '8e194922a0bd0a8873aba5615321871ed324adeb5470a7eda229020763ee3900'}, '2020.03.16.20036145-p19-128-2.png': {'bytes': 167806, 'sha256': '79a59fa5be97dc54a05e8e71ed068f37088f73e05f88e9578c4950f68dc1ec24'}, '2020.03.16.20036145-p19-128-3.png': {'bytes': 153651, 'sha256': '2b6c45da1d1cc6d94a7c816274d3fcc2090f54b644a6c89174432e07ccf2fad4'}, '2020.03.16.20036145-p19-128-4.png': {'bytes': 114569, 'sha256': '97de8dcd6aba64b53707c689385fec52cc56cabafb8d0a6ec52743ffbca62f20'}, '2020.03.18.20038125-p15-54-2.png': {'bytes': 36270, 'sha256': '78355c415ed6a1370d4e280e529e736cb18bf1548876e02ec7ee5965519888b4'}, '2020.03.18.20038125-p15-54-4.png': {'bytes': 35591, 'sha256': 'df83202381c2567a30ae05d19d046533f759db9f13f12a6bbfb6b7c9c877b9d5'}, '2020.03.18.20038125-p15-54-5.png': {'bytes': 42034, 'sha256': '7136684fcab9c0295284fb32a322cb36df9ba337e7c20e81788fadd1ca4e8e07'}, '2020.03.18.20038125-p15-54-6.png': {'bytes': 42222, 'sha256': '66f42ec3b6d3ddde3b085c696e0ed98464e3907b5b155f08cdf37322eeeb152a'}, '2020.03.18.20038125-p16-56-1.png': {'bytes': 40459, 'sha256': '8430d9b063d08b7dc47526d04178b5d378ef9e63c9e089a10fd8168a643e5947'}, '2020.03.18.20038125-p16-56-2.png': {'bytes': 45973, 'sha256': 'b8852dbd74787773e9cd57f36c2bbf4e758c42c9197486be6098253d34e1691a'}, '2020.03.18.20038125-p16-56-3.png': {'bytes': 40871, 'sha256': '33e903e7f828c3934e98dbc3029ebb9e8d4af0b16c42ac873b7c7f4848ec7cf0'}, '2020.03.18.20038125-p16-56-4.png': {'bytes': 37879, 'sha256': 'e1101d9b1f08f87c852b0eb0a09a812e516c44ba87a00625634278fe59521feb'}, '2020.03.18.20038125-p16-56-5.png': {'bytes': 47118, 'sha256': '4bedfdaa78246fbe799e472786d3d918d5725b3e3ad8d8202d0055791facf10a'}, '2020.03.18.20038125-p16-56-6.png': {'bytes': 46286, 'sha256': '6d9625b8d0d7c4829abe2f61337a6781723b8c353a296d86f41832c2413b1441'}, '2020.03.19.20038539-p10-57.png': {'bytes': 61897, 'sha256': 'a02e7c41895510fdc99e48b1df9f5d463a0869db11b85a2d92a7fdbb94a12538'}, '2020.03.19.20038539-p10-58.png': {'bytes': 153105, 'sha256': 'd2e632f349f61ecbc1cc6bff96b13b0585ad7acbd74813148c2690521e2fd294'}, '2020.03.19.20038539-p10-59.png': {'bytes': 139231, 'sha256': '48f811fd0c3463b5c8acacad1261e2d8715d51392ef5368e987e9598e7f75780'}, '2020.03.19.20038539-p10-60.png': {'bytes': 71384, 'sha256': '88207569fe7ac334ef71d90a44c2b815ce7dbc738f169d9cdf6bd90e4c45945b'}, '2020.03.19.20038539-p10-61.png': {'bytes': 125176, 'sha256': '9d8da8ede64ea1021c8a94375d60de91727f1bcebc88dc1c58a8842b8cdf7b9f'}, '2020.03.19.20038539-p10-62.png': {'bytes': 113133, 'sha256': '9ddb6f8a484e9da4a11f2c5738c0a3d6f9efd1bbc56078ca2b086b7d45f01ec5'}, '2020.03.20.20037325-p23-121.png': {'bytes': 77814, 'sha256': '60c766f0937883f56cd7189622671ff2d727f4f284f530cdf061cf3962e152fe'}, '2020.03.20.20037325-p23-122.png': {'bytes': 93025, 'sha256': '2dc0724fd9460971ae5835fc67a22340a7628710302804db156eced79e1d8c70'}, '2020.03.20.20037325-p23-123.png': {'bytes': 104614, 'sha256': '2d1a17cec78bce5cbaec21f0f4bcc93685c44d70bd49c7239a9d5db303f3a3b9'}, '2020.03.20.20037325-p23-124.png': {'bytes': 88943, 'sha256': '3fa49f341ac98f723879d965d620496b72e142639423eff1560fc3958ec28136'}, '2020.03.21.20040691-p18-6-6.png': {'bytes': 39193, 'sha256': '1b4a4b0ca13fd12fd1f2e6770f97ee69874313544f359e41e527b1949f1cab75'}, '2020.03.21.20040691-p18-67-1.png': {'bytes': 39536, 'sha256': '6a86c781f9b63a41a9745d91a5087551bb8f6c2d599cb4d99162c629d2b6da98'}, '2020.03.21.20040691-p18-67-2.png': {'bytes': 41864, 'sha256': 'aff12f1abdc325aee108fb55619a6ef61356eec02e705f6cb85a06de3da91c20'}, '2020.03.21.20040691-p18-67-3.png': {'bytes': 41080, 'sha256': '6d2c3692853821774cabb59417af191777d3a9ddcc3f17ae9ecbe8c5786176af'}, '2020.03.21.20040691-p18-67-4.png': {'bytes': 40983, 'sha256': 'd4600ace8a3250f445815b19eae0d1cc82c25d10d1051a0ab5f46b215e2954f2'}, '2020.03.21.20040691-p18-67-5.png': {'bytes': 34617, 'sha256': '138fded37959b6344b88380fa203e0e3dc2fa881706a341ec53c12c8315c4048'}, '2020.03.21.20040691-p18-67-7.png': {'bytes': 38334, 'sha256': '2b1304efa41724c982d80a82c3fd52b861dc12264225e15dd169a13b520c5670'}, '2020.03.21.20040691-p18-67-8.png': {'bytes': 39297, 'sha256': '9aa0f00878142a83fc947c8f9a918bd58ea57ec1bfb9e5dec649eed2881b67bd'}, '2020.03.22.20034041-p16-66.png': {'bytes': 76932, 'sha256': 'fdc39e6839bb0f583eedea33a80d61673cb1278dbe7dcdd0d3043a55e3841b62'}, '2020.03.22.20034041-p17-80-1.png': {'bytes': 117354, 'sha256': '34880a59ac3d8b79e050d54ede2fce5870dc8864601715ca9a34dbf5d73701d2'}, '2020.03.22.20034041-p17-80-2.png': {'bytes': 112774, 'sha256': '8f54505ae9c9c9db498627d9a64d8b80054097095405a2031861b040014b3b00'}, '2020.03.22.20034041-p18-92-1.png': {'bytes': 115089, 'sha256': '56e51e129a498ed207c50215d9fff46d88131a6ae9c0b0d07443755bb4e44cd5'}, '2020.03.22.20034041-p18-92-2.png': {'bytes': 65815, 'sha256': '1e958099ab312aae51c8b0344142d9f9ddf841b94e14492118d6bd83ce0cbb56'}, '2020.03.22.20034041-p18-92-3.png': {'bytes': 107130, 'sha256': 'a6e294398ab044d52025c51474687c7ae4471a0bf3534343ff4ff7168e84d10c'}, '2020.03.22.20040782-p24-1533-1.png': {'bytes': 143880, 'sha256': '2744b7ffb8d6aefaa663d418acd61de5b4cfbc66377f10b3aef1ab8d4a9d17f7'}, '2020.03.22.20040782-p24-1533-2.png': {'bytes': 152017, 'sha256': '59486be71c9805cb6c4395c67bf9e0af6fd85561fcb30a8a997254a6331b1fba'}, '2020.03.22.20040782-p24-1533-3.png': {'bytes': 155692, 'sha256': 'bca6034ade71ae55e4afee3501a8e6bc0a93644306f979e86b502c1abead0cdd'}, '2020.03.22.20040782-p24-1533-4.png': {'bytes': 118146, 'sha256': '25a4f36d1f94703b966b271bbadd0e7694f6d0721fd2dfa24fa7353e96cedd90'}, '2020.03.22.20040782-p24-1533-5.png': {'bytes': 154532, 'sha256': 'f42312e0312f4927537bd3dc2da284f916ca797eaec1ee88043b3b4d4ce14b37'}, '2020.03.22.20040782-p24-1533-6.png': {'bytes': 123132, 'sha256': 'a16cddafbc502f64a765f83fad41a70649c6832d6f6ecdb23ec1dfb171746ea2'}, '2020.03.22.20040782-p24-1533-7.png': {'bytes': 158296, 'sha256': '7bf5e6e32f97942072e409adf631fe9773e4ec94645d5db00db633893c48529c'}, '2020.03.22.20040782-p24-1533-8.png': {'bytes': 92399, 'sha256': '0915a5800e1f9a876af9f1d706d9720f6b6d6412dea5c8ad772038401fa83e49'}, '2020.03.22.20040782-p25-1541.png': {'bytes': 394008, 'sha256': '7b32d74410119e2a41f52660c3a40c50d53443711b31379d3de24dd19e31b885'}, '2020.03.22.20040782-p25-1542.png': {'bytes': 318695, 'sha256': 'c04242c5c368590f9d421d67e0aeb0714aa96351f9acb868b52fd3cd60646abd'}, '2020.03.22.20040782-p25-1543.png': {'bytes': 175056, 'sha256': 'ac6eb7cd5e43376229f7b8fb7c44fa464bb4a947b9166d43c7f82d3a005ad081'}, '2020.03.22.20040782-p25-1544.png': {'bytes': 262899, 'sha256': '9f277cc586f0e4add84c0ab44c42b66fccfd8817fe9b5a0ab3476844d4537221'}, '2020.03.22.20040782-p25-1545.png': {'bytes': 147285, 'sha256': '8c2c17b5ac7c336a0c59645e7906444a3cf66997ddcf0509b57f88d4e0ab537f'}, '2020.03.22.20040782-p25-1546.png': {'bytes': 267735, 'sha256': 'dfcaa00ec8c2577bea2f8d8c970cc191d5558fd57915064d36164408ee605055'}, '2020.03.25.20037721-p27-165.png': {'bytes': 71592, 'sha256': 'fbd67352653cc361c308a53ecbe07a9644553691984bdf2b7e24cd7df058ab25'}, '2020.03.25.20037721-p27-172.png': {'bytes': 155773, 'sha256': 'ace739bcbff9a2b5e9231044e5dff7194d47433be4ac55cc68c05b1502e01603'}, '2020.03.25.20037721-p27-173.png': {'bytes': 172796, 'sha256': 'd2d6bd8d49b0f37a989bb4dd9f39f26a73799acaa02f1068d0afdc1b09ebdac9'}, '2020.03.25.20037721-p27-174.png': {'bytes': 169729, 'sha256': 'a162b448b35b06f52a3c8bc4dfe930da3fb12369727fffa03647f469eeebe2a8'}, '2020.03.25.20037721-p27-181.png': {'bytes': 172304, 'sha256': '3f18c92b41f8ba22ecfa7b9de2cac7d3808f2e6f720e8624a40eb3b64634cad0'}, '2020.03.25.20037721-p27-182.png': {'bytes': 173875, 'sha256': '4bf7621a24a7e0dd7ac0554b9503c62cd52e3396fbf77251d28aa544e8e04b83'}, '2020.03.25.20043166-p17-78.png': {'bytes': 19722, 'sha256': '1ba438ce0424443c528d464a3fdf735f40a46902fb88f940c57ab1f564f06987'}, '2020.03.25.20043166-p17-79.png': {'bytes': 21201, 'sha256': 'b7cdd09d8aab75d81ce88b98062a3ccebb3f5330db39b0d391f4bb9f64c5d7ef'}, '2020.03.25.20043166-p17-80.png': {'bytes': 18926, 'sha256': 'd3db9c48d8e8c7518796fbfc66edb9d9aaa2443d2216f8461f81f8f71c5e10a7'}, '2020.03.25.20043166-p17-81.png': {'bytes': 22453, 'sha256': '842d7333b6c148599c2f688988195b30f688a1cc934fb7cec5b2c74b2286eca1'}, '2020.03.26.20041426-p11-119.png': {'bytes': 86015, 'sha256': '298c46df123e762b3318d2de6d4a94095d6059272b3bd0adffb25bb07057e371'}, '2020.03.26.20041426-p11-120.png': {'bytes': 83331, 'sha256': 'c3ce6a8d8c527434f4f6f6f99f64e5e4595ddfb2838d5b5300628601f46efd51'}, '2020.03.26.20041426-p11-121.png': {'bytes': 76028, 'sha256': '3cc62175e9e97edad3318b6880777d3fef6ff785c51834e0d924e4adcd8ae2a6'}, '2020.03.26.20041426-p11-122.png': {'bytes': 85296, 'sha256': '67a7924cc1d4eac4f350ac83e6314cf9a927fc4a96b8bab781823d20908414df'}, '2020.03.26.20041426-p11-123.png': {'bytes': 79826, 'sha256': '4f2bbe853316818fee9bb09fe513beb394a031de6b9f3abfdc29ac28fa945fce'}, '2020.03.26.20041426-p11-124.png': {'bytes': 93332, 'sha256': 'bf1aab7e10d579494e45af376bbdd8bc3929eff7e36d4ad343ddf8b3fd13b723'}, '2020.03.26.20041426-p11-125.png': {'bytes': 78146, 'sha256': '748feffe0bcf811a198af83f7cfe1afb3f7c7e9addbde32a551aec4aa966cbfb'}, '2020.03.26.20041426-p11-126.png': {'bytes': 92795, 'sha256': 'b4add170df5daa8c79baf2a4f0b4252152cfb64632db1b2182ee7dbc925bd8a3'}, '2020.03.26.20041426-p11-127.png': {'bytes': 75961, 'sha256': '8c8411ccba3bfef972766a79c7284f3d8996af06a7a98ce41edec4ca27a46dad'}, '2020.03.26.20041426-p11-128.png': {'bytes': 89178, 'sha256': '1809e916e6dec100c8ebe93cc8ef5ba6e0b42ab7b121535cb3fcc4bc5c605b36'}, '2020.03.26.20041426-p11-130.png': {'bytes': 78919, 'sha256': '68ce16b5cd192d72baba582afc5073207c80fba403020b048aca877963be668e'}, '2020.03.26.20041426-p11-131.png': {'bytes': 75651, 'sha256': 'f09c09bed6da006ed31e045d9f705298761d2c5ac1db06b6dd2f926811965351'}, '2020.03.26.20041426-p11-132.png': {'bytes': 83115, 'sha256': '9782e48f95b088e2725609cdfb4cce49e7a35fcc77a03a727d5b3fc456f2139c'}, '2020.03.26.20041426-p11-133.png': {'bytes': 83287, 'sha256': '44dc56a6305b00b70212b33919c86ed715f5b1511892ade7f63f7a0c055820c3'}, '2020.03.30.20047985-p9-56%0.png': {'bytes': 104007, 'sha256': '121637d4932c0f10b34fb797d6b5a1e0dac27069195690af0bb3c9d60cf7a7b7'}, '2020.03.30.20047985-p9-56%1.png': {'bytes': 112513, 'sha256': 'e96da6eec654b11fff874e05a2cd926ded25b0569699ceaea35f1fa82494b7c3'}, '2020.03.30.20047985-p9-56%2.png': {'bytes': 118665, 'sha256': '0bd257eb1723c25d9d73e7e0413795f5a6ec040f4fe0d6a32b0a01dda086dbdf'}, '2020.03.30.20047985-p9-56%3.png': {'bytes': 151831, 'sha256': 'ee2a1b696c781b77755323dc2ef02264b35ee61f36fbe0f47079e838d4a013d8'}, '2039.png': {'bytes': 177048, 'sha256': '1bf1ade136601e74ef41930f52dd7fad0b6b607fe2fd72406cc92a5e0f093d16'}, '2040.png': {'bytes': 171808, 'sha256': '360845a325a61c173d21bab76913e4b76419352f9cacbe11047d05bdf22e62dd'}, '2041.png': {'bytes': 168721, 'sha256': 'ff5f013b98d43c7aa21ab5e62d9ac2607c432d9b3c588ec9c671e781260157ed'}, '21%0.jpg': {'bytes': 109299, 'sha256': '502f30463f6eff2b1ab8f052e9c9c3b3ff0e83bc04727c02c1ed61aa769fd29a'}, '21%1.jpg': {'bytes': 109233, 'sha256': 'bfcc60e4104bb5a80ed6b313fbb7621cedd017fef1321faead89ac44db38f0a2'}, '2108.png': {'bytes': 305232, 'sha256': '4b939df74b7e5d25f65bd53742c1b21e11bba844e8e2f596b1a3a0a78785ccf6'}, '2138.png': {'bytes': 67636, 'sha256': '3516dcb7f7f56f6edc737a43f3812d9b7790e4af4cb97f47c07f92a93cc54317'}, '2139.png': {'bytes': 69272, 'sha256': '99e41e50ce4f4ca4473ec9aaacdb38131faa0000e3381e654e588af6c43bd945'}, '2140.png': {'bytes': 67366, 'sha256': '1b8aef7565e6ba092be3ccaef4bf1d8ea9285e38992f0ef77eb1c40a2f7283e7'}, '2141.png': {'bytes': 67634, 'sha256': '969a3fc3a388d99d03f523784a71b7577b67fbdb9f5ababa0fce4d92d825ea7a'}, '2142.png': {'bytes': 69175, 'sha256': '78bd78429b6ea74ccf2fa233ee94a46584bea84ce8e788414caa6458e34b1b35'}, '2143.png': {'bytes': 68453, 'sha256': 'd195f0195b10177208ea18dbbe08fae7f0dd79d36721a41eff0c93269df3d5de'}, '2144.png': {'bytes': 69833, 'sha256': '5d66e97c109e1eb4f6125681b0677b4a4e82b32fc37dd3da56145f715a4f38fa'}, '2145.png': {'bytes': 70188, 'sha256': 'a6de27ae33e9050b271969ac20edbb8fe5c0094b228018daecd4a47c58ac9cc0'}, '22%0.jpg': {'bytes': 54855, 'sha256': 'be41863d267aaa3c8bddf986754dcbbc6c2697f36d9d8ff05a4089f5c6637d01'}, '22%1.jpg': {'bytes': 58505, 'sha256': '507824012ab5aebe0aa907d16bec23000fccf089f8e238c7b5d92974f50405a3'}, '2237.png': {'bytes': 210441, 'sha256': 'adf9e6447e7c2521fb156f7d20a51b8fa37b69eb917ccad1c8f6ce684300934e'}, '226.png': {'bytes': 179089, 'sha256': '9aceee97fdd1c59a32ad3be066099d528a1a01c659722f693abf4e4b9cafb634'}, '227.png': {'bytes': 231145, 'sha256': '9796d4dff52a25c15945d8450f09aeb49d8d8d6319b814473a1bd741e9ca410a'}, '23.png': {'bytes': 152692, 'sha256': 'cb6b4beb44ca9a0baaf148bf82e26167daa2c74e5d461d00f9ff94b7b6a205b9'}, '2341.png': {'bytes': 229759, 'sha256': '48d1526fe84a71d5394073aadca847730990164e5275b70464e0d87a9823d77a'}, '24.png': {'bytes': 144126, 'sha256': '719aa461ea210f83bef21f4aa9b68cba7f2338ec0e5aa81e6fe07debb6e9fff6'}, '25%0.jpg': {'bytes': 59874, 'sha256': 'f6f6aac5f54078b5ed7c2c9605b7b4adda4c7e946a51905b1601f8812ebb9fa5'}, '25%1.jpg': {'bytes': 60209, 'sha256': '07eff948bc71cf91ecbaea9a5c9b15a9a2de656e5c8b960a1c94ba5151f8c354'}, '25%2.jpg': {'bytes': 85114, 'sha256': '1495a7e9ed347b7f1773d6c2b144177abce0c742b1ecbd45ebfba176f1a8e0fa'}, '25%3.jpg': {'bytes': 72592, 'sha256': '4869a98b2b71b19728633873a6ab12c66e4266d47213a9866f31a4e90baafc0b'}, '25.png': {'bytes': 156037, 'sha256': '22ea4b831620c3991e540cf6f5e4a265bccf71ea87e222bfe310a7e416ad06ab'}, '26%0.jpg': {'bytes': 60019, 'sha256': '28776fcde4ad5812612d91e3be1ade345c5a93553b8b90a82c82572387ee5eb5'}, '26%1.jpg': {'bytes': 58803, 'sha256': '905c3ca63ab856546ea28684c616f699067164685a9d09aaa22a5692d1b5a443'}, '26%2.jpg': {'bytes': 81564, 'sha256': '344eff2f6818754074440ce376e9614824e14501bfbc76536cadfab6e0afe634'}, '26%3.jpg': {'bytes': 62637, 'sha256': 'b99ff053b7d26a8fa2c2cac1f8309b249211677e0a315056ca38323e5fb07d7c'}, '26.png': {'bytes': 157349, 'sha256': '65395c5594707f6457041067acb00555427304026215aed5db077b030b2e03aa'}, '27.jpg': {'bytes': 137224, 'sha256': 'c352a5d90540803c5545ff485fcdd02d230fb230f36834e49626fc544fef267e'}, '28.jpg': {'bytes': 102059, 'sha256': 'c5d180f8baba9f6b81d8362a57b74028af133c415cc2922d52bd22ed272b1d59'}, '29%0.jpg': {'bytes': 55301, 'sha256': '02eafab93a3587bb9783393ea628a134e78f204d2f6e3f8545abce2132da6df2'}, '29%1.jpg': {'bytes': 63462, 'sha256': 'dce409810ad9eb9e309c0ebd6af665d30319ecdab7e16f418e9dcb526e8b966c'}, '29%2.jpg': {'bytes': 55342, 'sha256': '07b2deaac12b22584d9271569d3d2f1817dc6ac81f834c3f098573971cd4ae95'}, '29%3.jpg': {'bytes': 57805, 'sha256': 'aa968ef400f8e7436f027f00dead1fde361442807cbe8617745fa3d662bd8aba'}, '294.png': {'bytes': 144056, 'sha256': 'ed1195b893f358dc4d485746800272444e4d60bb2f2eb9afe2e92b14564e8c62'}, '3.jpg': {'bytes': 143398, 'sha256': 'b88087a3b5775268d3e1751dc3d688ff68de06a2b4f3215bcd1902ee1ec3b057'}, '30%0.jpg': {'bytes': 51366, 'sha256': 'ba7f756799e0b50a54a1c19ced4457159175ca098f4303b56c371c3f631a1f5b'}, '30%1.jpg': {'bytes': 54583, 'sha256': '3d7879c3bc8e2ddaf1328aae61458145b6e3a5f6d3eca7fe76250815f071423e'}, '30%2.jpg': {'bytes': 48679, 'sha256': 'f39c971d20a82fcbace82983d784b1c8ce11a5847fe9ebc622cd2dbc54211244'}, '30%3.jpg': {'bytes': 49176, 'sha256': 'a2ca17bab7216eaed21d185212bcf699aac13ec7428bc5e588b858378e6d15e1'}, '30%4.jpg': {'bytes': 57280, 'sha256': '387cca1f19d4ca3cfe598c812060cd193593b41cabd0513faff26c328acb7581'}, '30%5.jpg': {'bytes': 57514, 'sha256': '367a450b10ed89be5018c726471ea0ed461fdd9ccb3a74eec2656392b57c479f'}, '31.jpg': {'bytes': 77250, 'sha256': 'fc1f402f3b4f61cf8877038915aad11b2bacd60d0e2015c8736ef1340518a426'}, '32%0.jpg': {'bytes': 75049, 'sha256': '3c102e87356d86faf4719901699115af56a7acb2f0bbff70368eb15fb7452ab6'}, '32%1.jpg': {'bytes': 60015, 'sha256': '0d0979d805f6d4f100c90ad2ff9b9b779c514d67c19fd11bf678188ab6a02c23'}, '33%0.jpg': {'bytes': 61553, 'sha256': '69df94662547b3f5cb65117070574e1609ccd5266a1da3580a466559f2958458'}, '33%1.jpg': {'bytes': 69417, 'sha256': '8cfe1e82786cd98800ae3fa1c8f796176a805e0b56a18083e518c4e52e584d23'}, '33.png': {'bytes': 75320, 'sha256': 'e323fe79ef9eb5f6eb057e69f2cca8cb83925753adfe665e9cd019924bc54aeb'}, '34.jpg': {'bytes': 106721, 'sha256': 'de0a9424dc6203f7445fde06c38f2b71269fa17b2f07a3159f333f8891fbcd15'}, '35%0.jpg': {'bytes': 118089, 'sha256': '1b8887b037bcbef844456e1d410952af6c684c2350da59a4763bcbcaaf450419'}, '35%1.jpg': {'bytes': 134326, 'sha256': '0c75e7007ce909bff926f35da6c2bb902e5d406c694a3dc6d9cf5d1894cb3f31'}, '35%2.jpg': {'bytes': 133674, 'sha256': 'd38fb326a8fbf63b3e58b0109876268f9f22bacea5eb8cc082a894cde007cb3e'}, '354.png': {'bytes': 164729, 'sha256': '838ab6ccf22383adab7e82b04484b7c94241938089246be88e03793bf6e23b51'}, '36%0.jpg': {'bytes': 52088, 'sha256': 'c57464fb33b618d605c5cae1b9a1f517cf2375b7b74e5a9347d5bf56341064f3'}, '36%1.jpg': {'bytes': 65414, 'sha256': 'abda6304425b00e902644b8674def3d89a732922fdd3b7a2ef12db0d2a1f6d5c'}, '361.png': {'bytes': 227998, 'sha256': 'c4b98fbf774a2e3e0830229b6f0c42a92953dcb2e4681cf047cc21d21b5dc8f9'}, '37%0.jpg': {'bytes': 43972, 'sha256': 'c61851d150a29b87de31010bb092287643b2b0a437594808adf07268dfea4207'}, '37%1.jpg': {'bytes': 43641, 'sha256': '14c3bdb16eb94b816c5d245c45c3d38f0f0822dc320df64e0bb53255c91610cd'}, '37%2.jpg': {'bytes': 51260, 'sha256': 'aff9441e81642b8f8fbfd38bc75205b420c86551db54a936007bce217f8a8505'}, '37%3.jpg': {'bytes': 53883, 'sha256': '855f44d23c49daf3790ed101430c7e1481f43541c1ba4eeda7dda1800f1f3018'}, '37%4.jpg': {'bytes': 50380, 'sha256': 'bfd35387f5d883a71deb04e7f979c9844026e456153b5ecf46d71ca441efb746'}, '378.png': {'bytes': 256569, 'sha256': '3b5fcb501492672ec65a3711b1e311b98f2aa16a795d8d12dbf18e52ac5b9947'}, '38%0.jpg': {'bytes': 33265, 'sha256': '681186ffd86caca3c0cc901ed9d658f273fd8bf845ce1bfd4f9e19d2fda117d3'}, '38%1.jpg': {'bytes': 39253, 'sha256': '9f830c46a8cc498a546c7befb1bc31d51fc9b909081f6d0e0543e903d6d586ed'}, '38%2.jpg': {'bytes': 38101, 'sha256': '24dbc04943e61bcae6fadb02eb927fa9bcfca2bab8fe550b08824419ea60c21c'}, '381.png': {'bytes': 178096, 'sha256': 'f213a69b908f835b03095e703aa17a875da757bdb9f0349799e387e3dbaa5ccf'}, '382.png': {'bytes': 171585, 'sha256': '624ba8f7bb84e273e50c051514c4633c4f06a83dc667b44b262e1f3e61f766dc'}, '383.png': {'bytes': 188634, 'sha256': '88f13a964b2e77a618128d05c89d3912778e623b49f67218da95e4e897bb7eeb'}, '385.png': {'bytes': 171742, 'sha256': '63955d88e077efc10af1acc563823691d12c787ba56eeed73fb1c8bab46e8ee3'}, '39%0.jpg': {'bytes': 32742, 'sha256': '04b37d6392909550e2cac8f22e52a3ee3475a42aeed54ad8c83ebddac8f09822'}, '39%1.jpg': {'bytes': 36809, 'sha256': 'd69ace35ba3599c5483417ac8376d3875778c60d4154e272cce4bdb0781f97ce'}, '39%2.jpg': {'bytes': 35149, 'sha256': 'e689352ff6122f0529e4f21b4f8870ddd47544dc40a5ade2e0776a20330787bf'}, '39%3.jpg': {'bytes': 40246, 'sha256': '0240b79cf594e2e8c098d6a8c8d43e2d97ada37f08b912a86ae109e3796adc24'}, '39%4.jpg': {'bytes': 43489, 'sha256': 'ae8be686127c33a9cb7a33893f07208d286b291fab87e34aa132b03e7668cea8'}, '39%5.jpg': {'bytes': 40922, 'sha256': 'd6498de4bbfd74af6b6f67697f99923a43a3f64fb044b3b27da523220fdeb606'}, '39%6.jpg': {'bytes': 40190, 'sha256': '2a3640d8fe485873b63d2a76ddded5254b2b6d970e1d9ff776da4a1b46f2072d'}, '39%7.jpg': {'bytes': 43597, 'sha256': '18d6ee57f045d6233507e7167d0c6e1a8fd52afcbbc10de5836789cd961e05fc'}, '39%8.jpg': {'bytes': 42047, 'sha256': '0c7ee1ebeb3e9179d577536765ae9ad9fe0eeaa65189d61095c2058e818c0234'}, '4%1.jpg': {'bytes': 42713, 'sha256': '63227eb5742c2466060e8749798b58a20ac07c1dc16fa56bcb4eed999303b143'}, '4%3.jpg': {'bytes': 38437, 'sha256': 'e14184ebcc4066c827770bdac2e976d16e73faf9c5bd2928833bd665870d2d0a'}, '4%5.jpg': {'bytes': 39333, 'sha256': '86b27dc3345ebd5a266f35bdf7d5379758009e50878a8b7dd5fefa2e9422465f'}, '4%7.jpg': {'bytes': 34725, 'sha256': 'f31b459aca1a84dda56498d06292cb4e7e76424993a346329e15b10b805ea362'}, '40%0.jpg': {'bytes': 79100, 'sha256': '20ffb7b45fd77328fd6fd70d3461fa12a85e7fecdb5ee0e650e26ec5b56bc2c6'}, '40%1.jpg': {'bytes': 77646, 'sha256': '7e0f82c316d82f80a94a22b3ae06942a5b8d844b249a914cd2e5fef2d6191897'}, '41.jpg': {'bytes': 161840, 'sha256': '71fb15afe16f5ab6556862ae1e7804cfcac7286e979e2285e97f08336730d628'}, '412.png': {'bytes': 242362, 'sha256': '765fe3843a88d27c2fdf35f7d33675d5c2c5725a3a4e343e7497c9b5510f9e52'}, '413.png': {'bytes': 229804, 'sha256': 'c5c7ca54380a6a5f8d5b0926d3dde5c026847da619156cfb9c6a88b7c406e599'}, '414.png': {'bytes': 225899, 'sha256': '1115f5ccbd57e3d7de561d3a891aea756c3af478c3302c8a0354d0b79689da74'}, '42.jpg': {'bytes': 193441, 'sha256': 'efb71292558903d8c53e3b298b1f3621c8f3f2b9056977a91996cb3ce38909d4'}, '43%1.jpg': {'bytes': 51256, 'sha256': '0e4ffdb2cb586340a11899462c2fb5983ebae1739c5d32d65586d108c883f4d2'}, '43%2.jpg': {'bytes': 52398, 'sha256': 'f20d3b1cece20ac52fadaf8d88f339d161e0c42de970fc5e63246c5ef50260c1'}, '43%3.jpg': {'bytes': 61975, 'sha256': '05f9eaf080f12e6e94c4ae348b27702540f0884fa6112282313aeda2bd697848'}, '43.png': {'bytes': 217886, 'sha256': 'd3f51cf29ceab9044c8b23a79b6abd968f313e56167e5fd4a0a2b8e3b85aaa33'}, '44%0.jpg': {'bytes': 73553, 'sha256': '9a48f4282a0560cb779f8fb1212d18b9f16d1946ca10e1af2a269766a856007e'}, '44%1.jpg': {'bytes': 76823, 'sha256': 'e6b75e0a258bb31f7a0e5c2ee7b9cfdd95fe906a149fc6c33a1264a79876e341'}, '44.png': {'bytes': 224317, 'sha256': '4ca4aefff6f309ffe054d7928dfc3785082460372be6943583beddf73b1e17a7'}, '45.jpg': {'bytes': 64824, 'sha256': '1eae69fd62809f59c14b5109d5fe068871ad10e20887fbd04cc599c9f717704f'}, '452.png': {'bytes': 210545, 'sha256': 'e648c98a2130095ea8bed0194362ce9d9c50e9d7d362adea7104ff7828e6e03a'}, '46.jpg': {'bytes': 183015, 'sha256': '32b5de2222b3c2b8bca0016d9692deef2a39687b74b09491af57b1fcb1116742'}, '47.jpg': {'bytes': 207205, 'sha256': '7c09e07df776970fcb04ba53727e5c9e2deea59d6fc5288fd4e21b57794bae9f'}, '48.jpg': {'bytes': 120875, 'sha256': 'b2306809df69238de25a2d49388f4e32b95e764184c7e90d5899bc741e83ff8d'}, '486.png': {'bytes': 128624, 'sha256': 'f4d747ceda5e33137fbe8c7c3e628da15129aa0d22bced32385207fe28a1cfdf'}, '487.png': {'bytes': 165505, 'sha256': 'd7b043722b7d0f0fc98f4fecbcfed9801daaf45c486adeaa0a83691dc0b74777'}, '49%0.jpg': {'bytes': 82538, 'sha256': '5916b37e7ca44c446140f82051bfb3928f9acbe4c88979da43dc91196426928e'}, '49%1.jpg': {'bytes': 77398, 'sha256': '52f28ea4064a7a00b492c7c44fae5c3c94701dc8978d26d84085f0b55457cea0'}, '5%0.jpg': {'bytes': 23932, 'sha256': '6114fead0c757e081ce451d39a20b5b733b8990337cff354c0c077da3bbfdc7b'}, '5%1.jpg': {'bytes': 25743, 'sha256': '208921b6fce9a9ea773268316e1c0b7da4626fe897693f97bc2955b0d79529ba'}, '5%2.jpg': {'bytes': 26014, 'sha256': '3a25889e409a4f93e8fa2143a7a2c40fd5573af579d425e63ae6b4b17b1d9793'}, '5%3.jpg': {'bytes': 28034, 'sha256': '47db01c2d95b9269e77306207cccf5373608c4322fd0a48de46cceebf34f4cfb'}, '5%4.jpg': {'bytes': 28418, 'sha256': '80002eb6be2899d663a2e504da2a1e0f2b656b9ee57bfd54c98ab50edc0c7ba9'}, '5%5.jpg': {'bytes': 29160, 'sha256': 'b5f15dce765235fc215c749d60d4fcaf40fe20a152e7cbd76c97e9761b169371'}, '5%6.jpg': {'bytes': 28922, 'sha256': 'e2600e569ad59530e114e28cb33eaade2096f0f21911645985cea72d8c6e3a74'}, '5%7.jpg': {'bytes': 28771, 'sha256': 'c9c1d51cac05d4ff57c8b131a32c9e22a05663676d93d2d61e27888c2e07effc'}, '50.jpg': {'bytes': 105742, 'sha256': '80a8f91b5dce48e87cee14df1a6e1defdca82cca503923c5ca0fda0372d6425f'}, '51%0.jpg': {'bytes': 60885, 'sha256': '5f0ffa556df573cc950f00ab554acde83d048aa8ec1282ef53e7027ab466d899'}, '51%1.jpg': {'bytes': 48622, 'sha256': 'c847b3f8343b7cfbee5e6b826b98ad93559531b7e70b985882670a57aae75c2f'}, '51%2.jpg': {'bytes': 56320, 'sha256': 'eec942ea2e746d60c87d8d9667e4c768fd3e0c903e07893d7c7afcb7ba86d41d'}, '51%3.jpg': {'bytes': 54349, 'sha256': '36669aad8672dc61e0c50bdd4b644897cfc630a0dad215677fe52a7a28a9d1e8'}, '52%0.jpg': {'bytes': 97872, 'sha256': '66425153bd2f84359741335db447feb7596ead4a868c4138b57dddfa69c4ef2a'}, '52%1.jpg': {'bytes': 97871, 'sha256': '2569687c13d56bc5cbc2e2d94d43b2aefad3ad220646bd148070ed46467d89c7'}, '53.jpg': {'bytes': 165963, 'sha256': '8b37c97bea7b12c2ab41fa1ed531c7bebed76dcb462a832f7247d2b75f07e3ad'}, '54%0.jpg': {'bytes': 53615, 'sha256': '89fd9856d2117b3ef9204e29519395128cb81713e590d69a9153c6560892aba9'}, '54%1.jpg': {'bytes': 66815, 'sha256': '5c942bdf9c32e6309f7157d704af079e7ad33a4dadc18e2ab806d24af0a8220a'}, '54%2.jpg': {'bytes': 58093, 'sha256': 'c8dcb22463ad8fe90b0c524715c0a5f40d8b79b073120ba393b58b15c1eec687'}, '54%3.jpg': {'bytes': 55217, 'sha256': '220b9cc1c7cb4a0fe6f7154f896666f78f2ef8e8c9751d233bbb25264a99c97d'}, '56%0.jpg': {'bytes': 79083, 'sha256': 'e9ad6a2edc6fe38922b6bb24b8f7ee4dfe790dd6ffd1042e054b7d534a99ba79'}, '56%1.jpg': {'bytes': 81949, 'sha256': '4dd17b851280d6d7340330e15ac03c837a7bde20359df322fbdf32bdfcf1d430'}, '57%0.jpg': {'bytes': 39063, 'sha256': '1a07469a09f29d1b12b1562b47e4682df4531efa32041a2b0c35bd3acdc60707'}, '575.png': {'bytes': 141151, 'sha256': '9c2a891f3b82ef8f2c5bf626433884d20949a928a4673387df201de13ce09016'}, '576.png': {'bytes': 132719, 'sha256': '19df7cfb9e1321d259b3a7d889428966dcb18bd5db70874d2ba029c56e065590'}, '577.png': {'bytes': 86529, 'sha256': '4d8832bd35a47f1c009e66fb829092665591685a6707e08e4e150bc3e4a8f52e'}, '578.png': {'bytes': 86193, 'sha256': 'e0317c635269d0c854a3a4f8d3b77c78435defdd852fa772a7c936df686114ac'}, '579.png': {'bytes': 88388, 'sha256': '2e14005570b95c2b1074d89284e6954aeee715b5eb0561bc13f5e0cf51543cf5'}, '580.png': {'bytes': 87308, 'sha256': '569511ba934188d0a6fc528fb8349e336463ef5190498ef796df76aa2e71cb05'}, '583.png': {'bytes': 140924, 'sha256': '433826955d6dd2e0b2d0ac98f599258139046be990c523536ea55d8b3a59e95c'}, '584.png': {'bytes': 154961, 'sha256': '99c711c139278f06dbf52bba9a90a2d0647c9dc6dfc3eab31de467b709140b07'}, '585.png': {'bytes': 142065, 'sha256': 'a980a26ac3b0f316cee109f60d1b651fc6e5c78d19e352ac8eef694869d6d355'}, '586.png': {'bytes': 156115, 'sha256': '6c6ccf70a659b42aaf455f12dff25888ab743dac319ff1b4ef0074fa2602a9ac'}, '587.png': {'bytes': 142966, 'sha256': 'b0eb90125f887153a441ab60d6acf68c57cadea48a9b4bc4d1720ff1318ace19'}, '59%0.jpg': {'bytes': 46023, 'sha256': '13abb002c858f2a56eaa97f55e44968942977827a67abba4dc91a664c25e5ccd'}, '59.png': {'bytes': 132159, 'sha256': '01ae3899e2078b16e2b81cd63500c790779cb46f26f0c3d74245cae23f81cc86'}, '590.png': {'bytes': 145220, 'sha256': '2d3074f135e1df2de5eb3615d1771f0f70cc774321d11ee125aea7205687576f'}, '591.png': {'bytes': 137899, 'sha256': 'e758d03c7693f3c95d676c92c0fe85d1a7ddfabb1349c28d7d6770c568a4268e'}, '592.png': {'bytes': 126438, 'sha256': 'f6a3ff2448e1e3c19d88c998a7eeab7bc636c2280fc4644d5c89ba962be23eed'}, '6%0.jpg': {'bytes': 37065, 'sha256': '318d45c3e2f990ae2a9d9954656ec20293bb607bb9e578b6209fde6a254acbb6'}, '6%1.jpg': {'bytes': 35786, 'sha256': '10faa93010d37cba5f170130b73019c8c60b2434fca5165269c2dacaf22785da'}, '6%2.jpg': {'bytes': 35845, 'sha256': 'c1b2cc56f2793ab140f34de4fa4dc52eea5f8a72543cd7d3279011849a42079b'}, '6%3.jpg': {'bytes': 36346, 'sha256': '2feb21ab1630f508780e117d3af36d5a57f03d79008c906a6871fe93fff3229b'}, '60.jpg': {'bytes': 78057, 'sha256': 'aa2f9efb4a357bacc8988793da64e695e99509ad07bfb730f15d85d5c6766507'}, '62.jpg': {'bytes': 164253, 'sha256': '594edf87ae4394a8761b046fa9d9892255d04e899ca46a760baad4a3b96a25f3'}, '63%0.jpg': {'bytes': 33148, 'sha256': '98754d4eab2e15e30f8ad47e7a955e3f31640a981fe142ccbf2a294b409bcc11'}, '63%1.jpg': {'bytes': 40568, 'sha256': '902be8746276a1da7e0a0bf2a0b12e8ae35179344a5a35a761adaebbd75380cb'}, '63%2.jpg': {'bytes': 32966, 'sha256': '5a6ce52e96361dd648c2521dc96a4c70186c92eb5e16b80585db8f6eb4046a3a'}, '63%3.jpg': {'bytes': 41876, 'sha256': 'fdfdb8f06805dbdf2a43e3d664116718aa73d07573349443bccda97ea79476c9'}, '63%4.jpg': {'bytes': 33564, 'sha256': '371af7eb66d39575047c76b094166e8bb6c72ee96d939391ebbfd72ba705edd2'}, '63%5.jpg': {'bytes': 41918, 'sha256': '88dd952c044cc0dbb16232162c3d9662182e65cebe2f96e95fdc02ff595d341e'}, '63%6.jpg': {'bytes': 60409, 'sha256': '511446dfea8b741c3314928bebf21424bf8f07e8095fe91bc4ed79b66ba14c2b'}, '63%7.jpg': {'bytes': 74812, 'sha256': '6bfeaf6562c5599f29f7a9c0c38ac74d61a147063584f50763ef975c8ebe7f45'}, '64.jpg': {'bytes': 53694, 'sha256': '9f2220cf1d5af4648c6ba0821823e51f528810129f07283e0cf26c3a1ec29b25'}, '65%0.jpg': {'bytes': 43443, 'sha256': 'ba709b2a39ae1b21e6ab27ba9738d213ab4f914364a5805d14410f03f8fac4fd'}, '65%1.jpg': {'bytes': 53375, 'sha256': '2875be3e51c376470f1d7e193f79845362ef738b1a10f6b4284f54269826f22a'}, '65%2.jpg': {'bytes': 49938, 'sha256': '8285daca5e92dee262c0535a1c8c9e6c275f6e3fb7d71f124039c86ed3730957'}, '65%3.jpg': {'bytes': 50362, 'sha256': 'df013373584a24f77e86e69bc2025f218bd6b108536e4ae767526173d60f02e4'}, '65%4.jpg': {'bytes': 42393, 'sha256': '2ad347d32c95fc5cbaed701c5f2a6e5715c35df0ec3592a6739162efd2116221'}, '658.png': {'bytes': 188171, 'sha256': '6bcb5bbf7cf58dae882843494ce69a1b666520e2e34b8af4748166898263863c'}, '66.jpg': {'bytes': 77150, 'sha256': 'c08c7b132257ce6073c8efc9c04826e08bb598532a12a3a65a735c1ccbafa33b'}, '660.png': {'bytes': 183604, 'sha256': 'f873113544dad94aa274fcc5bf98a62704c07f17fd9bf31e4994879efbf212be'}, '661.png': {'bytes': 174593, 'sha256': '1c79ffe55f2e00b844f0ace9471ec21c7032a9a4a885585befed12c3add6ebfc'}, '662.png': {'bytes': 177267, 'sha256': '29e00c50c833861e6947eb49f1761b33f1fafa6c8a68c0f5b47035866eb95de6'}, '663.png': {'bytes': 193607, 'sha256': '3016ed65ad63af547688fec376807c5b0cb52bcdcaa86564ee77e54e90d067df'}, '67%0.jpg': {'bytes': 26556, 'sha256': '428aaa3f23f576ce629ecafe38650bca4a42ad20b9e4d0335d00df4cbaa43dbd'}, '67%1.jpg': {'bytes': 24822, 'sha256': '5dd238fcb19c7d4922ff3a75a88818536dda915d3e98f92f02f23a633b79eaf6'}, '67%2.jpg': {'bytes': 26490, 'sha256': '09dfd62a7bb49636d80569cf7543d7095bca0cc008779cfcae774b869ab9ec0d'}, '672.png': {'bytes': 177147, 'sha256': 'e12b4e78ba7cf1a689f86e73896bbc2de2927b1a52b770685863f3078d203c74'}, '673.png': {'bytes': 181217, 'sha256': 'ae989e8f1df6af66a2c0ca411173d4f4344a2ef6440419375ed24a30316fc791'}, '68.jpg': {'bytes': 248214, 'sha256': '51db43fb93834780579f08d9645d9f87a11ec0e11d8655c7fc23edae0f670a4f'}, '69%0.jpg': {'bytes': 94139, 'sha256': 'c9ab970256440cf6747d3963443dd5b682ecf6107a6dad0a6dda04fddb44313c'}, '69%1.jpg': {'bytes': 97800, 'sha256': 'df3adaf1c89f4b17bc55580abed652d95d81e7c4325dbc58cc2e3f36c06682f5'}, '7.jpg': {'bytes': 84904, 'sha256': 'df78595d2e22e8873c33004d1f2332aebce37d7f02c620abe1585f6c8a07b3b0'}, '70%0.jpg': {'bytes': 90947, 'sha256': 'f2dfc59f04571d0e3ffaf9cea7cd3779962ad8c02ec644d2a85cd0f20601048b'}, '70%1.jpg': {'bytes': 92409, 'sha256': '2f872a94587470cc3ba78986e6081d75fedb52cfc4bf26488df9f3a3f6909add'}, '704.png': {'bytes': 273886, 'sha256': '6dccb023cda92f58555065be83f089ec9b82e8fb67e2c5d7c8cb9320d46c1f0f'}, '705.png': {'bytes': 259950, 'sha256': '81538a69c02e9ec075e929a524859b702304163ca368100d06dbc1fdce63ab77'}, '706.png': {'bytes': 239263, 'sha256': '0950335978ae5633ed353ce53d69d7292b0694f2b211ecce4be55818cd2cfd2d'}, '709.png': {'bytes': 183515, 'sha256': '14f1b640af1aabc4ebbc7701ec665c0d72fed7c5ad899fc0b1aff7cd472dedd2'}, '71%0.jpg': {'bytes': 88985, 'sha256': '79c92f62e29718eab7abcb983bdfee6dfaa335ac641a8ab9b8c845002c46962e'}, '71%1.jpg': {'bytes': 89656, 'sha256': 'bf62879ff4a56adc82854adb16851fe1074a06e3c5a2c8656281fb688cb186db'}, '72.jpg': {'bytes': 127345, 'sha256': '7b40011c25cbf5c632a0cf0ebea169ee66b7e71f38a70ea42f51cd49efccdc32'}, '73.jpg': {'bytes': 225708, 'sha256': '8df266f43113d18b75dfac84788ee860cb4dae9d6235e2f75c3808d830be4d39'}, '74.jpg': {'bytes': 132009, 'sha256': '36291971df861092989f0f159fda1406a739edd51d8ea59f1b70a9d2ccf9a32e'}, '75%0.jpg': {'bytes': 50533, 'sha256': '9e5916bf782d80c497098a257b41a4290da2882fb318712ac2e0f63dba6ae06a'}, '75%1.jpg': {'bytes': 52006, 'sha256': 'e4132d3fe7b9372f3bdee718ea88f419e649336c2e0ae85bbce4542356eeb4f5'}, '753.png': {'bytes': 192207, 'sha256': 'ce588b43c1e163cfbe5316dfb67854f96c15e7ce206616c6ae6ce5dd022ab349'}, '754.png': {'bytes': 184378, 'sha256': '94dc949fbd0d420ac53cb80b117e3c0b38759e20f0d2bbc19be9f696b5527820'}, '76%0.jpg': {'bytes': 45096, 'sha256': '75a70d26cfccaebb8efc59681e16b5418f3380804b3d85931cd512ce887e8c02'}, '76%1.jpg': {'bytes': 43413, 'sha256': '162ef9681b67572922cf25f077eeac0f90afb1ebc783c506f50e50d817a82699'}, '76%2.jpg': {'bytes': 43424, 'sha256': '608423addf35e66f21bee2d90cf83234054476a01675c75e198086bf4e2e2a8b'}, '77.jpg': {'bytes': 76499, 'sha256': '6e5bc53415c78fd3b80dca6ea04bb21ddb24d0c7922cbf4e4fdef144f97b4c9e'}, '78.jpg': {'bytes': 40040, 'sha256': '26d49365c3ded616d9095ec5fc70324930bc9558487b6d6cd0635678af45988d'}, '781.png': {'bytes': 160859, 'sha256': 'e3e34fc1e846c38f81a03cfef3f3f21de4f444e85b86083f16f5757be6367fee'}, '782.png': {'bytes': 175265, 'sha256': 'ace4ba0d7caf9d5b6510d9c8cec8ba8bd1ac60f8dc65923fe9574b747164d634'}, '79.jpg': {'bytes': 106266, 'sha256': 'db6653fdf10bb4d06ce929dd752fc81d1bdf8f1c541ed58e27143a70028ab84a'}, '8.jpg': {'bytes': 60308, 'sha256': '979b05f05aa24378cca4d0beaa01ae414e37d1edd369e3d79ad006820b2687d3'}, '80%0.jpg': {'bytes': 82492, 'sha256': '6bc4ec6fad7df7c4716a5f894bc70dabf729c533e46018bd8a36f713ff56c127'}, '80%1.jpg': {'bytes': 81321, 'sha256': '0f555442306c4bbbcff5a3b040f6c42056d028d0bc9beb8695078a38cfd8a201'}, '81%0.jpg': {'bytes': 71282, 'sha256': '24cde7846d62f23325ff09fa78f66e3516e19ffeaaca2babe0001eb57b435bf7'}, '81%1.jpg': {'bytes': 75895, 'sha256': 'c8a7cb56ffa602dc8c231437cf7f6ef2ec0bc9fda50b0ea4339ec6f4b5d303c8'}, '81%2.jpg': {'bytes': 69161, 'sha256': '529a754bbf36b3937d953f722677096f75f0dc30d62f9a39b2a0c8c8662c4f4a'}, '81%3.jpg': {'bytes': 70064, 'sha256': '0f42c731964420ea4a1e8ddb89742bf2a2f07391cad5edf9e59f3fb4a47f35df'}, '81.png': {'bytes': 218295, 'sha256': 'e77ec5f045d5598bcf0a94acfd1a758c0b2234d2e3735a9beadc75d6653cd7a1'}, '82%0.jpg': {'bytes': 66136, 'sha256': 'cbda19aa43f3d4f9d47c5978dda15d0ac30f2379eb219ee783673606692c59af'}, '82%1.jpg': {'bytes': 66668, 'sha256': '18695094c7fa13ed6b9f733b2a0e0d7e3731820bd9a2f5f4c9495574a2ed8fdc'}, '82%2.jpg': {'bytes': 67854, 'sha256': 'c3c121c510d4ce75d8ec0b13251257594b58e6d8722b5179a98e4937ee06ac2f'}, '82%3.jpg': {'bytes': 69253, 'sha256': 'e2b6bbac57a3856e3beeaeeb91c9629c6be9abf084c41139f8e722b62b3ac3e3'}, '82.png': {'bytes': 224353, 'sha256': '1fbbd38f6264f7cf1ac6e13a8394479a47d80f93d11cb141a8c0088d86fe85ea'}, '83%0.jpg': {'bytes': 76613, 'sha256': '471dcf40e6d744b91f308b0e70e5d748f5457d6a9fc99cbd676dc0e2765bc6ce'}, '83%1.jpg': {'bytes': 77209, 'sha256': '58c117e0ca907bff0978269d5309c60a733c3003e619df0fd3b350d4e082cddb'}, '83%2.jpg': {'bytes': 73928, 'sha256': 'fc9ded83b1a0a0d5c7e90dc6623b53f0abf8b8e6c152cfd3d12a0779e68d5ea2'}, '83%3.jpg': {'bytes': 64038, 'sha256': 'c8d1709554aa0a4665dab1361606b8a1fb5563ac772d5ff4354f9f5576198a34'}, '84%0.jpg': {'bytes': 55551, 'sha256': '7cda81179354f0116297b41d9243617accfa464ab74b32099d91b937424f402c'}, '84%1.jpg': {'bytes': 50429, 'sha256': 'ac3d4811c79b836cb6502657288e6ef3f983a9df34ea93e85ba7ad165e37fa91'}, '84%2.jpg': {'bytes': 53905, 'sha256': '1e751733e6100f8d905458a1b345e111f7a9c37bacdd8f1503062094d73d7000'}, '84%3.jpg': {'bytes': 23208, 'sha256': 'f9cdfd56fae37d09f75b9b7100ca23786109d3c4409caaf05822a27a67952745'}, '848.png': {'bytes': 232743, 'sha256': '4e42c44e658de6d938939886d57ea5efee56e64730cb87872648d34f4636c423'}, '849.png': {'bytes': 243456, 'sha256': '8e911173d24c2d3625b9e2aca79f61f76bc7a4335a346429c49d1e86bf4a43d4'}, '85%0.jpg': {'bytes': 86391, 'sha256': 'f24150183bd3f85f5f872354c54947aab660a27c2e864bf184929c594879ef8e'}, '85%1.jpg': {'bytes': 95865, 'sha256': 'f6bbf03ef4f4fb5481abe3d8222324866f4364352c701722b8f89667df246ef6'}, '86.jpg': {'bytes': 46930, 'sha256': 'f0e30e75bb446f479dac7865d9d49d85ef45dd05096ae672a72db9f41f7a7942'}, '88.jpg': {'bytes': 140781, 'sha256': '644277999cda9bf6042ad0e8b9ae7ac1c9fc344a032ae9161372d5faa657f16c'}, '887.png': {'bytes': 240623, 'sha256': 'ab74dd760855ff84bb61c5f8b03704690adddbcd257446ac6de06933697e28d9'}, '889.png': {'bytes': 218040, 'sha256': '65feac5c84969e25bcde98317641ffa4ffd7e33b8200c06a7bf020354407b537'}, '89%2.jpg': {'bytes': 80111, 'sha256': '18c52c8679839deb7037bcf0a1b00625aee6e578b66ec8deee13df1760bce1c9'}, '89%3.jpg': {'bytes': 59983, 'sha256': '60a32cebc3ef8b22a4f63e8b2e9d0f6d2bb05d3a7021f2dee990acccaf653170'}, '890.png': {'bytes': 273685, 'sha256': 'e521d41483d5dffc7ba98027fe83052281a1b0875d7567d5e7377823223fb84a'}, '9.jpg': {'bytes': 209887, 'sha256': '3adaff036fe3ce702f0db0f65670c5a0c2cc52ab8f480b93b365fff53b0b6872'}, '90.jpg': {'bytes': 68440, 'sha256': '4b9256426c9a4a765f4736c5edad6e9a1ff292dc50bbceaba4ac5d7140b19a1a'}, '91%0.jpg': {'bytes': 52015, 'sha256': 'e8ffc1c45aeb67e882ab7df918e16a5c36d77212191d7f0f183befda7a1cfb44'}, '91%1.jpg': {'bytes': 42740, 'sha256': 'aa3c5bd1eee46086519ccc5141e2dac83fa7e89293293c063a9ef42326ae6d52'}, '921.png': {'bytes': 171318, 'sha256': 'a36b5138a0b49c31e60542bccc827878cef9c04fdfdbe8ea51af83381669bde7'}, '956.png': {'bytes': 157311, 'sha256': '4ef023a7650d54c0d75fe26cfbea872710b9b3a529ec4df9e0f7398dcbf16364'}, '957.png': {'bytes': 175413, 'sha256': 'c9a1ecff1dbbf1517abd3ef4d5ff6bb6dcfc7014b54cbf72ddb5159eb3d9db39'}, '969.png': {'bytes': 129211, 'sha256': '04ccfbb8705b2bc3bbcd6c4f638716ff09f248193ae34ce8ab9b2a0cd2a992bf'}, '996.png': {'bytes': 166202, 'sha256': 'db897587a54858be3a0af0e3323f4f6b125ef195e0c7411fd9a1e10e09705053'}, '997.png': {'bytes': 167433, 'sha256': '574323337b39325c1c2537243080080f2b7e67b8e5de278341d3516e8cf69120'}, 'bmj.m606.full-p4-22%0.png': {'bytes': 303940, 'sha256': '956d39417a2ad73aa340b5b32711134be04b1d0e5977823c2fbaaa73fe035a48'}, 'bmj.m606.full-p4-22%1.png': {'bytes': 295063, 'sha256': 'ecd9eb1924ac36e0f31ae130c85b8967c0af89fd9a52e2a384f6155effb5933e'}, 'bmj.m606.full-p4-22%2.png': {'bytes': 284274, 'sha256': 'e4c6bcde570b81bf00a87f9d88ef6d3cb1d9db437badb5cc815ce9f33cdfe6ce'}, 'bmj.m606.full-p4-22%3.png': {'bytes': 283586, 'sha256': '83b556968215e63b9114dadeb9107b48796d81ed9924064ab4cbde38142080fc'}, 'bmj.m606.full-p4-22%4.png': {'bytes': 100235, 'sha256': '9191dfe461fc3741e7a6d76058eb9f4551b84cb6dd0006febc19ae97abbc905f'}, 'bmj.m606.full-p4-22%5.png': {'bytes': 96580, 'sha256': '4d244d637438bae9afa104bda45f022c462efbb891bc81db45e0e11a6927eeb2'}, 'bmj.m606.full-p4-22%6.png': {'bytes': 109135, 'sha256': '0c74683ae78fe8849580600ec9aa0eccf5080673878540f388d135ca9bc4d359'}, 'bmj.m606.full-p4-22%7.png': {'bytes': 105217, 'sha256': '0fe91452e0689c908a30faf93dc3d3f12a9fde2b5558e30d80ee2aebbf79a6fb'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%0.png': {'bytes': 40685, 'sha256': 'd84fa4fe915a482f88782ec6adf7cfcc32575f240b45f7d1a26898082bd88912'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%1.png': {'bytes': 32911, 'sha256': '9ada2366af9ea23abbb6651347a59e2e49f542b20d71b1eec7d9a4bc1a3c3329'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%10.png': {'bytes': 42140, 'sha256': 'd415f63ac1715adaeffd759518a229fbb4f9c412236a87792ab81b47f81d43ab'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%11.png': {'bytes': 45554, 'sha256': '7618d235863de7b9b7dec6aec75090a98259bd05257fb39707b7d786cc49d468'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%2.png': {'bytes': 52279, 'sha256': '44326aeaec6f9000136a66b743f535cac1041f80383ce31752e82d090b6febf5'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%3.png': {'bytes': 34753, 'sha256': '37f83b937fcf335074d348cfe1ad0cc02dd538361e5f909778c8731fcd978c62'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%4.png': {'bytes': 41038, 'sha256': '955afe77f9487cbcbd2fec68ed96d4da466e6e6476de7a3855522f7e91e10c36'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%5.png': {'bytes': 41513, 'sha256': '431f96f1404da779f6502bc20901c93af3c67869a3e4f9dfc8ec2ee0ca340fc0'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%6.png': {'bytes': 36236, 'sha256': '3077682ac6059b4dd47e69cacda6bd32a6d6b999a1606d7946ce10896453016a'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%7.png': {'bytes': 36684, 'sha256': '2105da7d677827e524a5975bfdd1abea90716301bb9e2cf4d8ba33f6b032e7b6'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%8.png': {'bytes': 37307, 'sha256': '3eea0cde3d805059c17c46c232bb7c2738a1841f54bce587902b18f3a0eda36c'}, 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%9.png': {'bytes': 32560, 'sha256': '338a8eb784d53ccb3b85005b44d585fe048dda163ecaefdc71fb2570bf84bc62'}, 'ehaa254-p0-63%2.png': {'bytes': 85298, 'sha256': '9e1403c984eea5ada275c2e16425e6f6a7457dfc0809b7a4b36c9ebab0d91cc0'}, 'ehaa254-p0-63%3.png': {'bytes': 79277, 'sha256': '5692882334115ec95058cecb9ca6c9053b282cbaab1638a7c0f7889d02a29bee'}, 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%1.png': {'bytes': 34906, 'sha256': '1b766fa370e62ea29e6bd1a4e84cec10675e5acd72e8502be39ff533ffa3e67c'}, 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%3.png': {'bytes': 46590, 'sha256': '337eb176610e340ff3e31c3ca2fa9a8966fb7db1541d0511fd1395afdf481feb'}, 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%5.png': {'bytes': 40621, 'sha256': '4b64b3c119ffda60753d9d3f5bf09f844daacaa4250963050ed4f1133cc0ba8c'}, 'kjr-21-e24-p2-8.png': {'bytes': 176271, 'sha256': '1a291084f2e2c7e8e6d068ac23e0a6b86eca27201149e266cf283420be442a52'}, 'kjr-21-e24-p3-16.png': {'bytes': 281801, 'sha256': '1420ee3ca200e6745eb4a3c32f5be0c97ba000a27e4df416f827eab5ffabc46c'}, 'kjr-21-e24-p5-29.png': {'bytes': 219632, 'sha256': '0a970b488c7bbee3dbdc128fa0588b7cd0157fb763975d896fce6044c63e8bb9'}, 'kjr-21-e24-p5-31.png': {'bytes': 220566, 'sha256': '23b947285151fca457d31c2dfd6cbd45f17e8c1830243d01a2a079a4458eb1b8'}, 'kjr-21-e25-p1-10.png': {'bytes': 97186, 'sha256': '403d7080ae29c7ae0d3b69a6617c1b5505fdcb8a5ed7bb45f61d92fd0cef556a'}, 'kjr-21-e25-p1-12.png': {'bytes': 95232, 'sha256': '2aa679bbccf6784d27b151abdb6abec6ed5d8cc8dbfe5c07cf3e047196af2f86'}, 'kjr-21-e25-p1-14.png': {'bytes': 150822, 'sha256': '2d7411c8da8834f31a96b6794e0b43403d968619816f3bc640e3629246d26a7a'}, 'kjr-21-e25-p1-6.png': {'bytes': 82467, 'sha256': 'a0c006c35939a125d1cfc67bcc3104558426fa647068e30177b7c2deff023125'}, 'kjr-21-e25-p1-8.png': {'bytes': 104559, 'sha256': 'ee33284462dd70a378828c6d476e2aa229ee53faaba2a8478286bb70baad2b7d'}, 'med-9780198814924-chapter-17-p10-515.png': {'bytes': 104194, 'sha256': '64dcd0431c3def541c4fc05f72bf0016c26f2dbf4888ca92df191e335c5d00e8'}, 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%0.png': {'bytes': 170188, 'sha256': 'd07aa6bd516e665ca98dea878a6e744058daa034593d0be29d288ba62b68c654'}, 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%1.png': {'bytes': 200081, 'sha256': '7316ad63b115a76ba36993e4c923557fd6011186696651e34702a423f8872045'}, 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%0.png': {'bytes': 142128, 'sha256': '60380bd570237d8c69aadd7932176691ea5a5b60b79c334f6f9d9baa4acf6c85'}, 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%1.png': {'bytes': 74010, 'sha256': 'df458fbe58897ff2df9f4b039fbd95499bce7672ec13abd1e4b3e81c6d5a65b8'}, 'PIIS0140673620301549_0%0.png': {'bytes': 317804, 'sha256': '2b1ae92d3c69d9a4429113b46fd55c736c5d9af8e378620cde3362a6a52793b3'}, 'PIIS0140673620301549_0%1.png': {'bytes': 259565, 'sha256': 'aed1a076558e91f30df522d092fc117674c2855f0fa39e5a28618e62703bebbe'}, 'PIIS0140673620301549_1.png': {'bytes': 175247, 'sha256': '388faf74f32da8d2eadbc5ae886f8999a29673fdbee90b6db3f37426a9d8188b'}, 'PIIS0140673620301549_2.png': {'bytes': 237709, 'sha256': 'f0e7b75de0f28cf6fdaedd3758871035e0cbdd882f50d786966343f1bc8955a9'}, 'PIIS0140673620302117_1.png': {'bytes': 514710, 'sha256': 'bea0ed3ad82b0f7d1ec5c1ede229b44191f5c4137daf20c973a8dad2ab4a2ba4'}, 'PIIS0140673620302117_2.png': {'bytes': 404205, 'sha256': 'd2c63e43d7b2d3f67e889a376dadce4ac90ac5a84324db57a62c6dbed83fef37'}, 'PIIS0140673620303603%0.png': {'bytes': 171790, 'sha256': '043aa89b5fdc7fe4232049d031a224a452fc7b480980ebcd808ddef32a5b2616'}, 'PIIS0140673620303603%1.png': {'bytes': 198474, 'sha256': '32c728c80cb2516ed3ce920599c7c721be4dc88980bc982a533e27a4ad015584'}, 'PIIS0140673620303603%2.png': {'bytes': 64169, 'sha256': '3086190b5cfec05025b0114d3dde2635d5ba4505c7894e17c650cd953cd2869d'}, 'PIIS0140673620303603%3.png': {'bytes': 79792, 'sha256': '3c8a133a51f9142582222409b3b2f00f10847066387b6085b6541cc7388ff92c'}, 'PIIS0140673620303603%4.png': {'bytes': 96206, 'sha256': '049c794baa7be78dd6d205e45055d10f32e33812b930e37a41a5ecc0a05fcf6f'}, 'PIIS0140673620303603%5.png': {'bytes': 66571, 'sha256': '68a8a202e2559d2c1f4b3455672170dae3b01692542714193f29a08c3e01475f'}, 'PIIS0140673620303603%6.png': {'bytes': 77782, 'sha256': '9950a5e526c63b5b0317a1bc7084d19d450427ebfb8a58d9769e0f1db6b277d8'}, 'PIIS0140673620303603%7.png': {'bytes': 87308, 'sha256': '60b8e7bb29f3e16095e06360c93d3f790ae23521e680dbefc529bfbc3c879b3d'}, 'PIIS0140673620303603%8.png': {'bytes': 194841, 'sha256': '51f2df557842ab513460f408c4a7b737ca51c6cfe1f81cab6d2d3d430fdbca25'}, 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%0.png': {'bytes': 446293, 'sha256': 'db68a42ccf861cbca939dd1db90446baffb0699eaf737b8f1cdcf9d1d8f9a7d2'}, 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%1.png': {'bytes': 486699, 'sha256': 'c0d4271ad3506f9fba51fc79c0f52ed76498a658ac449d2a3996214b3f267e06'}, 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%0.png': {'bytes': 226817, 'sha256': '0d507947430ec1e25c4a52fc2ee7939480b1d5b4fca64ac44b65d58349c41db9'}, 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%1.png': {'bytes': 228052, 'sha256': 'c511534ebd40e678004a55fccc5f32087f3026a5c20597c10d79b17f09b9a9df'}, 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%2.png': {'bytes': 196712, 'sha256': '84d9976a4601c002e860f344c94419e9ece90ba5069d7657a5d3f9ef76d454c0'}, 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%3.png': {'bytes': 218644, 'sha256': '50db488a7ae5e68e1b044bc847878bec3fff33ac173097dd04618cf51ee35173'}}, 'physical_image_count': 746, 'splits': {'validated': ['0.jpg', '1%0.jpg', '1%1.jpg', '1%2.jpg', '10%0.jpg', '10%1.jpg', '10%2.jpg', '102.png', '1029.png', '103.png', '1030.png', '104.png', '1046.png', '1047.png', '1048.png', '1049.png', '1050.png', '1058.png', '1059.png', '1060.png', '1061.png', '1062.png', '1065.png', '1070.png', '1071.png', '1072.png', '1078.png', '1079.png', '1093.png', '1094.png', '1098.png', '1099.png', '11%0.jpg', '11%1.jpg', '11%2.jpg', '1100.png', '1101.png', '1115.png', '112.png', '1123.png', '1124.png', '114.png', '115.png', '116.png', '1164.png', '117.png', '118.png', '12.jpg', '120.png', '1210.png', '1211.png', '1212.png', '1213.png', '1214.png', '1215.png', '1216.png', '1217.png', '1218.png', '1221.png', '1238.png', '1239.png', '1240.png', '1261.png', '1262.png', '1263.png', '1264.png', '1277.png', '1297.png', '1298.png', '13.jpg', '1305.png', '1306.png', '1307.png', '1308.png', '1309.png', '1310.png', '1312.png', '1313.png', '1328.png', '1330.png', '1331.png', '1332.png', '1333.png', '1335.png', '1336.png', '14%0.jpg', '14%1.jpg', '14%2.jpg', '14%3.jpg', '14.png', '1428.png', '1463.png', '1497.png', '1498.png', '1499.png', '15%0.jpg', '15%1.jpg', '15%2.jpg', '15%3.jpg', '15.png', '1500.png', '1501.png', '1561.png', '1562.png', '1575.png', '1576.png', '1577.png', '158.png', '16.jpg', '1638.png', '166.png', '167.png', '17%0.jpg', '17%1.jpg', '1702.png', '171.png', '174.png', '176.png', '177.png', '18%0.jpg', '18%1.jpg', '18%2.jpg', '1814.png', '1845.png', '1846.png', '1857.png', '1858.png', '1859.png', '1868.png', '1876.png', '1877.png', '1888.png', '1889.png', '19%0.jpg', '19%1.jpg', '1919.png', '1920.png', '1921.png', '1922.png', '1923.png', '1924.png', '1952.png', '2%0.jpg', '2%1.jpg', '2%2.jpg', '2%3.jpg', '2007.png', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%0.png', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%1.png', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-91.png', '2020.01.24.919183-p27-132.png', '2020.01.24.919183-p27-133.png', '2020.01.24.919183-p27-135.png', '2020.02.10.20021584-p6-52%0.png', '2020.02.10.20021584-p6-52%1.png', '2020.02.10.20021584-p6-52%10.png', '2020.02.10.20021584-p6-52%11.png', '2020.02.10.20021584-p6-52%12.png', '2020.02.10.20021584-p6-52%13.png', '2020.02.10.20021584-p6-52%14.png', '2020.02.10.20021584-p6-52%15.png', '2020.02.10.20021584-p6-52%2.png', '2020.02.10.20021584-p6-52%3.png', '2020.02.10.20021584-p6-52%4.png', '2020.02.10.20021584-p6-52%5.png', '2020.02.10.20021584-p6-52%6.png', '2020.02.10.20021584-p6-52%7.png', '2020.02.10.20021584-p6-52%8.png', '2020.02.10.20021584-p6-52%9.png', '2020.02.11.20021493-p16-109%0.png', '2020.02.11.20021493-p16-109%1.png', '2020.02.11.20021493-p16-109%2.png', '2020.02.11.20022053-p12-67%0.png', '2020.02.11.20022053-p12-67%1.png', '2020.02.11.20022053-p12-67%2.png', '2020.02.11.20022053-p12-67%3.png', '2020.02.13.20022673-p13-77%0.png', '2020.02.13.20022673-p13-77%1.png', '2020.02.17.20024018-p17-61%0.png', '2020.02.17.20024018-p17-61%2.png', '2020.02.17.20024018-p17-61%4.png', '2020.02.19.20025023-p20-93%0.png', '2020.02.19.20025023-p20-93%1.png', '2020.02.19.20025023-p20-93%2.png', '2020.02.19.20025023-p20-93%3.png', '2020.02.22.20024927-p18-66%0.png', '2020.02.22.20024927-p18-66%1.png', '2020.02.22.20024927-p18-66%2.png', '2020.02.22.20024927-p19-68%3.png', '2020.02.22.20024927-p19-68%4.png', '2020.02.22.20024927-p19-68%5.png', '2020.02.22.20024927-p20-70%0.png', '2020.02.22.20024927-p20-70%1.png', '2020.02.22.20024927-p20-70%2.png', '2020.02.22.20024927-p20-70%3.png', '2020.02.23.20026856-p17-115%0.png', '2020.02.23.20026856-p17-115%1.png', '2020.02.23.20026856-p17-115%2.png', '2020.02.23.20026856-p17-115%3.png', '2020.02.23.20026856-p17-115%4.png', '2020.02.23.20026856-p17-115%5.png', '2020.02.24.20027052-p8-73%0.png', '2020.02.24.20027052-p8-73%1.png', '2020.02.24.20027052-p8-73%2.png', '2020.02.24.20027052-p8-73%3.png', '2020.02.24.20027201-p19-669.png', '2020.02.25.20021568-p23-108%0.png', '2020.02.25.20021568-p23-108%1.png', '2020.02.25.20021568-p23-108%10.png', '2020.02.25.20021568-p23-108%11.png', '2020.02.25.20021568-p23-108%12.png', '2020.02.25.20021568-p23-108%13.png', '2020.02.25.20021568-p23-108%14.png', '2020.02.25.20021568-p23-108%15.png', '2020.02.25.20021568-p23-108%2.png', '2020.02.25.20021568-p23-108%3.png', '2020.02.25.20021568-p23-108%4.png', '2020.02.25.20021568-p23-108%5.png', '2020.02.25.20021568-p23-108%6.png', '2020.02.25.20021568-p23-108%7.png', '2020.02.25.20021568-p23-108%8.png', '2020.02.25.20021568-p23-108%9.png', '2020.02.25.20021568-p24-111%0.png', '2020.02.25.20021568-p24-111%1.png', '2020.02.25.20021568-p24-111%2.png', '2020.02.25.20021568-p24-111%3.png', '2020.02.25.20021568-p24-111%4.png', '2020.02.25.20021568-p24-111%5.png', '2020.02.25.20021568-p24-111%6.png', '2020.02.25.20021568-p24-111%7.png', '2020.02.25.20021568-p24-111%8.png', '2020.02.25.20021568-p24-111%9.png', '2020.02.25.20027763-p15-53%0.png', '2020.02.25.20027763-p15-53%1.png', '2020.02.26.20026989-p34-114_1%1.png', '2020.02.26.20026989-p34-114_2%1.png', '2020.02.26.20027938-p6-80_1%0.png', '2020.02.26.20027938-p6-80_1%1.png', '2020.02.26.20027938-p6-80_2%0.png', '2020.02.26.20027938-p6-80_2%1.png', '2020.02.26.20027938-p6-80_3%0.png', '2020.02.26.20027938-p6-80_3%1.png', '2020.02.27.20027557-p25-137%0.png', '2020.02.27.20027557-p25-137%1.png', '2020.02.27.20027557-p25-137%2.png', '2020.02.27.20027557-p25-137%3.png', '2020.02.27.20027557-p25-137%4.png', '2020.02.28.20029025-p23-182.png', '2020.02.28.20029025-p23-183.png', '2020.03.01.20029769-p21-73_1%0.png', '2020.03.01.20029769-p21-73_1%1.png', '2020.03.01.20029769-p21-73_1%2.png', '2020.03.01.20029769-p21-73_2%0.png', '2020.03.01.20029769-p21-73_2%1.png', '2020.03.01.20029769-p21-73_2%2.png', '2020.03.01.20029769-p21-73_3.png', '2020.03.01.20029769-p21-73_4.png', '2020.03.03.20030353-p17-106%0.png', '2020.03.03.20030353-p17-106%1.png', '2020.03.03.20030353-p17-106%2.png', '2020.03.03.20030775-p10-88%0.png', '2020.03.03.20030775-p10-88%1.png', '2020.03.03.20030775-p11-91.png', '2020.03.03.20030775-p12-93%1.png', '2020.03.04.20026005-p12-1946%0.png', '2020.03.04.20026005-p12-1946%1.png', '2020.03.04.20026005-p12-1946%2.png', '2020.03.04.20026005-p12-1946%3.png', '2020.03.04.20026005-p12-1946%4.png', '2020.03.04.20026005-p12-1946%5.png', '2020.03.04.20030395-p27-108%0.png', '2020.03.04.20030395-p27-108%1.png', '2020.03.04.20030395-p27-108%2.png', '2020.03.04.20030395-p27-108%3.png', '2020.03.04.20030395-p27-108%4.png', '2020.03.04.20030395-p27-108%5.png', '2020.03.04.20030395-p27-108%6.png', '2020.03.04.20030395-p27-108%7.png', '2020.03.04.20030395-p27-108%8.png', '2020.03.04.20031039-p23-97_1%0.png', '2020.03.04.20031039-p23-97_1%1.png', '2020.03.04.20031039-p23-97_1%2.png', '2020.03.04.20031039-p23-97_2%0.png', '2020.03.04.20031039-p23-97_2%1.png', '2020.03.04.20031039-p23-97_2%2.png', '2020.03.04.20031047-p12-81%0.png', '2020.03.04.20031047-p12-81%1.png', '2020.03.04.20031047-p12-81%2.png', '2020.03.04.20031047-p12-81%3.png', '2020.03.04.20031047-p13-84%0.png', '2020.03.04.20031047-p13-84%1.png', '2020.03.04.20031047-p13-84%2.png', '2020.03.04.20031047-p13-84%3.png', '2020.03.04.20031047-p13-84%4.png', '2020.03.04.20031047-p13-84%5.png', '2020.03.09.20033118-p20-93%0.png', '2020.03.09.20033118-p20-93%1.png', '2020.03.09.20033118-p20-93%2.png', '2020.03.09.20033118-p20-93%3.png', '2020.03.10.20032136-p34-118_1%0.png', '2020.03.10.20032136-p34-118_1%1.png', '2020.03.10.20032136-p34-118_2%0.png', '2020.03.10.20032136-p34-118_2%1.png', '2020.03.11.20033159-p12-45%0.png', '2020.03.11.20033159-p12-45%1.png', '2020.03.11.20033159-p12-45%2.png', '2020.03.11.20033159-p12-45%3.png', '2020.03.11.20033159-p12-45%4.png', '2020.03.11.20033159-p12-45%5.png', '2020.03.12.20027185-p8-216.png', '2020.03.12.20027185-p8-217.png', '2020.03.12.20027185-p8-218.png', '2020.03.12.20027185-p8-221.png', '2020.03.12.20034686-p17-91-1.png', '2020.03.12.20034686-p17-91-2.png', '2020.03.12.20034686-p17-91-3.png', '2020.03.12.20034686-p17-91-4.png', '2020.03.12.20034686-p17-91-5.png', '2020.03.12.20034686-p17-91-6.png', '2020.03.13.20035212-p23-152.png', '2020.03.13.20035212-p23-153.png', '2020.03.13.20035212-p23-154.png', '2020.03.13.20035212-p23-155.png', '2020.03.13.20035212-p23-156.png', '2020.03.13.20035212-p23-157.png', '2020.03.13.20035212-p23-158.png', '2020.03.13.20035212-p23-159.png', '2020.03.13.20035212-p23-160.png', '2020.03.13.20035212-p23-161.png', '2020.03.13.20035212-p23-162.png', '2020.03.13.20035212-p23-163.png', '2020.03.16.20035105-p6-64-1.png', '2020.03.16.20035105-p6-64-2.png', '2020.03.16.20035105-p6-64-3.png', '2020.03.16.20035105-p6-64-4.png', '2020.03.16.20036145-p19-128-1.png', '2020.03.16.20036145-p19-128-2.png', '2020.03.16.20036145-p19-128-3.png', '2020.03.16.20036145-p19-128-4.png', '2020.03.18.20038125-p15-54-2.png', '2020.03.18.20038125-p15-54-4.png', '2020.03.18.20038125-p15-54-5.png', '2020.03.18.20038125-p15-54-6.png', '2020.03.18.20038125-p16-56-1.png', '2020.03.18.20038125-p16-56-2.png', '2020.03.18.20038125-p16-56-3.png', '2020.03.18.20038125-p16-56-4.png', '2020.03.18.20038125-p16-56-5.png', '2020.03.18.20038125-p16-56-6.png', '2020.03.19.20038539-p10-57.png', '2020.03.19.20038539-p10-58.png', '2020.03.19.20038539-p10-59.png', '2020.03.19.20038539-p10-60.png', '2020.03.19.20038539-p10-61.png', '2020.03.19.20038539-p10-62.png', '2020.03.20.20037325-p23-121.png', '2020.03.20.20037325-p23-122.png', '2020.03.20.20037325-p23-123.png', '2020.03.20.20037325-p23-124.png', '2020.03.21.20040691-p18-6-6.png', '2020.03.21.20040691-p18-67-1.png', '2020.03.21.20040691-p18-67-2.png', '2020.03.21.20040691-p18-67-3.png', '2020.03.21.20040691-p18-67-4.png', '2020.03.21.20040691-p18-67-5.png', '2020.03.21.20040691-p18-67-7.png', '2020.03.21.20040691-p18-67-8.png', '2020.03.22.20034041-p16-66.png', '2020.03.22.20034041-p17-80-1.png', '2020.03.22.20034041-p17-80-2.png', '2020.03.22.20034041-p18-92-1.png', '2020.03.22.20034041-p18-92-2.png', '2020.03.22.20034041-p18-92-3.png', '2020.03.22.20040782-p24-1533-1.png', '2020.03.22.20040782-p24-1533-2.png', '2020.03.22.20040782-p24-1533-3.png', '2020.03.22.20040782-p24-1533-4.png', '2020.03.22.20040782-p24-1533-5.png', '2020.03.22.20040782-p24-1533-6.png', '2020.03.22.20040782-p24-1533-7.png', '2020.03.22.20040782-p24-1533-8.png', '2020.03.22.20040782-p25-1541.png', '2020.03.22.20040782-p25-1542.png', '2020.03.22.20040782-p25-1543.png', '2020.03.22.20040782-p25-1544.png', '2020.03.22.20040782-p25-1545.png', '2020.03.22.20040782-p25-1546.png', '2020.03.25.20037721-p27-165.png', '2020.03.25.20037721-p27-172.png', '2020.03.25.20037721-p27-173.png', '2020.03.25.20037721-p27-174.png', '2020.03.25.20037721-p27-181.png', '2020.03.25.20037721-p27-182.png', '2020.03.25.20043166-p17-78.png', '2020.03.25.20043166-p17-79.png', '2020.03.25.20043166-p17-80.png', '2020.03.25.20043166-p17-81.png', '2020.03.26.20041426-p11-119.png', '2020.03.26.20041426-p11-120.png', '2020.03.26.20041426-p11-121.png', '2020.03.26.20041426-p11-122.png', '2020.03.26.20041426-p11-123.png', '2020.03.26.20041426-p11-124.png', '2020.03.26.20041426-p11-125.png', '2020.03.26.20041426-p11-126.png', '2020.03.26.20041426-p11-127.png', '2020.03.26.20041426-p11-128.png', '2020.03.26.20041426-p11-130.png', '2020.03.26.20041426-p11-131.png', '2020.03.26.20041426-p11-132.png', '2020.03.26.20041426-p11-133.png', '2020.03.30.20047985-p9-56%0.png', '2020.03.30.20047985-p9-56%1.png', '2020.03.30.20047985-p9-56%2.png', '2020.03.30.20047985-p9-56%3.png', '2039.png', '2040.png', '2041.png', '21%0.jpg', '21%1.jpg', '2108.png', '2138.png', '2139.png', '2140.png', '2141.png', '2142.png', '2143.png', '2144.png', '2145.png', '22%0.jpg', '22%1.jpg', '2237.png', '226.png', '227.png', '23.png', '2341.png', '24.png', '25%1.jpg', '25%2.jpg', '25%3.jpg', '25.png', '26%0.jpg', '26%1.jpg', '26%3.jpg', '26.png', '27.jpg', '28.jpg', '29%0.jpg', '29%1.jpg', '29%2.jpg', '29%3.jpg', '294.png', '3.jpg', '30%0.jpg', '30%1.jpg', '30%2.jpg', '30%3.jpg', '30%4.jpg', '30%5.jpg', '31.jpg', '32%0.jpg', '32%1.jpg', '33%0.jpg', '33%1.jpg', '33.png', '34.jpg', '35%0.jpg', '35%1.jpg', '35%2.jpg', '354.png', '36%0.jpg', '36%1.jpg', '361.png', '37%0.jpg', '37%1.jpg', '37%2.jpg', '37%3.jpg', '37%4.jpg', '378.png', '38%0.jpg', '38%1.jpg', '38%2.jpg', '381.png', '382.png', '383.png', '385.png', '39%0.jpg', '39%1.jpg', '39%2.jpg', '39%3.jpg', '39%4.jpg', '39%5.jpg', '39%6.jpg', '39%7.jpg', '39%8.jpg', '4%1.jpg', '4%3.jpg', '4%5.jpg', '4%7.jpg', '40%0.jpg', '40%1.jpg', '41.jpg', '412.png', '413.png', '414.png', '42.jpg', '43%1.jpg', '43%2.jpg', '43.png', '44%0.jpg', '44%1.jpg', '44.png', '45.jpg', '452.png', '46.jpg', '47.jpg', '48.jpg', '486.png', '487.png', '49%0.jpg', '49%1.jpg', '5%0.jpg', '5%1.jpg', '5%2.jpg', '5%3.jpg', '5%4.jpg', '5%5.jpg', '5%6.jpg', '50.jpg', '51%0.jpg', '51%1.jpg', '51%2.jpg', '51%3.jpg', '52%0.jpg', '52%1.jpg', '53.jpg', '54%0.jpg', '54%1.jpg', '54%2.jpg', '54%3.jpg', '56%0.jpg', '56%1.jpg', '57%0.jpg', '575.png', '576.png', '577.png', '578.png', '579.png', '580.png', '583.png', '584.png', '585.png', '586.png', '587.png', '59%0.jpg', '59.png', '590.png', '591.png', '592.png', '6%0.jpg', '6%1.jpg', '6%3.jpg', '60.jpg', '62.jpg', '63%0.jpg', '63%1.jpg', '63%2.jpg', '63%3.jpg', '63%4.jpg', '63%5.jpg', '63%6.jpg', '63%7.jpg', '64.jpg', '65%0.jpg', '65%1.jpg', '65%2.jpg', '65%3.jpg', '65%4.jpg', '658.png', '66.jpg', '660.png', '661.png', '662.png', '663.png', '67%0.jpg', '67%1.jpg', '67%2.jpg', '672.png', '673.png', '68.jpg', '69%0.jpg', '69%1.jpg', '7.jpg', '70%0.jpg', '70%1.jpg', '704.png', '705.png', '706.png', '709.png', '71%0.jpg', '71%1.jpg', '72.jpg', '73.jpg', '74.jpg', '75%0.jpg', '75%1.jpg', '753.png', '754.png', '76%0.jpg', '76%1.jpg', '76%2.jpg', '77.jpg', '78.jpg', '781.png', '782.png', '79.jpg', '8.jpg', '80%0.jpg', '80%1.jpg', '81%0.jpg', '81%1.jpg', '81%2.jpg', '81%3.jpg', '81.png', '82%0.jpg', '82%1.jpg', '82%2.jpg', '82%3.jpg', '82.png', '83%0.jpg', '83%1.jpg', '84%0.jpg', '84%1.jpg', '84%2.jpg', '84%3.jpg', '848.png', '849.png', '85%0.jpg', '85%1.jpg', '86.jpg', '88.jpg', '887.png', '889.png', '89%2.jpg', '890.png', '9.jpg', '90.jpg', '91%0.jpg', '91%1.jpg', '921.png', '956.png', '957.png', '969.png', '996.png', '997.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%0.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%1.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%10.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%11.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%2.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%3.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%4.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%5.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%6.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%7.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%8.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%9.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%0.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%1.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%0.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%1.png', 'PIIS0140673620301549_0%0.png', 'PIIS0140673620301549_0%1.png', 'PIIS0140673620301549_1.png', 'PIIS0140673620301549_2.png', 'PIIS0140673620302117_1.png', 'PIIS0140673620302117_2.png', 'PIIS0140673620303603%0.png', 'PIIS0140673620303603%1.png', 'PIIS0140673620303603%2.png', 'PIIS0140673620303603%3.png', 'PIIS0140673620303603%4.png', 'PIIS0140673620303603%5.png', 'PIIS0140673620303603%6.png', 'PIIS0140673620303603%7.png', 'PIIS0140673620303603%8.png', 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%0.png', 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%1.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%0.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%1.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%2.png', 'bmj.m606.full-p4-22%0.png', 'bmj.m606.full-p4-22%1.png', 'bmj.m606.full-p4-22%2.png', 'bmj.m606.full-p4-22%3.png', 'bmj.m606.full-p4-22%4.png', 'bmj.m606.full-p4-22%5.png', 'bmj.m606.full-p4-22%6.png', 'bmj.m606.full-p4-22%7.png', 'ehaa254-p0-63%2.png', 'ehaa254-p0-63%3.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%1.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%3.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%5.png', 'kjr-21-e24-p2-8.png', 'kjr-21-e24-p3-16.png', 'kjr-21-e24-p5-29.png', 'kjr-21-e24-p5-31.png', 'kjr-21-e25-p1-10.png', 'kjr-21-e25-p1-12.png', 'kjr-21-e25-p1-14.png', 'kjr-21-e25-p1-6.png', 'kjr-21-e25-p1-8.png', 'med-9780198814924-chapter-17-p10-515.png'], 'train': ['PIIS0140673620301549_0%0.png', '25%2.jpg', '2020.03.22.20034041-p18-92-2.png', 'kjr-21-e25-p1-12.png', '63%1.jpg', '385.png', '2020.03.13.20035212-p23-154.png', '2020.02.17.20024018-p17-61%4.png', '26%3.jpg', '1463.png', '2020.03.22.20034041-p16-66.png', '2020.02.11.20022053-p12-67%1.png', '1888.png', 'PIIS0140673620303603%5.png', 'PIIS0140673620303603%1.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%5.png', '2020.03.26.20041426-p11-124.png', '2020.02.10.20021584-p6-52%9.png', 'bmj.m606.full-p4-22%4.png', '2020.02.24.20027052-p8-73%1.png', '1094.png', '2%0.jpg', '2020.03.19.20038539-p10-61.png', '2020.01.24.919183-p27-135.png', '709.png', '2020.03.22.20040782-p25-1543.png', '2020.03.03.20030775-p10-88%0.png', '2020.02.13.20022673-p13-77%0.png', '2020.03.10.20032136-p34-118_2%1.png', 'PIIS0140673620303603%8.png', '2020.03.16.20036145-p19-128-4.png', 'med-9780198814924-chapter-17-p10-515.png', '2020.03.04.20030395-p27-108%8.png', '1313.png', '1868.png', '1306.png', '63%7.jpg', '2020.03.09.20033118-p20-93%3.png', '2020.03.26.20041426-p11-126.png', '672.png', '2140.png', '1098.png', '2020.01.24.919183-p27-133.png', '1216.png', '2020.03.04.20030395-p27-108%1.png', '1240.png', '2020.03.01.20029769-p21-73_4.png', '33%0.jpg', '64.jpg', '1212.png', '2020.02.10.20021584-p6-52%1.png', '83%0.jpg', '2020.03.25.20043166-p17-80.png', '51%1.jpg', '171.png', '30%2.jpg', '2020.03.04.20031039-p23-97_1%2.png', '849.png', '2020.02.11.20022053-p12-67%0.png', '2020.03.26.20041426-p11-132.png', '660.png', '14%1.jpg', '2020.02.26.20027938-p6-80_3%1.png', '781.png', '30%3.jpg', '1101.png', '2020.03.22.20040782-p24-1533-1.png', '2020.02.25.20021568-p24-111%4.png', '63%5.jpg', '361.png', '48.jpg', '2020.03.16.20035105-p6-64-4.png', '578.png', '2020.03.25.20037721-p27-165.png', '33.png', '1335.png', '158.png', '2143.png', 'PIIS0140673620303603%6.png', '25%3.jpg', '584.png', '75%1.jpg', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%0.png', '15%3.jpg', '10%2.jpg', '2020.02.25.20021568-p24-111%8.png', '15%1.jpg', '120.png', '35%2.jpg', '2020.03.21.20040691-p18-67-4.png', '2020.03.04.20031039-p23-97_1%1.png', '32%0.jpg', '2020.02.25.20021568-p23-108%7.png', '2020.02.10.20021584-p6-52%5.png', '2020.03.12.20027185-p8-217.png', '2020.02.28.20029025-p23-183.png', '91%0.jpg', '2020.02.25.20021568-p24-111%2.png', '1048.png', '782.png', '18%1.jpg', '26.png', '2020.02.27.20027557-p25-137%3.png', '2020.02.26.20027938-p6-80_1%1.png', '174.png', '2020.02.27.20027557-p25-137%2.png', '65%2.jpg', '1239.png', '2020.03.26.20041426-p11-120.png', '41.jpg', '2020.02.17.20024018-p17-61%2.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%0.png', '1924.png', '1500.png', '2020.02.11.20021493-p16-109%2.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%3.png', '2020.02.23.20026856-p17-115%4.png', '26%0.jpg', '2020.03.25.20043166-p17-78.png', '88.jpg', '486.png', '1093.png', '890.png', '63%3.jpg', '1061.png', '2020.03.26.20041426-p11-131.png', '2020.02.25.20021568-p23-108%6.png', '592.png', '2020.03.16.20036145-p19-128-3.png', '84%2.jpg', '658.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%7.png', '2020.03.11.20033159-p12-45%4.png', '2020.03.22.20040782-p25-1541.png', '59%0.jpg', '1846.png', '39%3.jpg', '1238.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%10.png', '2020.03.12.20034686-p17-91-3.png', '2020.03.19.20038539-p10-60.png', '83%1.jpg', '1164.png', '1919.png', 'bmj.m606.full-p4-22%3.png', '1124.png', '1263.png', '37%4.jpg', '2020.02.26.20027938-p6-80_1%0.png', '2020.03.20.20037325-p23-123.png', '1889.png', '2020.02.25.20021568-p24-111%9.png', '1922.png', '2020.03.16.20035105-p6-64-1.png', '118.png', '32%1.jpg', '22%0.jpg', '2020.03.30.20047985-p9-56%3.png', '2020.02.22.20024927-p20-70%2.png', '2020.03.04.20030395-p27-108%6.png', '2020.02.10.20021584-p6-52%14.png', '753.png', '2020.03.26.20041426-p11-122.png', '2020.02.25.20021568-p24-111%0.png', '2139.png', '62.jpg', '18%0.jpg', '1%1.jpg', '54%0.jpg', '117.png', '1100.png', '2020.03.01.20029769-p21-73_2%1.png', '2020.03.10.20032136-p34-118_1%0.png', '2020.02.25.20021568-p24-111%3.png', '2020.03.18.20038125-p16-56-3.png', '30%1.jpg', '35%1.jpg', '2020.02.25.20021568-p23-108%13.png', '166.png', '2020.02.25.20027763-p15-53%0.png', '103.png', '2020.02.24.20027052-p8-73%3.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%6.png', 'PIIS0140673620302117_2.png', '18%2.jpg', '67%1.jpg', '2020.02.11.20022053-p12-67%2.png', '2020.03.13.20035212-p23-161.png', '2020.03.25.20043166-p17-79.png', '2020.03.30.20047985-p9-56%2.png', '2020.02.19.20025023-p20-93%3.png', '2020.03.12.20027185-p8-216.png', '2141.png', '2144.png', '2%2.jpg', '15%2.jpg', '31.jpg', '56%1.jpg', '1214.png', 'kjr-21-e24-p5-31.png', '30%4.jpg', '452.png', '112.png', '2020.02.22.20024927-p20-70%1.png', '2020.01.24.919183-p27-132.png', '1277.png', '1331.png', '45.jpg', '2020.03.04.20031047-p13-84%0.png', '1501.png', 'bmj.m606.full-p4-22%1.png', '2020.03.12.20027185-p8-221.png', '661.png', '2041.png', '2020.02.25.20021568-p23-108%10.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%9.png', '0.jpg', '44%0.jpg', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%2.png', '2020.03.22.20040782-p24-1533-6.png', '2%3.jpg', '82%0.jpg', '2020.02.25.20021568-p23-108%5.png', '2020.03.13.20035212-p23-152.png', '1211.png', '2020.02.10.20021584-p6-52%11.png', '1428.png', '585.png', '590.png', '2020.02.22.20024927-p18-66%0.png', '997.png', '1307.png', '16.jpg', '23.png', '2020.02.27.20027557-p25-137%1.png', '68.jpg', '1298.png', '54%3.jpg', '63%2.jpg', '2020.03.03.20030353-p17-106%2.png', '29%1.jpg', '889.png', '81%0.jpg', 'bmj.m606.full-p4-22%0.png', '2020.03.22.20040782-p24-1533-5.png', '72.jpg', '2020.02.10.20021584-p6-52%10.png', '2020.03.22.20034041-p17-80-1.png', '586.png', '1877.png', '2020.02.25.20021568-p23-108%8.png', '47.jpg', '11%1.jpg', '2020.02.22.20024927-p20-70%3.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%11.png', '2020.03.11.20033159-p12-45%2.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%3.png', '51%3.jpg', '75%0.jpg', '1221.png', '81.png', '57%0.jpg', '2020.03.03.20030775-p10-88%1.png', '576.png', 'PIIS0140673620301549_0%1.png', '2020.03.01.20029769-p21-73_2%0.png', '86.jpg', '69%1.jpg', 'kjr-21-e25-p1-6.png', '89%2.jpg', '2020.02.25.20021568-p23-108%15.png', '1857.png', '2020.02.26.20026989-p34-114_2%1.png', '2020.02.10.20021584-p6-52%0.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%0.png', '6%3.jpg', '1078.png', '2020.02.17.20024018-p17-61%0.png', '591.png', '2020.03.04.20030395-p27-108%3.png', '2108.png', '28.jpg', '2341.png', 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%0.png', '1336.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%1.png', '1330.png', '2020.03.22.20040782-p24-1533-8.png', '12.jpg', '2020.02.10.20021584-p6-52%4.png', '22%1.jpg', '2020.03.20.20037325-p23-122.png', '2020.03.04.20031039-p23-97_2%0.png', '956.png', '2020.03.03.20030353-p17-106%1.png', '2020.02.10.20021584-p6-52%15.png', '1309.png', '2020.03.19.20038539-p10-58.png', '2020.02.23.20026856-p17-115%5.png', '74.jpg', '754.png', '4%5.jpg', '1876.png', 'bmj.m606.full-p4-22%7.png', '2020.03.04.20031047-p13-84%4.png', '2020.03.09.20033118-p20-93%0.png', '2020.02.26.20026989-p34-114_1%1.png', '2020.02.24.20027052-p8-73%2.png', '177.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%1.png', '2020.03.11.20033159-p12-45%3.png', 'PIIS0140673620301549_1.png', '2020.02.10.20021584-p6-52%7.png', '2020.03.26.20041426-p11-121.png', '2020.03.19.20038539-p10-59.png', '4%1.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%5.png', '705.png', '54%2.jpg', '51%2.jpg', '2020.03.04.20031039-p23-97_2%2.png', '2020.03.04.20026005-p12-1946%5.png', '354.png', '15.png', '1923.png', '29%0.jpg', '1858.png', '2020.03.13.20035212-p23-153.png', '2020.02.27.20027557-p25-137%4.png', '82%2.jpg', '2020.02.22.20024927-p20-70%0.png', '2020.03.04.20026005-p12-1946%0.png', '65%4.jpg', 'ehaa254-p0-63%2.png', '2020.03.16.20036145-p19-128-2.png', '2020.03.18.20038125-p16-56-1.png', '39%0.jpg', '2020.02.19.20025023-p20-93%2.png', '78.jpg', '39%8.jpg', '1029.png', '2020.03.03.20030775-p11-91.png', '2020.02.27.20027557-p25-137%0.png', '19%0.jpg', '2142.png', '27.jpg', '2020.03.04.20026005-p12-1946%4.png', '2020.02.26.20027938-p6-80_2%1.png', '2020.03.25.20037721-p27-172.png', '1333.png', '67%0.jpg', 'ehaa254-p0-63%3.png', '2020.03.21.20040691-p18-67-2.png', '1562.png', '49%1.jpg', '2020.03.13.20035212-p23-160.png', '1575.png', '70%0.jpg', '36%0.jpg', '1062.png', '2020.03.26.20041426-p11-128.png', '2020.03.04.20030395-p27-108%4.png', '1099.png', '49%0.jpg', '2039.png', '2020.03.01.20029769-p21-73_2%2.png', '1058.png', '1071.png', '2020.03.18.20038125-p16-56-4.png', '34.jpg', '21%1.jpg', '104.png', '90.jpg', '2020.03.13.20035212-p23-157.png', '2020.02.23.20026856-p17-115%3.png', '1217.png', '35%0.jpg', '2020.02.23.20026856-p17-115%2.png', '1060.png', '2020.02.10.20021584-p6-52%3.png', '2020.03.25.20037721-p27-174.png', '37%3.jpg', '2020.03.11.20033159-p12-45%5.png', '662.png', '5%3.jpg', '2020.03.13.20035212-p23-158.png', '1308.png', '36%1.jpg', '1%2.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%8.png', '1050.png', '85%1.jpg', '921.png', '65%3.jpg', 'bmj.m606.full-p4-22%6.png', '2020.03.04.20031039-p23-97_2%1.png', '2138.png', '2020.03.22.20040782-p24-1533-2.png', '583.png', '5%6.jpg', '39%2.jpg', '54%1.jpg', 'kjr-21-e24-p2-8.png', '1261.png', '1859.png', '2007.png', '39%7.jpg', '2237.png', '1079.png', '2020.03.01.20029769-p21-73_1%0.png', '2020.03.22.20040782-p24-1533-7.png', '4%7.jpg', '70%1.jpg', '2020.03.04.20031047-p13-84%3.png', '2020.03.04.20026005-p12-1946%1.png', '65%0.jpg', '2020.02.28.20029025-p23-182.png', '2020.03.13.20035212-p23-163.png', '6%0.jpg', '2020.02.25.20021568-p24-111%6.png', '381.png', '378.png', '2%1.jpg', '37%1.jpg', '2020.03.04.20031047-p12-81%3.png', '2020.03.04.20031047-p13-84%5.png', '1065.png', '2020.03.04.20031047-p12-81%0.png', '2020.02.25.20021568-p23-108%12.png', '2020.02.26.20027938-p6-80_2%0.png', '14.png', '46.jpg', '2020.03.19.20038539-p10-62.png', '80%1.jpg', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%1.png', '2020.02.23.20026856-p17-115%1.png', '81%1.jpg', '577.png', '19%1.jpg', '227.png', '1046.png', '33%1.jpg', '1297.png', '63%6.jpg', '2020.03.18.20038125-p15-54-4.png', '14%0.jpg', '2020.02.25.20021568-p24-111%7.png', '2020.02.24.20027052-p8-73%0.png', '9.jpg', '8.jpg', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%0.png', '102.png', '14%3.jpg', 'kjr-21-e24-p3-16.png', '84%0.jpg', '1072.png', '1497.png', '2020.03.11.20033159-p12-45%1.png', '2020.03.20.20037325-p23-124.png', '38%0.jpg', '2020.03.09.20033118-p20-93%2.png', '2020.03.18.20038125-p16-56-2.png', '2020.02.25.20027763-p15-53%1.png', '1638.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%4.png', '2020.03.20.20037325-p23-121.png', '2020.03.30.20047985-p9-56%1.png', '2020.03.04.20031047-p13-84%2.png', '2020.02.19.20025023-p20-93%0.png', '42.jpg', '2020.03.21.20040691-p18-67-1.png', '1310.png', '2020.03.01.20029769-p21-73_1%1.png', '673.png', '24.png', '2020.03.26.20041426-p11-127.png', '2020.03.25.20037721-p27-181.png', '2020.02.10.20021584-p6-52%6.png', '29%2.jpg', '6%1.jpg', '2020.03.13.20035212-p23-159.png', '2020.02.25.20021568-p23-108%11.png', '76%0.jpg', '2020.03.26.20041426-p11-123.png', 'PIIS0140673620303603%3.png', '37%0.jpg', '2020.03.21.20040691-p18-67-7.png', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-91.png', '2020.03.22.20040782-p25-1542.png', '44.png', '1305.png', '2020.03.01.20029769-p21-73_3.png', '1115.png', '2145.png', '2020.02.25.20021568-p23-108%0.png', '1215.png', '4%3.jpg', '5%1.jpg', '2020.03.12.20034686-p17-91-6.png', '704.png', '2020.02.11.20021493-p16-109%0.png', 'kjr-21-e25-p1-14.png', 'PIIS0140673620301549_2.png', '38%1.jpg', '2020.03.13.20035212-p23-155.png', '848.png', '2020.02.25.20021568-p23-108%3.png', '116.png', '1070.png', '2020.02.10.20021584-p6-52%13.png', '1920.png', 'PIIS0140673620303603%4.png', '15%0.jpg', '2020.03.18.20038125-p16-56-6.png', '39%1.jpg', '2020.02.11.20022053-p12-67%3.png', '2020.03.18.20038125-p15-54-5.png', '60.jpg', '2020.02.22.20024927-p18-66%1.png', '996.png', '2020.03.03.20030775-p12-93%1.png', '2020.03.30.20047985-p9-56%0.png', '2020.03.03.20030353-p17-106%0.png', '63%4.jpg', '2020.03.12.20034686-p17-91-4.png', '2020.03.01.20029769-p21-73_1%2.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%0.png', '43%1.jpg', '1328.png', '2020.02.26.20027938-p6-80_3%0.png', '2020.03.22.20040782-p24-1533-4.png', '2020.02.25.20021568-p23-108%2.png', '21%0.jpg', '82%1.jpg', '2020.03.04.20030395-p27-108%0.png', '1047.png', '39%6.jpg', '2020.02.10.20021584-p6-52%2.png', '2020.03.25.20037721-p27-182.png', '2020.02.25.20021568-p23-108%4.png', '2020.02.25.20021568-p23-108%1.png', '17%0.jpg', '43%2.jpg', '1059.png', '5%2.jpg', '2020.02.10.20021584-p6-52%12.png', '2020.02.24.20027201-p19-669.png', '2020.03.16.20036145-p19-128-1.png', '30%5.jpg', '2020.02.25.20021568-p24-111%1.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%2.png', '5%0.jpg', '2020.03.21.20040691-p18-67-3.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%1.png', '44%1.jpg', '2020.03.26.20041426-p11-130.png', '65%1.jpg', '10%0.jpg', '1499.png', '38%2.jpg', '40%0.jpg', '2020.03.13.20035212-p23-156.png', '1498.png', '7.jpg', '71%1.jpg', '2020.03.25.20037721-p27-173.png', '2020.03.26.20041426-p11-119.png', '63%0.jpg', '11%0.jpg', '2020.03.22.20040782-p25-1546.png', '2020.03.21.20040691-p18-67-5.png'], 'validation': ['30%0.jpg', '25.png', '2020.02.25.20021568-p24-111%5.png', '2020.03.22.20034041-p17-80-2.png', '2040.png', '3.jpg', '1332.png', '66.jpg', 'kjr-21-e25-p1-8.png', '2020.03.22.20034041-p18-92-1.png', '2020.03.12.20027185-p8-218.png', '29%3.jpg', '1213.png', '2020.03.18.20038125-p15-54-6.png', '56%0.jpg', '2020.03.04.20026005-p12-1946%3.png', '1577.png', '1814.png', '2020.03.25.20043166-p17-81.png', 'kjr-21-e25-p1-10.png', '52%0.jpg', '2020.03.22.20040782-p25-1544.png', '1576.png', '2020.03.04.20026005-p12-1946%2.png', '2020.03.18.20038125-p16-56-5.png', '2020.02.22.20024927-p19-68%5.png', '76%2.jpg', '82%3.jpg', '1123.png', '2020.03.18.20038125-p15-54-2.png', '2020.02.22.20024927-p19-68%3.png', '2020.03.26.20041426-p11-125.png', '5%4.jpg', '2020.02.22.20024927-p19-68%4.png', '14%2.jpg', '382.png', '10%1.jpg', '706.png', 'PIIS0140673620303603%0.png', '26%1.jpg', '84%1.jpg', '2020.03.22.20040782-p25-1545.png', '2020.03.16.20035105-p6-64-2.png', '2020.03.21.20040691-p18-6-6.png', '69%0.jpg', '414.png', 'PIIS0140673620303603%7.png', '2020.03.22.20040782-p24-1533-3.png', '1702.png', '2020.02.11.20021493-p16-109%1.png', '1845.png', '2020.03.10.20032136-p34-118_2%0.png', '79.jpg', '73.jpg', '969.png', '43.png', '71%0.jpg', '2020.02.25.20021568-p23-108%9.png', '167.png', '37%2.jpg', '383.png', '2020.03.13.20035212-p23-162.png', '2020.03.04.20031039-p23-97_1%0.png', '2020.03.04.20030395-p27-108%7.png', '2020.03.19.20038539-p10-57.png', 'PIIS0140673620303603%2.png', '2020.02.22.20024927-p18-66%2.png', '1264.png', 'bmj.m606.full-p4-22%2.png', '77.jpg', '663.png'], 'test': ['13.jpg', '2020.03.26.20041426-p11-133.png', '85%0.jpg', '2020.02.13.20022673-p13-77%1.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%1.png', '176.png', '67%2.jpg', '2020.03.22.20034041-p18-92-3.png', '1030.png', '82.png', '1262.png', '40%1.jpg', '1%0.jpg', '115.png', '84%3.jpg', '51%0.jpg', '2020.02.10.20021584-p6-52%8.png', '1312.png', '2020.03.12.20034686-p17-91-1.png', '2020.03.09.20033118-p20-93%1.png', '412.png', '39%4.jpg', '575.png', '887.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%1.png', '80%0.jpg', '2020.03.04.20031047-p13-84%1.png', '2020.02.25.20021568-p23-108%14.png', '59.png', 'bmj.m606.full-p4-22%5.png', '2020.03.12.20034686-p17-91-5.png', '1218.png', '53.jpg', '2020.03.04.20030395-p27-108%2.png', '1049.png', '226.png', '413.png', '81%2.jpg', '1921.png', '2020.03.04.20031047-p12-81%2.png', '294.png', '50.jpg', '2020.03.10.20032136-p34-118_1%1.png', '81%3.jpg', '2020.02.23.20026856-p17-115%0.png', '11%2.jpg', '2020.03.04.20031047-p12-81%1.png', '2020.03.12.20034686-p17-91-2.png', '579.png', '2020.03.16.20035105-p6-64-3.png', '2020.02.19.20025023-p20-93%1.png', '1210.png', '487.png', '1561.png', 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%1.png', '957.png', '114.png', '587.png', '2020.03.21.20040691-p18-67-8.png', '52%1.jpg', '2020.03.04.20030395-p27-108%5.png', 'PIIS0140673620302117_1.png', '2020.03.11.20033159-p12-45%0.png', '17%1.jpg', '25%1.jpg', '39%5.jpg', '5%5.jpg', '1952.png', 'kjr-21-e24-p5-29.png', '580.png', '76%1.jpg', '91%1.jpg'], 'prompt': ['2020.02.10.20021584-p6-52%13.png', '48.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%7.png', '15%0.jpg', '43%2.jpg', '54%2.jpg', '2020.03.25.20043166-p17-79.png', 'PIIS0140673620303603%8.png', '75%1.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%5.png', '2020.02.11.20021493-p16-109%0.png', '2020.02.25.20021568-p24-111%1.png', '82%1.jpg', '1308.png', '2020.02.26.20027938-p6-80_3%0.png', '15%3.jpg', '2020.03.26.20041426-p11-130.png', '62.jpg', '26.png', '591.png', '2020.03.25.20037721-p27-165.png', '1330.png', '2020.03.22.20040782-p25-1546.png', '57%0.jpg', '2020.03.01.20029769-p21-73_2%0.png', '158.png', '2020.03.22.20040782-p24-1533-5.png', '10%0.jpg', '63%2.jpg', '578.png', '2020.02.23.20026856-p17-115%5.png', '2020.02.11.20022053-p12-67%3.png', '2020.02.25.20021568-p23-108%12.png', '2020.03.04.20030395-p27-108%0.png', '2020.02.27.20027557-p25-137%1.png', '30%2.jpg', '2020.03.22.20034041-p16-66.png', '2141.png', '2020.03.13.20035212-p23-154.png', '2020.03.04.20031039-p23-97_1%1.png', '174.png', '1575.png', '2020.03.01.20029769-p21-73_2%2.png', '15%2.jpg', '2020.02.25.20021568-p24-111%0.png', '41.jpg', '2020.02.17.20024018-p17-61%0.png', '38%2.jpg', '1%1.jpg', '2020.03.22.20040782-p24-1533-8.png', '2020.02.27.20027557-p25-137%4.png', '2020.03.16.20036145-p19-128-4.png', '32%1.jpg', '2020.02.26.20027938-p6-80_1%0.png', '1221.png', '2020.03.04.20031039-p23-97_2%1.png', '68.jpg', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%0.png', '672.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%0.png', '25%3.jpg', '2020.03.22.20034041-p18-92-2.png', '2020.03.26.20041426-p11-119.png', '2020.03.25.20037721-p27-182.png', '2020.02.22.20024927-p20-70%1.png', '996.png', '2020.03.25.20043166-p17-78.png', 'PIIS0140673620303603%6.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%2.png', '51%3.jpg', '82%2.jpg', '2020.02.11.20022053-p12-67%1.png', '5%6.jpg', '1065.png', '38%1.jpg', '1333.png', '2020.03.04.20030395-p27-108%4.png', '354.png', '1124.png', '662.png', '2020.03.11.20033159-p12-45%4.png', '18%2.jpg', '15%1.jpg', '1307.png', '2020.03.04.20031047-p13-84%0.png', '1638.png', '1463.png', '673.png', 'PIIS0140673620301549_2.png', '2020.03.30.20047985-p9-56%0.png', '10%2.jpg', '583.png', '1115.png', '2020.03.21.20040691-p18-67-1.png', '2020.03.18.20038125-p16-56-1.png', '2020.02.17.20024018-p17-61%2.png', '63%0.jpg', '2020.01.24.919183-p27-132.png', '1501.png', '2020.03.16.20035105-p6-64-1.png', '39%0.jpg', '2020.03.19.20038539-p10-60.png', '2143.png', '70%1.jpg', '227.png', '29%0.jpg', '1217.png', '35%1.jpg', '1%2.jpg', 'kjr-21-e24-p5-31.png', '1335.png', '2020.02.25.20021568-p23-108%8.png', '1328.png', 'kjr-21-e25-p1-6.png', '63%5.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%6.png', '781.png', '2020.02.22.20024927-p20-70%2.png', '660.png', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%5.png', 'ehaa254-p0-63%2.png', '1238.png', '76%0.jpg', '46.jpg', '35%2.jpg', 'kjr-21-e24-p3-16.png', '26%0.jpg', '2020.03.04.20026005-p12-1946%4.png', '1071.png', '1313.png', 'PIIS0140673620301549_0%0.png', '71%1.jpg', '4%5.jpg', '2020.02.24.20027201-p19-669.png', 'bmj.m606.full-p4-22%1.png', '2020.03.19.20038539-p10-61.png', '39%2.jpg', '586.png', '2020.03.21.20040691-p18-67-4.png', '2020.03.21.20040691-p18-67-3.png', '2020.02.24.20027052-p8-73%3.png', 'bmj.m606.full-p4-22%4.png', 'PIIS0140673620301549_0%1.png', '22%1.jpg', 'bmj.m606.full-p4-22%3.png', '39%1.jpg', '72.jpg', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-91.png', '2020.02.26.20026989-p34-114_2%1.png', '67%1.jpg', '1099.png', '171.png', '49%0.jpg', '2020.03.01.20029769-p21-73_4.png', '2020.02.10.20021584-p6-52%5.png', '1499.png', '2020.03.26.20041426-p11-120.png', '2020.03.22.20034041-p17-80-1.png', '2020.03.21.20040691-p18-67-5.png', '19%1.jpg', '2138.png', '2020.03.12.20027185-p8-221.png', '29%2.jpg', '78.jpg', '2020.03.03.20030353-p17-106%0.png', '2020.03.25.20037721-p27-172.png', '33%1.jpg', '361.png', '2020.03.11.20033159-p12-45%2.png', '2020.02.25.20021568-p23-108%13.png', '1336.png', '7.jpg', '2020.03.20.20037325-p23-123.png', '1061.png', '2020.03.04.20031047-p13-84%5.png', '31.jpg', '576.png', '116.png', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%0.png', '1261.png', '2020.03.19.20038539-p10-62.png', '2020.03.01.20029769-p21-73_1%2.png', '2020.03.04.20026005-p12-1946%1.png', '2020.02.11.20021493-p16-109%2.png', '83%1.jpg', '2020.03.13.20035212-p23-157.png', '2020.02.17.20024018-p17-61%4.png', '6%1.jpg', '2020.02.24.20027052-p8-73%1.png', '2020.03.10.20032136-p34-118_1%0.png', '4%7.jpg', '2020.03.04.20031039-p23-97_1%2.png', '21%0.jpg', '2020.03.04.20031047-p13-84%4.png', '2020.03.22.20040782-p24-1533-7.png', '8.jpg', '2108.png', '1212.png', '65%0.jpg', '2020.03.22.20040782-p25-1543.png', 'kjr-21-e24-p2-8.png', '34.jpg', '848.png', '2020.01.24.919183-p27-133.png', '2020.02.23.20026856-p17-115%2.png', '590.png', '5%2.jpg', '2020.03.16.20036145-p19-128-2.png', '2020.02.26.20027938-p6-80_2%0.png', '118.png', '2020.02.25.20021568-p23-108%6.png', '65%4.jpg', 'PIIS0140673620301549_1.png', '2007.png', '2020.03.18.20038125-p16-56-3.png', 'med-9780198814924-chapter-17-p10-515.png', '1500.png', 'kjr-21-e25-p1-14.png', '2020.02.25.20021568-p24-111%3.png', '65%3.jpg', '585.png', '849.png', '2020.03.20.20037325-p23-122.png', '2020.02.25.20021568-p23-108%5.png', '709.png', '2020.02.10.20021584-p6-52%9.png', '1857.png', '2020.03.22.20040782-p24-1533-1.png', '6%3.jpg', '704.png', '74.jpg', '1215.png', '2020.02.19.20025023-p20-93%3.png', '2020.03.20.20037325-p23-121.png', '1050.png', '37%0.jpg', 'bmj.m606.full-p4-22%6.png', '2020.03.12.20027185-p8-217.png', '103.png', '1877.png', '2%1.jpg', '2020.02.25.20021568-p23-108%7.png', '2020.01.24.919183-p27-135.png', '2020.03.13.20035212-p23-159.png', '2020.03.11.20033159-p12-45%3.png', '2020.03.04.20026005-p12-1946%0.png', '28.jpg', '385.png', '2020.02.23.20026856-p17-115%4.png', '1240.png', '2%3.jpg', '2020.02.10.20021584-p6-52%7.png', '2020.03.04.20031047-p12-81%3.png', '2020.02.25.20021568-p23-108%1.png', '2020.03.26.20041426-p11-124.png', '64.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%2.png', '2020.03.18.20038125-p15-54-4.png', 'kjr-21-e25-p1-12.png', '4%3.jpg', '1079.png', '2020.02.10.20021584-p6-52%11.png', '2020.03.03.20030775-p10-88%0.png', '2020.02.10.20021584-p6-52%1.png', '2020.02.22.20024927-p18-66%0.png', '2020.03.09.20033118-p20-93%3.png', '2020.03.30.20047985-p9-56%3.png', '54%3.jpg', '2020.02.10.20021584-p6-52%12.png', '30%1.jpg', '63%4.jpg', '2020.03.22.20040782-p24-1533-4.png', '2%2.jpg', '2020.03.16.20036145-p19-128-1.png', '2020.02.25.20021568-p24-111%4.png', '2020.02.10.20021584-p6-52%4.png', '54%0.jpg', '2020.03.26.20041426-p11-128.png', '2020.03.04.20031047-p13-84%2.png', '2144.png', '1922.png', '1309.png', '2020.03.09.20033118-p20-93%0.png', '63%6.jpg', '1876.png', '2020.02.26.20027938-p6-80_1%1.png', '49%1.jpg', '33.png', '2020.03.11.20033159-p12-45%5.png', '30%5.jpg', '88.jpg', '2020.02.10.20021584-p6-52%2.png', '21%1.jpg', '1046.png', '2020.02.25.20021568-p23-108%0.png', '51%2.jpg', 'Talaromyces-marneffei-infection-relapse-presenting-as-ost_2020_International-p1-12%1.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%1.png', '2020.03.18.20038125-p16-56-6.png', '2020.03.03.20030775-p11-91.png', '63%7.jpg', '2020.03.12.20034686-p17-91-3.png', '2020.02.22.20024927-p20-70%3.png', '1562.png', '2020.03.26.20041426-p11-132.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p3-10%0.png', '2020.03.11.20033159-p12-45%1.png', '2020.03.04.20031039-p23-97_2%2.png', '2020.03.03.20030775-p10-88%1.png', '9.jpg', '2020.03.19.20038539-p10-58.png', '1889.png', '86.jpg', '1059.png', '2020.03.26.20041426-p11-126.png', '2020.02.19.20025023-p20-93%0.png', '2020.02.24.20027052-p8-73%2.png', '1214.png', '2020.03.04.20026005-p12-1946%5.png', '1263.png', '2020.03.21.20040691-p18-67-7.png', '37%4.jpg', '2020.03.25.20037721-p27-174.png', '2020.03.13.20035212-p23-160.png', '63%3.jpg', '40%0.jpg', '91%0.jpg', '1919.png', '2020.03.03.20030353-p17-106%2.png', '1029.png', '2020.02.25.20021568-p24-111%2.png', '1923.png', '44.png', '956.png', '81%1.jpg', '890.png', 'PIIS0140673620302117_2.png', '112.png', '15.png', '2020.02.10.20021584-p6-52%10.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%4.png', '2020.03.18.20038125-p16-56-4.png', '577.png', '104.png', '1305.png', '2020.03.21.20040691-p18-67-2.png', '2020.03.26.20041426-p11-131.png', '2020.02.23.20026856-p17-115%3.png', '2020.03.13.20035212-p23-152.png', '2020.02.27.20027557-p25-137%0.png', '51%1.jpg', '2020.03.22.20040782-p24-1533-6.png', '84%2.jpg', '2020.03.18.20038125-p15-54-5.png', '70%0.jpg', '2237.png', '42.jpg', '2020.02.25.20021568-p24-111%7.png', '1297.png', '2039.png', '2020.03.03.20030353-p17-106%1.png', '1497.png', '378.png', '5%0.jpg', '1924.png', '2020.02.25.20027763-p15-53%0.png', '37%3.jpg', '381.png', '2020.03.22.20040782-p25-1542.png', '2020.03.19.20038539-p10-59.png', '2020.02.25.20021568-p24-111%9.png', '2020.02.24.20027052-p8-73%0.png', '23.png', '2020.02.26.20026989-p34-114_1%1.png', '2020.02.25.20021568-p24-111%6.png', '2139.png', '705.png', '39%3.jpg', '56%1.jpg', '2019-novel-Coronavirus-severe-adult-respiratory-dist_2020_International-Jour-p3-89%1.png', '75%0.jpg', '16.jpg', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%3.png', '486.png', '80%1.jpg', '35%0.jpg', '5%3.jpg', '1211.png', '5%1.jpg', '2020.02.28.20029025-p23-182.png', '2020.02.26.20027938-p6-80_2%1.png', '592.png', '2020.03.13.20035212-p23-156.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%1.png', 'Ni2020_Article_CombinationOfWesternMedicineAn-p4-13%0.png', '17%0.jpg', '1239.png', '2020.03.09.20033118-p20-93%2.png', '1859.png', '2020.03.22.20040782-p25-1541.png', '102.png', '2020.03.04.20030395-p27-108%1.png', '117.png', '1101.png', 'Recurrence-of-positive-SARS-CoV-2-RNA-in-C_2020_International-Journal-of-Inf-p1-21%0.png', '2020.02.25.20021568-p24-111%8.png', '81%0.jpg', '2020.02.11.20022053-p12-67%2.png', '2020.02.10.20021584-p6-52%6.png', '2020.03.04.20030395-p27-108%6.png', '1888.png', '1306.png', '2020.03.10.20032136-p34-118_2%1.png', '1100.png', '1164.png', '2145.png', '37%1.jpg', '14%3.jpg', '82%0.jpg', '60.jpg', '2020.02.28.20029025-p23-183.png', '83%0.jpg', '29%1.jpg', '2020.02.25.20027763-p15-53%1.png', 'ehaa254-p0-63%3.png', '2020.03.01.20029769-p21-73_1%1.png', '166.png', '1047.png', '1098.png', '43%1.jpg', '2020.02.25.20021568-p23-108%11.png', '1331.png', '921.png', '2341.png', '36%0.jpg', '2020.03.22.20040782-p24-1533-2.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%11.png', 'PIIS0140673620303603%4.png', '63%1.jpg', 'impact_of_covid19_infection_on_pregnancy_outcomes_and_the_risk_of_maternaltoneonatal_intrapartum_transmission_of_covid19_during_natural_birth-p8-41%3.png', '39%6.jpg', '14.png', '18%1.jpg', '1428.png', '2020.03.03.20030775-p12-93%1.png', '2041.png', '1062.png', '2020.03.01.20029769-p21-73_2%1.png', '1048.png', '2020.03.04.20031039-p23-97_2%0.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%9.png', '14%1.jpg', '39%7.jpg', '33%0.jpg', '2020.03.04.20030395-p27-108%8.png', '2020.03.13.20035212-p23-163.png', '69%1.jpg', '2020.03.13.20035212-p23-153.png', '30%4.jpg', '59%0.jpg', '2020.02.13.20022673-p13-77%0.png', '2020.02.25.20021568-p23-108%3.png', '997.png', '81.png', '1060.png', '12.jpg', '6%0.jpg', '2020.03.26.20041426-p11-122.png', '85%1.jpg', '2020.02.22.20024927-p18-66%1.png', 'Comparison-of-different-samples-for-2019-novel-cor_2020_International-Journa-p2-21%10.png', '2020.02.26.20027938-p6-80_3%1.png', '1310.png', '1846.png', '2020.03.16.20036145-p19-128-3.png', '44%1.jpg', '26%3.jpg', '1858.png', '2142.png', '30%3.jpg', '754.png', 'PIIS0140673620303603%1.png', '22%0.jpg', '2140.png', '2020.02.10.20021584-p6-52%3.png', '2020.03.13.20035212-p23-161.png', '2020.03.12.20034686-p17-91-6.png', '0.jpg', '2020.03.01.20029769-p21-73_1%0.png', '2020.03.04.20030395-p27-108%3.png', '2020.03.25.20043166-p17-80.png', '2020.03.12.20034686-p17-91-4.png', '2020.02.19.20025023-p20-93%2.png', '1298.png', '45.jpg', '661.png', '2020.02.10.20021584-p6-52%15.png', '2020.02.25.20021568-p23-108%4.png', '47.jpg', '1498.png']}, 'split_ids_sha256': {'validated': 'f8f268e1001fa3214de2383c7df3a176c55a7786ef40d61af648794d0bdb437e', 'train': '1539dd12f89c7a9a38e28a3911a87c5509c00e45bad2d64b534ffd8557c3e869', 'validation': 'e27b60657e5d0ddf4dc9eb4d34ca02c909d34d599369e24fc656c6cf3390b0c1', 'test': 'd25db8139687e666c2e5c9bb5a8e1f000be78ee0eb244f114438f4bf9fc13f44', 'prompt': '2e1e001a0d2e45d35e8ee16522e4347bc9ee1bccb67ff9532d8c21cb9c2f65d0'}, 'manifest_sha256': {'validated_manifest.csv': '1536c6eb78e2857e599da4f7b3854ed5fa01fd66e50f390dd315d72fe5572aee', 'train_manifest.csv': '8902b8fea1163429aec6940a9744f7c792d1b7f9a00ca300a1fab17448ead973', 'validation_manifest.csv': '94c555cb25a19485616065079139722bfe7122a17639267ba5e03d9e4cfa7c5d', 'test_manifest.csv': 'fcbcd1bd12220b0a3a286c4b1a82a8543e34c0a3495c5d0b8c89c81033b3eb66', 'prompt_manifest.csv': 'f60c45a1dc9beecba16917d767f1cf6b0f67231141a9226efdb771a27f9281de'}, 'prompt_ids_sha256': '2e1e001a0d2e45d35e8ee16522e4347bc9ee1bccb67ff9532d8c21cb9c2f65d0', 'report_normalization': 'Historical NFC + whitespace collapse + strip', 'exclusions': {'conflicting_duplicate_keys': 6, 'unannotated_images': 26}, 'no_report_text_exported': True}

In [ ]:
import os, queue, signal, subprocess, threading, time
require = lambda condition, message: None if condition else (_ for _ in ()).throw(ValueError(message))
require(not globals().get('_gate4_attempted', False), 'Already attempted; no automatic retry')
require(Path('/kaggle/input').is_dir(), 'Kaggle-only execution; local execution prohibited')
_gate4_attempted = True
from datetime import datetime, timezone
supervisor_started = time.monotonic()
def supervisor_record(phase, state='event', elapsed_seconds=None, **details):
    return dict(phase=phase,state=state,timestamp_utc=datetime.now(timezone.utc).isoformat(),
                elapsed_seconds=round(time.monotonic()-supervisor_started,6) if elapsed_seconds is None else elapsed_seconds, **details)
print(json.dumps(supervisor_record('notebook_start',state='start')),flush=True)
RUN_ROOT = Path('/kaggle/working') / ('zeromrg_gate4b_' + uuid.uuid4().hex[:12])
RUN_ROOT.mkdir(exist_ok=False)
(RUN_ROOT/'supervisor_status.json').write_text(json.dumps(supervisor_record('notebook_export_start',state='start'))+'\n',encoding='utf-8')
for name, source in SOURCES.items():
    require(hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], 'Embedded source differs')
    (RUN_ROOT/name).write_text(source, encoding='utf-8')
(RUN_ROOT/'configuration.json').write_text(json.dumps(CONFIG,indent=2)+'\n',encoding='utf-8')
(RUN_ROOT/'dataset-contract.json').write_text(json.dumps(DATASET_CONTRACT,indent=2)+'\n',encoding='utf-8',newline='\n')
require(hashlib.sha256((RUN_ROOT/'dataset-contract.json').read_bytes()).hexdigest()==CONFIG['dataset_contract_sha256'], 'Exported contract bytes differ')
print('Gate4B <=5 distinct training images plus ONE repeat; output:', RUN_ROOT, flush=True)
print('Hard worker limit:', CONFIG['hard_timeout_seconds'], 'seconds; engineering bound, not measured feasibility', flush=True)
started = time.monotonic()
command = [sys.executable, '-u', str(RUN_ROOT/'gate4b_diagnostic_worker.py'),
           '--run-root', str(RUN_ROOT), '--config', str(RUN_ROOT/'configuration.json')]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, start_new_session=True, env={**os.environ, 'PYTHONUNBUFFERED':'1'})
messages = queue.Queue()
def read_output():
    for line in process.stdout: messages.put(line)
reader = threading.Thread(target=read_output,daemon=True)
reader.start()
last_heartbeat = started
try:
    with (RUN_ROOT/'execution.log').open('x',encoding='utf-8') as output:
        while process.poll() is None or reader.is_alive() or not messages.empty():
            now = time.monotonic()
            if now-started > CONFIG['hard_timeout_seconds']:
                failure=supervisor_record('worker_timeout',state='failed',elapsed_seconds=now-supervisor_started,status='HARD_TIMEOUT',seconds=now-started,error_type='TimeoutError',no_retry=True)
                (RUN_ROOT/'supervisor_failure.json').write_text(json.dumps(failure,indent=2)+'\n')
                print(json.dumps(failure),flush=True)
                try:
                    os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError:
                    pass
                if process.poll() is None:
                    process.wait(timeout=10)
                raise TimeoutError('Gate4 worker exceeded hard bound; no retry')
            try:
                line=messages.get(timeout=1)
                output.write(line); output.flush(); print(line,end='',flush=True)
            except queue.Empty:
                pass
            if now-last_heartbeat >= 30:
                heartbeat=supervisor_record('supervisor_heartbeat',state='running',elapsed_seconds=round(now-supervisor_started,3),worker_returncode=process.poll())
                output.write(json.dumps(heartbeat)+'\n'); output.flush()
                (RUN_ROOT/'supervisor_status.json').write_text(json.dumps(heartbeat)+'\n',encoding='utf-8')
                print(json.dumps(heartbeat),flush=True)
                last_heartbeat=now
    require(process.returncode == 0, 'Diagnostic failed; inspect execution.log/failure.json; no retry')
    result=json.loads((RUN_ROOT/'diagnostic.json').read_text(encoding='utf-8'))
    require(result['image_count'] == CONFIG['max_images'] and result['generation_count'] == CONFIG['max_generations'] and result['split'] == 'checkpoint_train' and
            not result['test_references_accessed'] and not result['optimizer_created'], 'Diagnostic scope differs')
    print('Prepared diagnostic result:', RUN_ROOT/'diagnostic.json',flush=True)
except BaseException as error:
    failure=supervisor_record('supervisor_exception',state='failed',error_type=type(error).__name__,no_retry=True)
    (RUN_ROOT/'supervisor_exception.json').write_text(json.dumps(failure,indent=2)+'\n',encoding='utf-8')
    print(json.dumps(failure),flush=True)
    raise
finally:
    if process.poll() is None:
        os.killpg(process.pid,signal.SIGKILL)
        process.wait(timeout=10)